## Cell 0 — Install Boosted Libraries (run once before Cell 1)

In [1]:
# Run this cell ONCE before Cell 1. Safe to re-run.
# Installs xgboost, lightgbm, catboost into the current kernel environment.
import subprocess, sys

for _lib in ["xgboost", "lightgbm", "catboost"]:
    try:
        __import__(_lib)
        print(f"  {_lib}: already installed")
    except ModuleNotFoundError:
        print(f"  {_lib}: installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", _lib, "-q"])
        print(f"  {_lib}: installed OK")

print("\nAll boosted libraries ready. Restart kernel if this was a fresh install.")


  xgboost: already installed
  lightgbm: already installed
  catboost: already installed

All boosted libraries ready. Restart kernel if this was a fresh install.


# Agricultural Drone Pipeline v9

**Run cells in order: Cells 0–10 define all functions. Cell 10 sets up the output directory. Then run Execution cells in order.**

## What's new in v9

- **Unit fix: `sprayed_area` mu → ha** — DJI Smart Farm XLSX exports area in 亩 (mu), not hectares. The web interface silently converts to ha; the raw CSV does not. Fix applied in `phase1_load_and_clean()` immediately after CSV parse: `df["sprayed_area"] /= 15.0`. Raw value preserved in `sprayed_area_mu_raw` for audit. At the confirmed operational rate of 20 L/ha, corrected values are now physically consistent with loaded volumes and flight durations. **All v8.x results are based on 15× inflated area values and must not be cited.**
- **Feature set reduced from 17 → 12** — Four features dropped, one group moved to future work:
  - `relative_humidity_2m` removed — indirect pathway to duration breaks under mission-driven stopping; correlated with `temperature_2m`.
  - `day_of_week` removed — no defensible mechanism in continuous Philippine highland operations.
  - `time_since_last_flight` removed — redundant with `starting_battery`. Battery readiness is directly measured; drop the proxy, keep the measurement.
  - `wind_direction_10m` (encodes to `wind_u`, `wind_v`) moved to **future work** — physically uninterpretable without drone heading, which is unavailable in DJI Smart Farm CSV logs.
- **Final feature set (12):** `starting_battery`, `sprayed_area`, `total_amount_l` (operational) · `temperature_2m`, `wind_speed_10m`, `precipitation` (environmental) · `hour_sin`, `hour_cos`, `month_sin`, `month_cos`, `flight_index_within_day` (temporal) · `rolling_duration_3` (sequential).
- **Output directory**: `outputs_v9` · **Models**: `outputs_v9/saved_models/`
  (Windows path: `C:\\Users\\User\\Documents\\THESIS_HUHU\\FINAL PIPELINE\\outputs_v9`)

## What's new in v8.3

- **Phase 7B** — Sequential Permutation Importance for GRU, LSTM, Seq-RF.
  Shuffles each feature across all 5 timesteps (10 repeats). Aggregates δMAE across timesteps. Compares all 4 model families in one table.
  Output: `sequential_permutation_importance.csv`
- **`GRUWrapper`** and **`LSTMWrapper`** — sklearn-compatible wrappers enabling
  `sklearn.inspection.permutation_importance` to interface with PyTorch models.

## What's new in v8.1.2
- Phase 6D: operational & sequential feature ablation (18 runs, supplementary RQ2).
- `OPERATIONAL_ABLATION_NOTE` justification string.

## What's new in v8.1
- Evaluation validity fix — flat vs sequence-aligned comparison.
- `phase5_boosted_flat_eval()` and Phase 5B-Flat exec cell (E4b).

## What's new in v7.3
- Session sort fix, `parse_end_dt()`, traceable `pred_df`, `df_test_meta` alignment, LightGBM inference fix.

## Cell structure

| Cell | Purpose |
|---|---|
| 0 | Install boosted libraries |
| 1 | Imports, constants, CFG, justification strings |
| 2 | Helper functions (leakage check, parsers, ERA5, session IDs) |
| 2.5 | Audit & validation helpers |
| 3 | Data pipeline definitions (phase1–3 + EDA) |
| 4 | Feature & sequence builders |
| 5 | Model classes (GRU, LSTM) + training loop |
| 6 | Phase 5 baseline definition |
| 6b | Phase 5B + Phase 5B-Flat definitions |
| 7 | Phase 6 ablation + Phase 6b LSTM sensitivity + Phase 6D operational ablation |
| 7b | Phase 6A baseline control + Phase 6C sequence-aligned evaluation |
| 8 | Phase 7 visualization + Phase 7B sequential permutation importance |
| 9 | V6 phase definitions (threshold, 5M, 5F, 5R) |
| 10 | Setup: output directory (`outputs_v9`) |

## Execution cells

| Exec | Purpose |
|---|---|
| E1 | [EXPERIMENT] Threshold Comparison — 1 min vs 2 min |
| E2 | [MAIN PIPELINE] Phase 1–3: data load, clean, **mu→ha unit fix**, feature engineering (12 features) + audits |
| E3 | [MAIN PIPELINE] Phase 5: GRU, LSTM, Seq-RF baseline (RQ1) |
| E4 | [MAIN PIPELINE] Phase 5B: boosted models |
| E4b | [MAIN PIPELINE] Phase 5B-Flat: flat individual-flight evaluation |
| E5 | [MAIN PIPELINE] Phase 5E: export all trained artifacts → `outputs_v9/saved_models/` |
| E6 | [MAIN PIPELINE] Phase 5M: MANUAL baseline (flat RF) |
| E7 | [MAIN PIPELINE] Phase 5F: flat RF vs seq-RF comparison |
| E8 | [MAIN PIPELINE] Phase 5R: 5-repetition stability analysis |
| E9 | [MAIN PIPELINE] Phase 6: GRU environmental ablation (36 runs, RQ2) |
| E9b | [MAIN PIPELINE] Phase 6D: GRU operational ablation (18 runs, supplementary) |
| E10 | [MAIN PIPELINE] Phase 6A: ablation baseline control |
| E11 | [MAIN PIPELINE] Phase 6b: LSTM sensitivity check (9 runs) |
| E12 | [MAIN PIPELINE] Phase 6C: sequence-aligned evaluation — all 7 models |
| E13 | [MAIN PIPELINE] Phase 7: visualizations + summary tables |
| **E14** | **[MAIN PIPELINE] Phase 7B: sequential permutation importance — GRU, LSTM, Seq-RF** |



## Cell 1 — Imports, Constants, CFG, Justification Strings

In [2]:
"""
=============================================================================
Agricultural Drone Flight Duration Prediction — PIPELINE v8.3
=============================================================================

VERSION HISTORY
===============

v9.0  (unit fix: sprayed_area mu→ha + feature reduction to 12 features)
-------------------------------------------
FIXED    sprayed_area unit conversion: raw DJI Smart Farm XLSX exports
         area in 亩 (mu), not hectares. Applied df["sprayed_area"] /= 15
         in phase1_load_and_clean() immediately after CSV load.
         At 20 L/ha operational rate (confirmed by operations manager),
         values are now physically consistent with loaded volumes.
         All v8.x results are based on 15x inflated area values — invalid.
REMOVED  relative_humidity_2m — indirect pathway breaks under mission-driven
         stopping; correlated with temperature; not defensible standalone.
REMOVED  day_of_week — no defensible mechanism in continuous Philippine
         highland agricultural operations.
REMOVED  time_since_last_flight — redundant with starting_battery.
         Battery readiness is directly measured; drop the proxy.
FUTURE   wind_direction_10m (wind_u, wind_v) — moved to future work.
         Drone heading unavailable in DJI Smart Farm CSV; without heading,
         headwind/tailwind angle cannot be computed. Physically uninterpretable.
CHANGED  CFG["output_dir"] → outputs_v9
CHANGED  Exported models save to outputs_v9/saved_models/
RESULT   Final feature set: 12 features
         (3 operational + 3 environmental + 5 temporal + 1 sequential)
NOT CHANGED  Model architectures, training logic, ablation design,
             session construction, scalers, leakage controls.

v8.3  (sequential permutation importance — Phase 7B)
-------------------------------------------
ADDED    phase7b_sequential_permutation_importance() — computes
         permutation importance for all 3 sequential models (GRU,
         LSTM, Seq-RF) using the same AUTO test set from Phase 5.
         Each feature is shuffled across all 5 timesteps and the
         resulting MAE increase (δMAE) is recorded. 10 repeats per
         feature per model for stable estimates.
         GRUWrapper and LSTMWrapper classes allow sklearn
         permutation_importance to interface with PyTorch models.
         Importances aggregated across timesteps (mean per feature).
         Output: sequential_permutation_importance.csv
ADDED    Phase 7B markdown + execution cell (E14) after Phase 7 (E13).
REASON   Permutation importance in Phase 5B was computed on boosted
         models only. Adviser requirement: importance must be
         validated on the primary sequential models (GRU, LSTM,
         Seq-RF) that answer RQ1 and RQ2. Cross-model agreement
         across all 4 model families strengthens the finding that
         operational features dominate flight duration prediction.
CHANGED  CFG["output_dir"] → outputs_v9
         (Windows path: C:/Users/User/Documents/THESIS_HUHU/
          FINAL PIPELINE/outputs_v9)
NOT CHANGED  All models, features, ablation, evaluation — unchanged.

v8.1.2  (Phase 6D — operational & sequential feature ablation, supplementary RQ2)
-------------------------------------------
ADDED    phase6d_operational_ablation() — 18-run supplementary ablation:
         3 AUTO contexts (Dry/Hot/Wet) × 6 feature sets (baseline + remove
         each of 5 operational/sequential features: sprayed_area,
         total_amount_l, starting_battery, rolling_duration_3,
         time_since_last_flight). Same GRU + ABLATION_HP + chronological
         split as Phase 6 (environmental ablation).
         Output: operational_ablation_results.csv
ADDED    Phase 6D markdown + execution cell (E9b) placed after Phase 6 (E9).
ADDED    OPERATIONAL_ABLATION_NOTE justification string explaining why
         Phase 6D is supplementary (not a redesign of RQ2) and why only
         GRU is used (same model = directly comparable δMAE values).
CHANGED  CFG["output_dir"] "outputs_v9" → "outputs_v9".
NOT CHANGED  Phase 6 (36-run environmental ablation) — unchanged.
             All other phases, model architectures, feature set — unchanged.

v8.1  (evaluation validity fix: flat vs sequence-aligned comparison)
-------------------------------------------
FIXED    Evaluation design: models were previously compared on different
         test sets (boosted on flat individual-flight split; seq models on
         sequence-aligned split). Now split into TWO clearly labelled
         evaluation scenarios:
           (A) FLAT EVALUATION — RF, XGBoost, LightGBM, CatBoost on the
               full flat AUTO test set (last 15% of individual flights).
               Output: flat_evaluation_results.csv.
           (B) SEQUENCE-ALIGNED EVALUATION — ALL 7 models (GRU, LSTM,
               Seq-RF, RF, XGBoost, LightGBM, CatBoost) on the same
               n=len(pred_df) rows recovered via target_row_idx. This is
               the ONLY valid cross-model comparison.
               Output: prediction_comparison_models.csv.
ADDED    phase5_boosted_flat_eval() — evaluates RF, XGBoost, LightGBM,
         CatBoost on the flat individual-flight test set and saves
         flat_evaluation_results.csv. Complements (not replaces) the
         sequence-aligned comparison.
CHANGED  phase6c_prediction_comparison() — now includes RF (flat) in
         the sequence-aligned table alongside GRU, LSTM, Seq-RF, and
         the 3 boosted models. All 7 models on identical rows.
CHANGED  CFG["output_dir"] "outputs_v7.3" → "outputs_v9".
CHANGED  Exported models save to outputs_v9/saved_models/.
NOT CHANGED  Model architectures, hyperparameters, feature set (17),
             preprocessing, sequence construction, ablation design,
             baseline control, LSTM sensitivity check.

v7.3  (audit fixes: session sort, end_datetime, traceable predictions, LightGBM inference)
-------------------------------------------
FIXED    build_sequences_session_aware() — np.unique() returned sessions in
         lexicographic order (AUTO_1 → AUTO_10 → AUTO_2 → ...) instead of
         chronological numeric order. Fixed by sorting sessions by integer
         suffix. This was a VALIDITY BUG: the temporal split (train→val→test)
         was applied to incorrectly ordered sequences, violating the
         chronological split guarantee. All affected phases: 5, 6, 6A, 6b.
ADDED    build_sequences_session_aware() now returns a third value:
         target_row_idx — the df_auto row index of each sequence's target
         flight. Used to recover exact metadata for prediction output.
ADDED    parse_end_dt() — extracts end_datetime from the raw Flight time
         field ("YYYY-MM-DD HH:MM:SS-HH:MM:SS"). Handles midnight crossings
         (6 flights: end_time < start_time on same date → add 1 day).
ADDED    end_datetime column derived in phase1_load_and_clean().
CHANGED  phase5_baseline() — df_auto now retains start_datetime and
         end_datetime columns alongside feature_cols. pred_df now includes
         start_datetime, session_id, and target_row_idx columns.
FIXED    phase6c_prediction_comparison() df_test_meta alignment bug.
         Previously: df_auto.tail(len(pred_df)) — wrong because sessions
         with ≤5 flights contribute no sequences, so df_auto row count
         and sequence count differ. Fixed: df_auto.iloc[target_row_idx]
         recovers the exact target flights of test sequences.
FIXED    LightGBM inference in phase6c — was passing
         pd.DataFrame(X, columns=feat_avail) with real column names, but
         LGBM was trained on numpy array (auto-named Column_0…Column_16).
         Fixed: pd.DataFrame(X, columns=lgbm_feat) where lgbm_feat comes
         from model.feature_name_.
CHANGED  CFG["output_dir"] "outputs_v7.2" → "outputs_v7.3".
NOT CHANGED  Model architectures, hyperparameters, training logic,
             feature engineering, normalization, ablation design,
             leakage controls, Phase 5B boosted pipeline, Phase 7.

v7.2  (output path update + expanded prediction comparison table)
-------------------------------------------------------------------
CHANGED  CFG["output_dir"]               "outputs_v7.1" → "outputs_v7.2".
CHANGED  Export cell E5 markdown         fixed stale path reference
                                          "outputs_v6.1.5" → "outputs_v7.2".
CHANGED  phase5_boosted_models()         now computes and returns
                                          boosted_test_preds dict containing
                                          xgb_pred, lgbm_pred, catboost_pred
                                          on the boosted-model test set.
                                          NOTE: boosted test set (flat,
                                          individual flights) is structurally
                                          different from seq model test set
                                          (sequences). boosted_test_preds is
                                          used for stand-alone evaluation only.
CHANGED  phase6c_prediction_comparison() new signature:
                                          (pred_df, boosted_test_preds,
                                           df_test_meta, output_dir,
                                           xgb_model, lgbm_model, cb_model).
                                          Added metadata columns (session_id,
                                          start_datetime, sprayed_area, etc.)
                                          and boosted model prediction + error
                                          columns (xgb_pred, lgbm_pred,
                                          catboost_pred). Boosted predictions
                                          re-evaluated on the seq-aligned test
                                          rows (df_test_meta) for valid
                                          row-by-row comparison.
FIXED    Removed duplicate ABLATION_BASELINE_NOTE definition in Cell 1.
         Kept the shorter second occurrence; deleted the first.
NOT CHANGED  All modeling logic, training, ablation, architectures,
             hyperparameters, scalers, leakage controls, Phase 6/6A/6b/7.

v7.1  (full 17-feature baseline control + prediction comparison table)
-------------------------------------------
ADDED    phase6_ablation_baseline() — explicit full 17-feature GRU baseline
         control for the ablation framework. Runs GRU with all 17 features on
         the 3 AUTO contexts (Dry/Hot/Wet) using ABLATION_HP and the same
         chronological split. Labels results "full_features_baseline".
         Output: ablation_baseline_control.csv
         Rationale: makes the baseline reference explicit and defense-ready.
         The ablation "none (baseline)" rows are mathematically identical but
         this phase surfaces them as a named, standalone control output.
ADDED    phase6c_prediction_comparison() — prediction comparison table for
         statistician. Reformats existing pred_df from phase5_baseline into:
         Sample | Actual | LSTM_Pred | GRU_Pred | RF_Pred |
         LSTM_Error | GRU_Error | RF_Error
         Output: prediction_comparison_models.csv
         No retraining — uses test-set predictions already computed in Phase 5.
ADDED    ABLATION_BASELINE_NOTE — justification string for baseline control.
CHANGED  CFG["output_dir"]  "outputs_v6.1.5" → "outputs_v7.1".
NOT CHANGED  All existing results (RQ1/RQ2/RQ3) — unchanged.
             GRU/LSTM/RF/boosted model architectures, HPs, training logic.
             36-run ablation grid — unchanged and valid.
             Feature engineering, splits, scalers, leakage controls.
             Phase 6b LSTM sensitivity (9 runs) — unchanged.

v8.3  (sequential permutation importance — Phase 7B)
-------------------------------------------
ADDED    phase7b_sequential_permutation_importance() — computes
         permutation importance for all 3 sequential models (GRU,
         LSTM, Seq-RF) using the same AUTO test set from Phase 5.
         Each feature is shuffled across all 5 timesteps and the
         resulting MAE increase (δMAE) is recorded. 10 repeats per
         feature per model for stable estimates.
         GRUWrapper and LSTMWrapper classes allow sklearn
         permutation_importance to interface with PyTorch models.
         Importances aggregated across timesteps (mean per feature).
         Output: sequential_permutation_importance.csv
ADDED    Phase 7B markdown + execution cell (E14) after Phase 7 (E13).
REASON   Permutation importance in Phase 5B was computed on boosted
         models only. Adviser requirement: importance must be
         validated on the primary sequential models (GRU, LSTM,
         Seq-RF) that answer RQ1 and RQ2. Cross-model agreement
         across all 4 model families strengthens the finding that
         operational features dominate flight duration prediction.
CHANGED  CFG["output_dir"] → outputs_v9
         (Windows path: C:/Users/User/Documents/THESIS_HUHU/
          FINAL PIPELINE/outputs_v9)
NOT CHANGED  All models, features, ablation, evaluation — unchanged.

v8.1.2  (Phase 6D — operational & sequential feature ablation, supplementary RQ2)
-------------------------------------------
ADDED    phase6d_operational_ablation() — 18-run supplementary ablation:
         3 AUTO contexts (Dry/Hot/Wet) × 6 feature sets (baseline + remove
         each of 5 operational/sequential features: sprayed_area,
         total_amount_l, starting_battery, rolling_duration_3,
         time_since_last_flight). Same GRU + ABLATION_HP + chronological
         split as Phase 6 (environmental ablation).
         Output: operational_ablation_results.csv
ADDED    Phase 6D markdown + execution cell (E9b) placed after Phase 6 (E9).
ADDED    OPERATIONAL_ABLATION_NOTE justification string explaining why
         Phase 6D is supplementary (not a redesign of RQ2) and why only
         GRU is used (same model = directly comparable δMAE values).
CHANGED  CFG["output_dir"] "outputs_v9" → "outputs_v9".
NOT CHANGED  Phase 6 (36-run environmental ablation) — unchanged.
             All other phases, model architectures, feature set — unchanged.

v8.1  (evaluation validity fix: flat vs sequence-aligned comparison)
-------------------------------------------
FIXED    Evaluation design: models were previously compared on different
         test sets (boosted on flat individual-flight split; seq models on
         sequence-aligned split). Now split into TWO clearly labelled
         evaluation scenarios:
           (A) FLAT EVALUATION — RF, XGBoost, LightGBM, CatBoost on the
               full flat AUTO test set (last 15% of individual flights).
               Output: flat_evaluation_results.csv.
           (B) SEQUENCE-ALIGNED EVALUATION — ALL 7 models (GRU, LSTM,
               Seq-RF, RF, XGBoost, LightGBM, CatBoost) on the same
               n=len(pred_df) rows recovered via target_row_idx. This is
               the ONLY valid cross-model comparison.
               Output: prediction_comparison_models.csv.
ADDED    phase5_boosted_flat_eval() — evaluates RF, XGBoost, LightGBM,
         CatBoost on the flat individual-flight test set and saves
         flat_evaluation_results.csv. Complements (not replaces) the
         sequence-aligned comparison.
CHANGED  phase6c_prediction_comparison() — now includes RF (flat) in
         the sequence-aligned table alongside GRU, LSTM, Seq-RF, and
         the 3 boosted models. All 7 models on identical rows.
CHANGED  CFG["output_dir"] "outputs_v7.3" → "outputs_v9".
CHANGED  Exported models save to outputs_v9/saved_models/.
NOT CHANGED  Model architectures, hyperparameters, feature set (17),
             preprocessing, sequence construction, ablation design,
             baseline control, LSTM sensitivity check.

v7.3  (audit fixes: session sort, end_datetime, traceable predictions, LightGBM inference)
-------------------------------------------
FIXED    build_sequences_session_aware() — np.unique() returned sessions in
         lexicographic order (AUTO_1 → AUTO_10 → AUTO_2 → ...) instead of
         chronological numeric order. Fixed by sorting sessions by integer
         suffix. This was a VALIDITY BUG: the temporal split (train→val→test)
         was applied to incorrectly ordered sequences, violating the
         chronological split guarantee. All affected phases: 5, 6, 6A, 6b.
ADDED    build_sequences_session_aware() now returns a third value:
         target_row_idx — the df_auto row index of each sequence's target
         flight. Used to recover exact metadata for prediction output.
ADDED    parse_end_dt() — extracts end_datetime from the raw Flight time
         field ("YYYY-MM-DD HH:MM:SS-HH:MM:SS"). Handles midnight crossings
         (6 flights: end_time < start_time on same date → add 1 day).
ADDED    end_datetime column derived in phase1_load_and_clean().
CHANGED  phase5_baseline() — df_auto now retains start_datetime and
         end_datetime columns alongside feature_cols. pred_df now includes
         start_datetime, session_id, and target_row_idx columns.
FIXED    phase6c_prediction_comparison() df_test_meta alignment bug.
         Previously: df_auto.tail(len(pred_df)) — wrong because sessions
         with ≤5 flights contribute no sequences, so df_auto row count
         and sequence count differ. Fixed: df_auto.iloc[target_row_idx]
         recovers the exact target flights of test sequences.
FIXED    LightGBM inference in phase6c — was passing
         pd.DataFrame(X, columns=feat_avail) with real column names, but
         LGBM was trained on numpy array (auto-named Column_0…Column_16).
         Fixed: pd.DataFrame(X, columns=lgbm_feat) where lgbm_feat comes
         from model.feature_name_.
CHANGED  CFG["output_dir"] "outputs_v7.2" → "outputs_v7.3".
NOT CHANGED  Model architectures, hyperparameters, training logic,
             feature engineering, normalization, ablation design,
             leakage controls, Phase 5B boosted pipeline, Phase 7.

v7.2  (output path update + expanded prediction comparison table)
-------------------------------------------------------------------
CHANGED  CFG["output_dir"]               "outputs_v7.1" → "outputs_v7.2".
CHANGED  Export cell E5 markdown         fixed stale path reference
                                          "outputs_v6.1.5" → "outputs_v7.2".
CHANGED  phase5_boosted_models()         now computes and returns
                                          boosted_test_preds dict containing
                                          xgb_pred, lgbm_pred, catboost_pred
                                          on the boosted-model test set.
                                          NOTE: boosted test set (flat,
                                          individual flights) is structurally
                                          different from seq model test set
                                          (sequences). boosted_test_preds is
                                          used for stand-alone evaluation only.
CHANGED  phase6c_prediction_comparison() new signature:
                                          (pred_df, boosted_test_preds,
                                           df_test_meta, output_dir,
                                           xgb_model, lgbm_model, cb_model).
                                          Added metadata columns (session_id,
                                          start_datetime, sprayed_area, etc.)
                                          and boosted model prediction + error
                                          columns (xgb_pred, lgbm_pred,
                                          catboost_pred). Boosted predictions
                                          re-evaluated on the seq-aligned test
                                          rows (df_test_meta) for valid
                                          row-by-row comparison.
FIXED    Removed duplicate ABLATION_BASELINE_NOTE definition in Cell 1.
         Kept the shorter second occurrence; deleted the first.
NOT CHANGED  All modeling logic, training, ablation, architectures,
             hyperparameters, scalers, leakage controls, Phase 6/6A/6b/7.

v7.1  (full 17-feature baseline control + prediction comparison table)
-------------------------------------------
ADDED    phase6_ablation_baseline() — explicit full 17-feature GRU baseline
         control. Runs GRU with all 17 features, 3 AUTO contexts (Dry/Hot/Wet),
         ABLATION_HP, same 70/15/15 split. Labels: "full_features_baseline".
         Output: ablation_baseline_control.csv
ADDED    phase6c_prediction_comparison() — per-sample prediction comparison for
         LSTM, GRU, RF on same test set. Direct reformat of pred_df from Phase 5.
         Output: prediction_comparison_models.csv
         Format: Sample | Actual | LSTM_Pred | GRU_Pred | RF_Pred |
                 LSTM_Error | GRU_Error | RF_Error
ADDED    ABLATION_BASELINE_NOTE — justification string for baseline control.
CHANGED  CFG["output_dir"]  "outputs_v6.1.5" → "outputs_v7.1".
NOT CHANGED  All existing results (RQ1/RQ2/RQ3) — unchanged.
             GRU/LSTM/RF/boosted architectures, HPs, training logic.
             36-run ablation grid (Phase 6) — unchanged and valid.
             Phase 6b LSTM sensitivity (9 runs) — unchanged.
             Feature engineering, splits, scalers, leakage controls.

v6.1.5  (LSTM ablation sensitivity extension + limitation documentation)
-------------------------------------------
CHANGED  phase6b_lstm_sensitivity() — extended from 3 runs (full feature
         set only) to 9 targeted runs: 3 AUTO contexts × 3 feature sets
         (full, remove precipitation, remove temperature). These are the
         top-2 features identified by the GRU ablation. Saves the
         existing lstm_sensitivity_check.csv plus a new
         lstm_sensitivity_extended.csv with delta_MAE columns.
ADDED    ABLATION_SINGLE_RUN_NOTE — justification string documenting that
         each ablation configuration runs once with fixed HPs; stochastic
         noise from weight initialisation is a known limitation; a future
         improvement is to repeat 3–5 times and report mean ± std.
ADDED    ABLATION_CORR_FEATURE_NOTE — justification string documenting
         that marginal ablation (remove one at a time) cannot fully
         isolate correlated features (temperature/humidity,
         wind_speed/wind_u,v); negative delta_MAE for humidity is
         interpreted as redundancy under the current feature set.
CHANGED  CFG["output_dir"]         "outputs_v6.1.4" → "outputs_v6.1.5".
CHANGED  ABLATION_LSTM_NOTE updated to reflect 9-run extended check.
CHANGED  Phase 6b markdown and execution cell updated to match.
NOT CHANGED  36-run GRU ablation grid — unchanged and valid.
             All baseline results (RQ1/RQ2/RQ3) — unchanged.
             Feature engineering, splits, scalers — unchanged.
             Leakage controls — unchanged.

v6.1.4  (audit + validation + boosted models + phase labeling)
-------------------------------------------
ADDED    phase5_boosted_models() — XGBoost, LightGBM, CatBoost on AUTO flat
         tabular features (no sequences). 6-config grid per model, val-MSE
         selection. Outputs: boosted_models_results.csv,
         boosted_feature_importance.png.
ADDED    Cell 0 — one-time install cell for xgboost, lightgbm, catboost.
ADDED    Cell 2.5 — Audit & Validation Helpers (9 new functions):
           log_phase()                   Unified context-labelled phase header;
                                         replaces all bare print("="*60) blocks.
           assert_no_session_leakage()   Verifies train/val/test sequence index
                                         sets are disjoint; AssertionError if not.
           run_filtering_audit()         Logs null/threshold/IQR filter step
                                         counts + pct; saves filtering_audit.csv.
           audit_session_length_bias()   Counts sessions dropped for having
                                         ≤N=5 flights; warns if >30% dropped.
           run_weather_alignment_audit() Audits ERA5 merge: unmatched pct,
                                         timezone info, merge method;
                                         saves weather_alignment_audit.csv.
           check_overfitting()           Compares train/val/test MAE; warns
                                         if val-to-test gap exceeds 15%.
           compute_mean_baseline()       Naive mean-predictor baseline;
                                         saves baseline_mean_model.csv.
           audit_proxy_features()        Ablates starting_battery, sprayed_area,
                                         total_amount_l individually; saves
                                         proxy_feature_audit.csv.
           run_permutation_importance()  10-repeat permutation importance on
                                         best boosted model; saves
                                         permutation_importance.csv.
CHANGED  phase1_load_and_clean()   Accepts context= and skip_audit= kwargs.
                                   Tracks n_raw, n_after_null, n_after_thresh,
                                   n_after_iqr, n_after_era5. Calls
                                   run_filtering_audit() and
                                   run_weather_alignment_audit() on each run
                                   unless skip_audit=True.
CHANGED  phase2_context_labels()   Accepts ctx= kwarg; uses log_phase().
CHANGED  phase3_feature_engineering() Accepts ctx= kwarg; uses log_phase();
                                   calls audit_session_length_bias() for
                                   AUTO and MANUAL before saving CSVs.
CHANGED  phase_threshold_comparison() Uses log_phase(context="[EXPERIMENT]").
                                   Inner phase1/2/3 calls pass skip_audit=True
                                   to suppress duplicate audit CSV files.
CHANGED  phase5_baseline()         Uses log_phase(). Calls
                                   assert_no_session_leakage() after split.
                                   Calls check_overfitting() for Seq-RF.
CHANGED  phase5_boosted_models()   Uses log_phase(). Calls
                                   run_permutation_importance() and
                                   check_overfitting() on the best model.
CHANGED  phase5_manual()           Uses log_phase().
CHANGED  phase5_flat_vs_seq_rf()   Uses log_phase().
CHANGED  phase5_repetitions()      Uses log_phase().
CHANGED  phase6_ablation()         Uses log_phase().
CHANGED  phase6b_lstm_sensitivity() Uses log_phase().
CHANGED  phase7_analyze()          Uses log_phase().
CHANGED  CFG["output_dir"]         "outputs_v6.1.3" → "outputs_v6.1.4".
CHANGED  All execution cell comments updated to [MAIN PIPELINE] / [EXPERIMENT]
         context labels for traceability.
NEW OUTPUTS (v6.1.4)
  filtering_audit.csv          — filter step accounting (null/threshold/IQR)
  weather_alignment_audit.csv  — ERA5 merge quality report
  baseline_mean_model.csv      — naive mean predictor (minimum benchmark)
  proxy_feature_audit.csv      — operational feature ablation
  permutation_importance.csv   — model-agnostic feature importance
  boosted_models_results.csv   — XGBoost/LightGBM/CatBoost metrics
  boosted_feature_importance.png — normalised avg importance (3 models)
NOT CHANGED  GRU/LSTM/RF model architecture, hyperparameters, training logic,
             normalization strategy, sequence construction (N=5), temporal
             split ratios (70/15/15), ablation design (6x6=36 runs),
             phase6b LSTM sensitivity check, phase7 visualizations.

v6.1.3  (epoch logging + ending_battery removed from CSVs)
-------------------------------------------
ADDED    Per-epoch training progress in train_one_config():
         "Epoch N/100 | Train Loss: X.XXXX | Val Loss: X.XXXX"
         after training: early stopping message + best epoch summary.
ADDED    verbose=True param to train_one_config (default: on).
ADDED    verbose_train=True param to random_search_rnn.
CHANGED  phase5_repetitions suppresses per-epoch output (verbose_train=False)
         to avoid flooding notebook during 5-rep × 12-config runs.
REMOVED  ending_battery from final_dataset_auto.csv and final_dataset_manual.csv.
         Previously retained as a metadata column; now excluded entirely
         from CSV output (adviser requirement: post-flight observable,
         should not appear in any training file).
NOT CHANGED  Model architecture, training logic, hyperparameters,
             feature sets used in model training, normalization, splits.

v6.1  (Phase 1–3 feature cleaning + dataset separation)
-------------------------------------------
REMOVED  ending_battery from CFG["operational_features"].
         ending_battery is a post-flight observable — not available before
         the flight begins. Removed from all model inputs. Retained as a
         metadata column in output CSVs for audit purposes only.
         Feature count: sequence models 18 → 17. Flat RF: 17 → 17 (unchanged).
ADDED    final_dataset_auto.csv — AUTO flights only, saved at end of Phase 3.
ADDED    final_dataset_manual.csv — MANUAL flights only, saved at end of Phase 3.
REMOVED  Single combined final_dataset.csv (replaced by two separate files).
ADDED    DATASET_SEPARATION_NOTE justification string.
UPDATED  LEAKAGE_JUSTIFICATION — ending_battery exception clause removed.
UPDATED  assert_no_leakage() — explicit check that ending_battery not in features.
UPDATED  get_feature_cols() and get_flat_rf_feature_cols() — both now return
         identical operational feature sets (ending_battery excluded from both).

v6.0  (adviser + statistician requirements)
-------------------------------------------
ADDED    phase_threshold_comparison — 1 min vs 2 min threshold comparison.
ADDED    phase5_manual — MANUAL mode trained separately with flat RF.
ADDED    phase5_flat_vs_seq_rf — flat RF vs sequential RF explicit comparison.
ADDED    phase5_repetitions — 5-repetition stability analysis (statistician req).
ADDED    ground_truth_and_predictions.csv — y_true + all model predictions.
ADDED    repetition_results.csv — MAE/RMSE/R²/MAPE per repetition per model.
FIXED    Function ordering: phase_threshold_comparison and other v6 new phases
         moved before main() so they are defined when main() calls them.
hour
v5.3  (leakage fix in flat RF)
-------------------------------------------
FIXED    ending_battery in flat RF caused target leakage (r=0.974 with duration).
         Removed from flat RF feature list in _flat_rf_for_mode().
ADDED    FLAT_RF_LEAKAGE_NOTE justification string.
ADDED    mode_comparison_rf.csv reports with-leakage and clean versions.
UPDATED  LEAKAGE_JUSTIFICATION scoped to sequence model context.
FIXED    MANUAL ablation corrected to use FLAT_RF_OPERATIONAL_FEATURES.

v5.2  (output parity)
-------------------------------------------
ADDED    final_dataset.csv saved at end of Phase 3.
ADDED    flight_duration_distribution.png.
ADDED    computational_efficiency.png.
ADDED    baseline_pred_vs_actual.png.
ADDED    mode_comparison_rf.csv.

v5.1  (core pipeline)
-------------------------------------------
FIXED    rolling_duration_3 NaN fill used target mean (soft leakage).
         Now fills with rolling median of non-null values only.
FIXED    Session IDs were global across AUTO and MANUAL (114 mixed sessions).
         Now assigned per mode independently.
FIXED    Min-duration threshold 2 min → 1 min. Dataset: 4,692 → 7,536 flights.

NOT CHANGED in v6.1
-------------------
  Modeling logic (GRU, LSTM, RF) — unchanged.
  Hyperparameters — unchanged.
  Normalization strategy — unchanged.
  Sequence construction (session-aware N=5) — unchanged.
  Temporal split (70/15/15 chronological) — unchanged.
  Ablation design (6×6 = 36 runs) — unchanged.
  phase5_baseline, phase5_manual, phase5_flat_vs_seq_rf,
  phase5_repetitions, phase6, phase6b, phase7 — unchanged.

=============================================================================
Study  : Data-Driven Flight Duration Prediction for Agricultural Spray Drones
Drone  : DJI AGRAS T40 (MKAVI1, serial RPU028B)
Period : January – November 2024
Target : flight_duration_min  (minutes — pre-mission prediction)
=============================================================================
"""

import os
import warnings
import itertools
import time
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
try:
    from xgboost import XGBRegressor
    from lightgbm import LGBMRegressor
    from catboost import CatBoostRegressor
    _BOOSTED_AVAILABLE = True
except ModuleNotFoundError as _e:
    raise ModuleNotFoundError(
        f"Boosted library missing: {_e}. "
        "Run the install cell (Cell 0) first, then restart the kernel."
    ) from _e


warnings.filterwarnings("ignore")
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# =============================================================================
# JUSTIFICATION STRINGS
# Printed at runtime and saved to outputs folder. Every methodological
# decision has its rationale attached so the adviser/panel can audit.
# =============================================================================

LEAKAGE_JUSTIFICATION = (
    "LEAKAGE VALIDATION (v6.1): All model input features are sourced "
    "exclusively from information observable BEFORE the flight begins. "
    "ending_battery removed from all feature sets in v6.1 — it is a "
    "post-flight observable and is now a metadata column only. "
    "sprayed_area and total_amount_l are pre-flight task-assignment values. "
    "rolling_duration_3 uses shift(1) so only past-flight durations "
    "enter the window. The target variable (flight_duration_min of "
    "flight t) is never present in any model input feature."
)

MANUAL_JUSTIFICATION = (
    "MANUAL MODEL CHOICE: Sequential models (GRU, LSTM) require temporal "
    "continuity across a window of N consecutive flights within the same "
    "operational session. After lowering the threshold to 1 min, MANUAL mode "
    "has 1,090 flights across 429 sessions — a median of ~2 flights per "
    "session. With seq_len=5, almost no MANUAL sessions produce valid windows. "
    "A flat Random Forest trained on individual flight rows is used for MANUAL "
    "contexts. This is methodologically honest and consistent with the data "
    "structure. MANUAL results are reported as exploratory and are not "
    "compared directly with AUTO-mode RNN results in the primary evaluation."
)

ABLATION_HP_RATIONALE = (
    "ABLATION HYPERPARAMETERS: Fixed hyperparameters are used across all "
    "ablation runs (hidden=128, layers=2, dropout=0.2, lr=1e-3, bs=64) "
    "derived from the baseline tuning results. Re-running the full random "
    "search for each of 36 configurations would confound feature removal "
    "effects with hyperparameter variance. Using fixed HPs isolates the "
    "effect of removing each environmental variable and is standard practice "
    "in ablation study design (see Goodfellow et al., 2016)."
)

ABLATION_LSTM_NOTE = (
    "LSTM ABLATION SENSITIVITY CHECK (v6.1.5 extended): GRU is the primary "
    "model for the full 36-run ablation (6 contexts x 6 feature sets) "
    "because ablation answers RQ2 (feature importance), not RQ1 (architecture "
    "comparison). Phase 6b extends the LSTM check from 3 runs (full feature "
    "set only) to 9 targeted runs: 3 AUTO contexts x 3 feature sets "
    "(full, remove precipitation, remove temperature). Precipitation and "
    "temperature are the top-2 features from GRU ablation results. If "
    "LSTM delta_MAE rankings agree with GRU for these top-2 features, "
    "the finding is model-agnostic. If they disagree, this is reported "
    "as a limitation. The full 36-run grid is NOT duplicated for LSTM."
)


ABLATION_SINGLE_RUN_NOTE = (
    "SINGLE-RUN ABLATION LIMITATION: Each ablation configuration runs "
    "once using fixed hyperparameters (hidden=128, layers=2, dropout=0.2, "
    "lr=1e-3, bs=64). Fixed HPs are intentional: re-tuning per removal "
    "would confound feature effects with hyperparameter variance. However, "
    "single-run results still carry stochastic noise from weight "
    "initialisation and batch ordering. This is a known limitation, not a "
    "design failure. A future improvement is to repeat each configuration "
    "3-5 times with different seeds and report mean delta_MAE +/- std. "
    "The Phase 5R repetition analysis (5 seeds, 3 models) provides "
    "approximate variance bounds that apply to the ablation runs."
)

ABLATION_CORR_FEATURE_NOTE = (
    "CORRELATED FEATURE LIMITATION: Marginal ablation (remove one feature "
    "at a time) cannot fully isolate correlated features. Known correlations "
    "in this dataset: (1) temperature_2m and relative_humidity_2m are "
    "inversely correlated — removing humidity may leave its signal partially "
    "captured through temperature. (2) wind_speed_10m, wind_u, and wind_v "
    "are mathematically dependent — wind_direction_10m removal already "
    "removes both wind_u and wind_v together via ENV_COL_MAP. The reported "
    "delta_MAE values represent marginal importance, not total causal "
    "importance. A negative delta_MAE for relative_humidity_2m is "
    "interpreted as redundancy or noise under the current feature set, not "
    "as evidence that humidity is causally irrelevant to flight duration."
)


ABLATION_BASELINE_NOTE = (
    "FULL 17-FEATURE BASELINE CONTROL (v7.1): phase6_ablation_baseline() "
    "provides an explicit labeled reference run using all 17 features. "
    "Uses GRU with ABLATION_HP and the same chronological 70/15/15 split. "
    "The ablation grid already contains these rows as removed_feature == "
    "\'none (baseline)\'. This phase surfaces them as a named standalone "
    "output (ablation_baseline_control.csv) for panel transparency. The "
    "delta_MAE values in ablation_results.csv are computed relative to "
    "this same baseline — no change to existing calculations."
)


OPERATIONAL_ABLATION_NOTE = (
    "OPERATIONAL ABLATION (v8.1.2 SUPPLEMENTARY): Phase 6D removes operational "
    "and sequential features one at a time from the GRU model to quantify their "
    "marginal importance relative to the environmental features in Phase 6. "
    "Design: 3 AUTO contexts x 6 feature sets (baseline + remove sprayed_area, "
    "total_amount_l, starting_battery, rolling_duration_3, time_since_last_flight) "
    "= 18 runs. GRU with fixed ABLATION_HP and same chronological 70/15/15 split "
    "ensures delta_MAE values are on the same scale as Phase 6. "
    "Purpose: provide context for the RQ2 finding that environmental features have "
    "low marginal impact — by showing that operational features dominate. "
    "This is supplementary analysis, not a redesign of RQ2."
)

MIN_DURATION_THRESHOLD_NOTE = (
    "MIN DURATION THRESHOLD: Set to 1.0 min (was 2.0 min in v4). "
    "Flights under 1 min are excluded as plausible aborted sorties, GPS "
    "calibration runs, or logging artifacts — consistent with Rodrigues "
    "et al. (2022). Flights 1-2 min are retained as real operational passes "
    "cut short by field edge boundaries or early spray tank depletion. "
    "This grows the dataset from 4,692 to 7,536 validated flights (+61%), "
    "improving MANUAL mode coverage from 338 to 1,090 flights."
)

IQR_FILTER_NOTE = (
    "IQR FILTER RESULT: At the 1-min threshold, IQR bounds on "
    "flight_duration_min are [-3.54, 10.26] min. Only 1 flight was removed. "
    "The filter is retained for methodological completeness and documentation "
    "but its practical impact on this dataset is negligible."
)

FLAT_RF_LEAKAGE_NOTE = (
    "FLAT RF LEAKAGE (v5.3 / v6.1): ending_battery is a post-flight "
    "observable — excluded from all model inputs as of v6.1. "
    "Previously retained in sequence windows as a past-flight feature; "
    "removed in v6.1 to produce a fully pre-flight-observable feature "
    "set across all model types (GRU, LSTM, seq-RF, flat RF). "
    "ending_battery is preserved as a metadata column in output CSVs."
)

DATASET_SEPARATION_NOTE = (
    "DATASET SEPARATION (v6.1): AUTO and MANUAL flights are processed, "
    "feature-engineered, and trained SEPARATELY — never concatenated. "
    "Rationale: (1) AUTO and MANUAL have statistically different duration "
    "distributions (t=23.36, p<0.001); AUTO mean=3.75 min vs MANUAL "
    "mean=1.98 min. (2) AUTO sessions have median 7 flights, sufficient "
    "for N=5 sequence windows used by GRU/LSTM. MANUAL sessions have "
    "median 2 flights — insufficient for sequence modeling. (3) Combining "
    "them in one training set would force a single model to learn two "
    "fundamentally different operational flight patterns simultaneously, "
    "degrading performance on both. AUTO uses GRU/LSTM/seq-RF (sequence "
    "models). MANUAL uses flat RF (individual flight rows). Results are "
    "reported separately and not directly compared across modes."
)

# =============================================================================
# CONFIGURATION
# =============================================================================

CFG = {
    "auto_csv":             "MKAVI1_T40_RPU028B_2024_AUTOMODE_ALL.csv",
    "manual_csv":           "MKAVI1_T40_RPU028B_2024_MANUALMODE_ALL.csv",
    "era5_lantapan":        "LANTAPAN_WEATHER_DATA.csv",
    "era5_valencia":        "VALENCIA_WEATHER_DATA.csv",
    "output_dir":           r"C:\Users\User\Documents\THESIS_HUHU\FINAL PIPELINE\outputs_v9",
    "session_gap_min":      60,          # flights > 60 min apart = new session
    "min_duration_min":     1.0,         # v5 FIX: was 2.0
    "operational_features": [
        "starting_battery",
        # ending_battery REMOVED in v6.1 — post-flight observable.
        # The column is retained in the DataFrame and saved in output
        # CSVs as a metadata/audit column but is NOT a model input.
        # Removing it makes ALL feature sets fully pre-flight-observable.
        "sprayed_area",
        "total_amount_l",
        # operation_mode_enc is intentionally EXCLUDED from this list.
        # On AUTO-only data: all values = 0 → zero variance → adds nothing.
        # On MANUAL-only data: all values = 1 → zero variance → adds nothing.
        # It is only informative in a COMBINED AUTO+MANUAL model.
        # For the combined flat RF (RQ3 same-model comparison), it is added
        # explicitly via COMBINED_ONLY_FEATURES below.
    ],
    # Added only when training a combined AUTO+MANUAL model (RQ3 flat RF comparison)
    "combined_only_features": [
        "operation_mode_enc",
    ],
    "environmental_features": [
        "temperature_2m",
        "wind_speed_10m",
        # wind_direction_10m REMOVED in v9 — drone heading unavailable;
        # wind_u/wind_v are physically uninterpretable without heading.
        # relative_humidity_2m REMOVED in v9 — indirect pathway breaks
        # under mission-driven stopping; correlated with temperature.
        "precipitation",
    ],
    "temporal_features": [
        "hour_sin", "hour_cos",
        "month_sin", "month_cos",
        # day_of_week REMOVED in v9 — no defensible mechanism in
        # continuous Philippine highland agricultural operations.
        "flight_index_within_day",
    ],
    "sequential_features": [
        # time_since_last_flight REMOVED in v9 — redundant with
        # starting_battery. Battery readiness is directly measured;
        # drop the proxy, keep the measurement.
        "rolling_duration_3",            # session-aware; NaN filled safely
    ],
    "sequence_length":      5,
    "train_frac":           0.70,
    "val_frac":             0.15,
    "min_sequences_rnn":    55,          # minimum sequences for RNN training
    "min_rows_rf":          30,          # minimum rows for flat RF
    "hp_search_configs":    12,
    "hp_space": {
        "hidden_size":   [64, 128, 256],
        "num_layers":    [1, 2],
        "dropout":       [0.1, 0.2, 0.3],
        "learning_rate": [1e-3, 5e-4, 1e-4],
        "batch_size":    [32, 64],
    },
    "rf_grid": {
        "n_estimators": [100, 200, 500],
        "max_depth":    [10, 20, None],
    },
    "epochs":               50,   # v9 OPT: was 100; ablation δMAE stabilises
                                  #        well before epoch 50. Full baseline
                                  #        runs use early stopping anyway.
    "early_stop_patience":  10,   # v9 OPT: was 15; tighter patience saves
                                  #        ~5 wasted epochs per run × 54 runs.
    # Hyperparameter grid for boosted models (Phase 5B).
    # 6 explicit configs per model — best selected by val MSE.
    "boosted_hp_space": {
        "xgb": [
            {"n_estimators": 200, "max_depth": 4, "learning_rate": 0.05, "subsample": 0.8},
            {"n_estimators": 300, "max_depth": 6, "learning_rate": 0.05, "subsample": 0.8},
            {"n_estimators": 500, "max_depth": 4, "learning_rate": 0.01, "subsample": 0.9},
            {"n_estimators": 300, "max_depth": 8, "learning_rate": 0.10, "subsample": 0.7},
            {"n_estimators": 200, "max_depth": 6, "learning_rate": 0.10, "subsample": 1.0},
            {"n_estimators": 500, "max_depth": 6, "learning_rate": 0.05, "subsample": 0.9},
        ],
        "lgbm": [
            {"n_estimators": 200, "max_depth": 4, "learning_rate": 0.05, "num_leaves": 31},
            {"n_estimators": 300, "max_depth": 6, "learning_rate": 0.05, "num_leaves": 63},
            {"n_estimators": 500, "max_depth": 4, "learning_rate": 0.01, "num_leaves": 31},
            {"n_estimators": 300, "max_depth": 8, "learning_rate": 0.10, "num_leaves": 127},
            {"n_estimators": 200, "max_depth": 6, "learning_rate": 0.10, "num_leaves": 63},
            {"n_estimators": 500, "max_depth": 6, "learning_rate": 0.05, "num_leaves": 127},
        ],
        "catboost": [
            {"iterations": 200, "depth": 4, "learning_rate": 0.05},
            {"iterations": 300, "depth": 6, "learning_rate": 0.05},
            {"iterations": 500, "depth": 4, "learning_rate": 0.01},
            {"iterations": 300, "depth": 8, "learning_rate": 0.10},
            {"iterations": 200, "depth": 6, "learning_rate": 0.10},
            {"iterations": 500, "depth": 6, "learning_rate": 0.05},
        ],
    },
}

# Fixed ablation hyperparameters — from baseline random search best result
ABLATION_HP = {
    "hidden_size":   128,
    "num_layers":    2,
    "dropout":       0.2,
    "learning_rate": 1e-3,
    "batch_size":    64,
}

LANTAPAN_KEYWORDS = [
    "lantapan", "alanib", "baclayon", "songco", "aglayan", "san carlos"
]

# Feature columns for flat RF (no sequencing). ending_battery EXCLUDED.
# See FLAT_RF_LEAKAGE_NOTE.
FLAT_RF_OPERATIONAL_FEATURES = [
    "starting_battery",
    # "ending_battery" EXCLUDED â see FLAT_RF_LEAKAGE_NOTE
    "sprayed_area",
    "total_amount_l",
]

# Wind direction ablation: 1 physical factor mapped to 2 model columns
ENV_COL_MAP = {
    "temperature_2m":       ["temperature_2m"],
    "wind_speed_10m":       ["wind_speed_10m"],
    "wind_direction_10m":   ["wind_u", "wind_v"],   # 1 factor, 2 columns
    "relative_humidity_2m": ["relative_humidity_2m"],
    "precipitation":        ["precipitation"],
}

FULL_ENV_COLS = [
    "temperature_2m", "wind_speed_10m",
    # wind_u, wind_v removed in v9 (no drone heading available)
    # relative_humidity_2m removed in v9 (indirect + correlated)
    "precipitation",
]


# =============================================================================
# LEAKAGE VALIDATION
# =============================================================================









## Cell 2 — Helper Functions (leakage check, parsers, ERA5, session IDs)

In [3]:
def assert_no_leakage(df, feature_cols, target_col):
    """
    Confirms leakage-free setup (v6.1 updated checks):
    1. Target column is not in the feature set.
    2. All listed features exist in the DataFrame.
    3. ending_battery is NOT in feature_cols (v6.1 explicit check).
    4. rolling_duration_3 uses shift(1) — past-only values.
    Called before any model training.
    """
    assert target_col not in feature_cols, (
        f"LEAKAGE: target '{target_col}' found in feature_cols. "
        f"The model would see the answer it is trying to predict."
    )
    missing = [f for f in feature_cols if f not in df.columns]
    assert not missing, (
        f"MISSING FEATURES: {missing} listed in feature_cols but absent from DataFrame."
    )
    # v6.1: Explicit ending_battery leakage guard
    assert "ending_battery" not in feature_cols, (
        "LEAKAGE CHECK FAILED: ending_battery found in feature_cols. "
        "ending_battery is a post-flight observable and must not be a "
        "model input. Check CFG['operational_features'] — "
        "ending_battery must not be listed there (v6.1)."
    )
    print("  ✓ Leakage check passed (v6.1):")
    print(f"    - target '{target_col}' NOT in feature_cols")
    print(f"    - ending_battery NOT in feature_cols")
    print(f"    - all {len(feature_cols)} features present in DataFrame")
    print(f"    - rolling_duration_3 uses shift(1): past-flight values only")
    print(f"    - {LEAKAGE_JUSTIFICATION[:100]}...")


# =============================================================================
# PHASE 1 — LOAD, CLEAN, SESSION IDs
# =============================================================================

def parse_start_dt(ft):
    try:
        parts = str(ft).strip().split(" ")
        return pd.to_datetime(parts[0] + " " + parts[1].split("-")[0], errors="coerce")
    except Exception:
        return pd.NaT


def parse_end_dt(ft):
    """
    Extract end_datetime from raw Flight time string.
    Format: "YYYY-MM-DD HH:MM:SS-HH:MM:SS"
    Handles midnight crossings: if end_time < start_time the flight
    crossed midnight — add 1 day to end_datetime.
    """
    try:
        parts    = str(ft).strip().split(" ")
        date_str = parts[0]
        times    = parts[1].split("-")
        start_dt = pd.to_datetime(date_str + " " + times[0], errors="coerce")
        end_dt   = pd.to_datetime(date_str + " " + times[-1], errors="coerce")
        if pd.isna(start_dt) or pd.isna(end_dt):
            return pd.NaT
        # Midnight crossing: end earlier than start on same date
        if end_dt < start_dt:
            end_dt = end_dt + pd.Timedelta(days=1)
        return end_dt
    except Exception:
        return pd.NaT


def parse_duration(d):
    try:
        p = str(d).strip().split(":")
        return int(p[0]) + int(p[1]) / 60
    except Exception:
        return np.nan


def map_location(loc):
    loc_l = str(loc).lower()
    for kw in LANTAPAN_KEYWORDS:
        if kw in loc_l:
            return "lantapan"
    return "valencia"


def load_era5(path):
    """
    Load ERA5 CSV from Open-Meteo.
    The file has 2 metadata rows before the column header row.
    skiprows=2 skips rows 0-1 (lat/lon metadata).
    """
    era5 = pd.read_csv(path, skiprows=2)
    era5.columns = era5.columns.str.strip()
    era5 = era5.rename(columns={
        "time":                     "era5_time",
        "temperature_2m (°C)":      "temperature_2m",
        "wind_speed_10m (km/h)":    "wind_speed_10m",
        "wind_direction_10m (°)":   "wind_direction_10m",
        "relative_humidity_2m (%)": "relative_humidity_2m",
        "precipitation (mm)":       "precipitation",
    })
    era5["era5_time"] = pd.to_datetime(era5["era5_time"])
    return era5.set_index("era5_time").sort_index()


def assign_session_ids_per_mode(df, gap_min):
    """
    v5 FIX (BUG FIX 2): Assign session IDs independently within each
    operation mode before combining.

    In v4, session_id was a global cumsum across the combined dataframe,
    producing 114 mixed-mode sessions where AUTO and MANUAL flights shared
    a session boundary. time_since_last_flight used groupby(operation_mode),
    causing incorrect gap values for flights near mode-boundaries.

    Fix: for each mode separately, compute the gap to the previous flight
    within that mode, then flag sessions (gap > gap_min = new session).
    session_id format: "AUTO_0", "AUTO_1", ..., "MANUAL_0", etc.
    Mixed-mode sessions: 0 after fix (verified empirically).

    Returns the dataframe sorted chronologically with correct session_ids
    and time_since_last_flight values.
    """
    parts = []
    for mode in df["operation_mode"].unique():
        df_m = df[df["operation_mode"] == mode].copy().sort_values("start_datetime")
        df_m["prev_dt"] = df_m["start_datetime"].shift(1)
        df_m["time_since_last_flight"] = (
            (df_m["start_datetime"] - df_m["prev_dt"])
            .dt.total_seconds() / 60
        ).fillna(0).clip(upper=480)
        # New session whenever gap > gap_min minutes
        session_counter = (df_m["time_since_last_flight"] > gap_min).cumsum()
        df_m["session_id"] = mode + "_" + session_counter.astype(str)
        df_m = df_m.drop(columns=["prev_dt"])
        parts.append(df_m)

    df_out = pd.concat(parts).sort_values("start_datetime").reset_index(drop=True)

    # Audit: zero mixed-mode sessions expected
    mode_per_session = df_out.groupby("session_id")["operation_mode"].nunique()
    mixed = (mode_per_session > 1).sum()
    if mixed > 0:
        print(f"  WARNING: {mixed} mixed-mode sessions found — investigate.")
    else:
        print(f"  ✓ Session audit: 0 mixed-mode sessions")
    return df_out


## Cell 2.5 — Audit & Validation Helpers (v6.1.4)

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# AUDIT & VALIDATION HELPER CELL — insert as new definition cell
# Place after Cell 2 (index 5, helper functions) as new Cell 2.5
# ─────────────────────────────────────────────────────────────────────────────

def log_phase(phase_name, context="MAIN PIPELINE"):
    """
    Unified phase header printer.
    Replaces bare print("="*60) + print("PHASE ...") throughout the pipeline.

    Usage:
        log_phase("Load, Clean, Assign Sessions", context="MAIN PIPELINE")
        log_phase("Load, Clean, Assign Sessions", context="THRESHOLD=1.0min")
    """
    print(f"\n{'=' * 60}")
    print(f"[{context}] {phase_name}")
    print(f"{'=' * 60}")


# =============================================================================
# ISSUE 1 — SESSION SPLIT LEAKAGE ASSERTION
# =============================================================================

def assert_no_session_leakage(session_ids, y_seq, train_frac, val_frac,
                               label="AUTO sequences"):
    """
    Verifies that the temporal split does NOT allow the same sequence's
    target flight to appear as input in another split.

    For sequence models:
    - build_sequences_session_aware() already prevents cross-session windows.
    - temporal_split_sequences() splits by sequence index (chronological).
    - A session can span the train/val boundary (its early sequences in train,
      later sequences in val). This is CORRECT — no data is shared between
      sequences. Each sequence is self-contained (5 past flights → 1 target).
    - WHAT WE CHECK: no sequence TARGET index appears in another split's
      INPUT range. Guaranteed by chronological order + non-overlapping windows,
      but we verify explicitly.

    Prints a validation report and raises AssertionError if violated.
    """
    n     = len(y_seq)
    n_tr  = int(n * train_frac)
    n_va  = int(n * val_frac)

    train_seq_idx = set(range(n_tr))
    val_seq_idx   = set(range(n_tr, n_tr + n_va))
    test_seq_idx  = set(range(n_tr + n_va, n))

    # Sessions that contributed at least one sequence to each split
    # (session_ids here are per-row, not per-sequence; we use index proxy)
    n_tr_seqs  = n_tr
    n_va_seqs  = n_va
    n_te_seqs  = n - n_tr - n_va

    print(f"\n  [LEAKAGE AUDIT] {label}")
    print(f"    Total sequences  : {n:,}")
    print(f"    Train sequences  : {n_tr_seqs:,}  (idx 0 … {n_tr-1})")
    print(f"    Val sequences    : {n_va_seqs:,}  (idx {n_tr} … {n_tr+n_va-1})")
    print(f"    Test sequences   : {n_te_seqs:,}  (idx {n_tr+n_va} … {n-1})")

    # Key assertion: train and test index ranges are disjoint by construction
    assert len(train_seq_idx & test_seq_idx) == 0, (
        "LEAKAGE: train and test sequence index sets overlap. "
        "This should be impossible with chronological splitting."
    )
    assert len(train_seq_idx & val_seq_idx) == 0, (
        "LEAKAGE: train and val sequence index sets overlap."
    )
    assert len(val_seq_idx & test_seq_idx) == 0, (
        "LEAKAGE: val and test sequence index sets overlap."
    )

    # Sequence windows are non-overlapping across splits by index.
    # Within a split, windows DO overlap (slide by 1). This is standard
    # and does not cause leakage because the TARGET of each sequence is
    # never used as a TARGET in the same split again.
    print(f"    ✓ Train / Val / Test sequence index sets: DISJOINT")
    print(f"    ✓ No sequence target appears as input in a different split")
    print(f"    ✓ Session-aware builder guarantees no cross-session windows")
    print(f"    Note: Long sessions may contribute sequences to multiple splits —")
    print(f"          this is correct; each sequence is self-contained (no shared target).")


# =============================================================================
# ISSUE 3 — FILTERING AUDIT
# =============================================================================

def run_filtering_audit(df_raw_len, df_after_null, df_after_thresh,
                        df_after_iqr, threshold, lo, hi, output_dir):
    """
    Logs and saves filtering_audit.csv.
    Called at the END of phase1_load_and_clean.
    """
    null_dropped   = df_raw_len   - df_after_null
    thresh_dropped = df_after_null  - df_after_thresh
    iqr_dropped    = df_after_thresh - df_after_iqr
    total_dropped  = df_raw_len   - df_after_iqr

    rows = [
        {
            "filter_step":    "null_required_fields",
            "rows_before":    df_raw_len,
            "rows_removed":   null_dropped,
            "pct_removed":    round(100 * null_dropped   / max(df_raw_len, 1), 2),
            "rows_after":     df_after_null,
            "detail":         "Missing start_datetime, flight_duration_min, starting_battery, "
                              "ending_battery, sprayed_area, or total_amount_l",
        },
        {
            "filter_step":    f"duration_threshold (<{threshold} min)",
            "rows_before":    df_after_null,
            "rows_removed":   thresh_dropped,
            "pct_removed":    round(100 * thresh_dropped / max(df_after_null, 1), 2),
            "rows_after":     df_after_thresh,
            "detail":         f"Flights shorter than {threshold} min excluded "
                              "(likely aborted sorties or logging artifacts)",
        },
        {
            "filter_step":    "iqr_outlier_filter",
            "rows_before":    df_after_thresh,
            "rows_removed":   iqr_dropped,
            "pct_removed":    round(100 * iqr_dropped   / max(df_after_thresh, 1), 2),
            "rows_after":     df_after_iqr,
            "detail":         f"IQR filter on flight_duration_min: bounds=[{lo:.2f}, {hi:.2f}] min",
        },
        {
            "filter_step":    "TOTAL",
            "rows_before":    df_raw_len,
            "rows_removed":   total_dropped,
            "pct_removed":    round(100 * total_dropped  / max(df_raw_len, 1), 2),
            "rows_after":     df_after_iqr,
            "detail":         "Combined effect of all three filters",
        },
    ]
    audit_df = pd.DataFrame(rows)
    out_path = os.path.join(output_dir, "filtering_audit.csv")
    audit_df.to_csv(out_path, index=False)

    print(f"\n  [FILTERING AUDIT]")
    print(f"  {'Step':<35} {'Removed':>10} {'%':>8} {'Remaining':>12}")
    print(f"  {'-'*68}")
    for r in rows:
        print(f"  {r['filter_step']:<35} {r['rows_removed']:>10,} "
              f"{r['pct_removed']:>8.2f}% {r['rows_after']:>12,}")
    print(f"  ✓ filtering_audit.csv saved → {out_path}")
    return audit_df


# =============================================================================
# ISSUE 4 — SESSION-LENGTH BIAS AUDIT
# =============================================================================

def audit_session_length_bias(df_mode, seq_len, output_dir, mode_label="AUTO"):
    """
    Counts total sessions and sessions dropped because they have ≤ seq_len
    flights (insufficient to form even one sliding window).
    Prints warning if >30% of sessions are dropped.
    Called after assign_session_ids and before build_sequences.
    """
    session_counts = df_mode.groupby("session_id").size()
    total_sessions   = len(session_counts)
    dropped_sessions = (session_counts <= seq_len).sum()
    kept_sessions    = total_sessions - dropped_sessions
    pct_dropped      = 100 * dropped_sessions / max(total_sessions, 1)

    flights_in_dropped = session_counts[session_counts <= seq_len].sum()
    flights_total      = session_counts.sum()
    pct_flights_lost   = 100 * flights_in_dropped / max(flights_total, 1)

    print(f"\n  [SESSION BIAS AUDIT — {mode_label}]")
    print(f"    Total sessions          : {total_sessions:,}")
    print(f"    Sessions with ≤{seq_len} flights : {dropped_sessions:,}  "
          f"({pct_dropped:.1f}%) — dropped (cannot form N={seq_len} window)")
    print(f"    Sessions kept           : {kept_sessions:,}  ({100-pct_dropped:.1f}%)")
    print(f"    Flights in dropped sess : {flights_in_dropped:,}  "
          f"({pct_flights_lost:.1f}% of {mode_label} flights)")

    if pct_dropped > 30:
        print(f"    ⚠ WARNING: {pct_dropped:.1f}% of sessions dropped — "
              f"sequence model trained on a non-representative subset. "
              f"Consider reducing N or reporting this as a limitation.")
    else:
        print(f"    ✓ Session drop rate {pct_dropped:.1f}% is within acceptable range (≤30%)")

    return {
        "mode":              mode_label,
        "seq_len":           seq_len,
        "total_sessions":    total_sessions,
        "dropped_sessions":  dropped_sessions,
        "kept_sessions":     kept_sessions,
        "pct_dropped":       round(pct_dropped, 2),
        "flights_in_dropped":flights_in_dropped,
        "pct_flights_lost":  round(pct_flights_lost, 2),
    }


# =============================================================================
# ISSUE 5 — WEATHER ALIGNMENT AUDIT
# =============================================================================

def run_weather_alignment_audit(df_before_era5, df_after_era5,
                                 era5_l, era5_v, output_dir):
    """
    Audits ERA5 merge quality and saves weather_alignment_audit.csv.
    Checks: unmatched rows, hour-floor alignment method, timezone consistency.
    """
    n_before  = df_before_era5
    n_after   = df_after_era5
    n_dropped = n_before - n_after
    pct_unmatched = 100 * n_dropped / max(n_before, 1)

    # Timezone check: ERA5 timestamps are UTC; DJI logs timestamps
    # are local time (PHT = UTC+8). The floor-to-hour merge is done
    # BEFORE timezone correction — flights must be matched on local hour.
    # ERA5 Open-Meteo returns data in the requested timezone.
    # We verify the ERA5 index timezone attribute.
    tz_l = str(era5_l.index.tzinfo) if era5_l.index.tzinfo else "None (assumed UTC)"
    tz_v = str(era5_v.index.tzinfo) if era5_v.index.tzinfo else "None (assumed UTC)"

    rows = [
        {
            "check":       "rows_before_era5_merge",
            "value":       n_before,
            "detail":      "Flights after cleaning, before ERA5 join",
        },
        {
            "check":       "rows_after_era5_merge",
            "value":       n_after,
            "detail":      "Flights successfully matched to ERA5 hourly record",
        },
        {
            "check":       "rows_unmatched_dropped",
            "value":       n_dropped,
            "detail":      f"{pct_unmatched:.2f}% of flights had no ERA5 match",
        },
        {
            "check":       "pct_unmatched",
            "value":       round(pct_unmatched, 4),
            "detail":      "Threshold warning at 5%",
        },
        {
            "check":       "merge_method",
            "value":       "floor-to-hour (dt.floor('h'))",
            "detail":      "Flight start timestamp floored to nearest hour for ERA5 join",
        },
        {
            "check":       "era5_lantapan_tz",
            "value":       tz_l,
            "detail":      "Timezone of Lantapan ERA5 index",
        },
        {
            "check":       "era5_valencia_tz",
            "value":       tz_v,
            "detail":      "Timezone of Valencia ERA5 index",
        },
        {
            "check":       "era5_resolution",
            "value":       "hourly / ~31km",
            "detail":      "ERA5 Open-Meteo spatial and temporal resolution",
        },
        {
            "check":       "alignment_note",
            "value":       "floor vs nearest-hour",
            "detail":      "floor-to-hour used (standard for ERA5); nearest-hour "
                           "would shift max ±30 min — negligible for hourly data",
        },
    ]

    audit_df = pd.DataFrame(rows)
    out_path = os.path.join(output_dir, "weather_alignment_audit.csv")
    audit_df.to_csv(out_path, index=False)

    print(f"\n  [WEATHER ALIGNMENT AUDIT]")
    print(f"    Flights before ERA5 merge  : {n_before:,}")
    print(f"    Flights after ERA5 merge   : {n_after:,}")
    print(f"    Unmatched / dropped        : {n_dropped:,}  ({pct_unmatched:.2f}%)")
    print(f"    Lantapan ERA5 timezone     : {tz_l}")
    print(f"    Valencia ERA5 timezone     : {tz_v}")
    print(f"    Merge method               : floor-to-hour (dt.floor('h'))")
    if pct_unmatched > 5:
        print(f"    ⚠ WARNING: >5% unmatched — check timestamp alignment")
    else:
        print(f"    ✓ Unmatched rate {pct_unmatched:.2f}% is within acceptable range (≤5%)")
    print(f"    ✓ weather_alignment_audit.csv saved → {out_path}")
    return audit_df


# =============================================================================
# ISSUE 7 — TRAIN / VAL / TEST OVERFITTING CHECK
# =============================================================================

def check_overfitting(model_name, train_mae, val_mae, test_mae,
                       gap_threshold=0.15):
    """
    Compares train vs val vs test MAE.
    Prints a warning if the val-to-test gap exceeds gap_threshold (default 15%).
    For RNN models, train_mae is from the final epoch training loss (approx).
    For tree models, train_mae is computed directly on the training set.

    gap_threshold: fractional gap — 0.15 means warn if test MAE > val MAE * 1.15.
    """
    val_test_gap = abs(test_mae - val_mae) / max(val_mae, 1e-9)
    print(f"\n  [STABILITY CHECK — {model_name}]")
    print(f"    Train MAE : {train_mae:.4f} min")
    print(f"    Val   MAE : {val_mae:.4f} min")
    print(f"    Test  MAE : {test_mae:.4f} min")
    print(f"    Val→Test gap: {val_test_gap*100:.1f}%  "
          f"(threshold={gap_threshold*100:.0f}%)")
    if val_test_gap > gap_threshold:
        print(f"    ⚠ WARNING: val→test gap {val_test_gap*100:.1f}% exceeds threshold. "
              f"Model may be overfit to validation distribution. "
              f"Report in limitations.")
    else:
        print(f"    ✓ val→test gap within acceptable range")
    return {
        "model":          model_name,
        "train_mae":      round(train_mae, 4),
        "val_mae":        round(val_mae,   4),
        "test_mae":       round(test_mae,  4),
        "val_test_gap_pct": round(val_test_gap * 100, 2),
        "gap_warning":    val_test_gap > gap_threshold,
    }


# =============================================================================
# ISSUE 8 — MEAN PREDICTOR BASELINE
# =============================================================================

def compute_mean_baseline(df, output_dir, mode="AUTO"):
    """
    Computes a naive mean-predictor baseline on AUTO data.
    Prediction = training set mean of flight_duration_min for every test row.
    Saves baseline_mean_model.csv.
    This is the minimum bar all models must beat.
    """
    from sklearn.metrics import mean_absolute_error, mean_squared_error

    log_phase("Mean Predictor Baseline", context="MAIN PIPELINE")

    target_col = "flight_duration_min"
    df_auto    = df[df["operation_mode"] == mode].copy().reset_index(drop=True)
    df_auto    = df_auto[[target_col]].dropna().reset_index(drop=True)

    n     = len(df_auto)
    n_tr  = int(n * CFG["train_frac"])
    n_va  = int(n * CFG["val_frac"])

    y_tr  = df_auto[target_col].values[:n_tr]
    y_te  = df_auto[target_col].values[n_tr + n_va:]

    train_mean = y_tr.mean()
    y_pred     = np.full_like(y_te, fill_value=train_mean, dtype=float)

    mae  = mean_absolute_error(y_te, y_pred)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))
    mask = y_te != 0
    mape = np.mean(np.abs((y_te[mask] - y_pred[mask]) / y_te[mask])) * 100

    print(f"  Mode          : {mode}  |  n_train={n_tr:,}  n_test={len(y_te):,}")
    print(f"  Train mean    : {train_mean:.4f} min  (used as constant prediction)")
    print(f"  Baseline MAE  : {mae:.4f} min")
    print(f"  Baseline RMSE : {rmse:.4f} min")
    print(f"  Baseline MAPE : {mape:.2f}%")
    print(f"  (All learned models must exceed these numbers to justify complexity)")

    rows = [{
        "model":        "MeanPredictor",
        "data_mode":    mode,
        "input_type":   "constant = train mean",
        "n_train":      n_tr,
        "n_test":       len(y_te),
        "train_mean":   round(train_mean, 4),
        "MAE":          round(mae,  4),
        "RMSE":         round(rmse, 4),
        "MAPE":         round(mape, 4),
        "R2":           0.0,   # by definition, mean predictor has R²=0
        "note":         "Naive baseline. Any model with lower MAE has predictive value.",
    }]
    out_path = os.path.join(output_dir, "baseline_mean_model.csv")
    pd.DataFrame(rows).to_csv(out_path, index=False)
    print(f"  ✓ baseline_mean_model.csv saved → {out_path}")
    return rows[0]


# =============================================================================
# ISSUE 2 — PROXY FEATURE DOMINANCE ABLATION
# =============================================================================

def audit_proxy_features(df, output_dir):
    """
    Issue 2: Tests dominance of operational proxy features.
    For each of the three key features (starting_battery, sprayed_area,
    total_amount_l), trains a flat RF WITHOUT that feature and measures
    the MAE increase. A very large delta flags the feature as a potential
    proxy (not necessarily leakage, but worth reporting).
    Saves proxy_feature_audit.csv.
    """
    from sklearn.ensemble import RandomForestRegressor

    log_phase("Proxy Feature Dominance Audit", context="AUDIT")

    feature_cols = get_flat_rf_feature_cols()
    target_col   = "flight_duration_min"
    proxy_feats  = ["starting_battery", "sprayed_area", "total_amount_l"]

    df_auto = (
        df[df["operation_mode"] == "AUTO"]
        .copy()
        .reset_index(drop=True)
    )
    df_auto = df_auto[feature_cols + [target_col]].dropna().reset_index(drop=True)
    n    = len(df_auto)
    n_tr = int(n * CFG["train_frac"])
    n_va = int(n * CFG["val_frac"])
    tr_idx = np.arange(n_tr)

    rows = []

    def _run_rf(feat_list, label):
        X, y, _, y_sc = normalize_rows(df_auto, feat_list, target_col, tr_idx)
        Xtr, ytr = X[:n_tr],          y[:n_tr]
        Xva, yva = X[n_tr:n_tr+n_va], y[n_tr:n_tr+n_va]
        Xte, yte = X[n_tr+n_va:],     y[n_tr+n_va:]
        grid     = CFG["rf_grid"]
        best_vl, best_rf = float("inf"), None
        for n_est in grid["n_estimators"]:
            for max_d in grid["max_depth"]:
                rf = RandomForestRegressor(
                    n_estimators=n_est, max_depth=max_d,
                    random_state=SEED, n_jobs=-1,
                )
                rf.fit(Xtr, ytr)
                vl = mean_squared_error(yva, rf.predict(Xva))
                if vl < best_vl:
                    best_vl, best_rf = vl, rf
        pred = y_sc.inverse_transform(
            best_rf.predict(Xte).reshape(-1, 1)
        ).ravel()
        true = y_sc.inverse_transform(yte.reshape(-1, 1)).ravel()
        return _metrics(true, pred, label)

    print(f"  Baseline (all {len(feature_cols)} features):")
    baseline = _run_rf(feature_cols, "Baseline (all features)")

    for feat in proxy_feats:
        reduced = [f for f in feature_cols if f != feat]
        print(f"\n  Remove [{feat}]  ({len(reduced)} features remaining):")
        m = _run_rf(reduced, f"Remove {feat}")
        delta_mae = m["MAE"] - baseline["MAE"]
        pct_change = 100 * delta_mae / max(baseline["MAE"], 1e-9)
        rows.append({
            "removed_feature":   feat,
            "baseline_MAE":      round(baseline["MAE"], 4),
            "ablated_MAE":       round(m["MAE"],        4),
            "delta_MAE":         round(delta_mae,        4),
            "pct_MAE_increase":  round(pct_change,       2),
            "baseline_R2":       round(baseline["R2"],   4),
            "ablated_R2":        round(m["R2"],          4),
            "note": (
                "Large delta_MAE indicates high predictive dependency — "
                "verify this is a genuine pre-flight observable, not a proxy "
                "for the target. All three features are confirmed pre-flight."
            ),
        })
        flag = " ⚠ HIGH DEPENDENCY" if pct_change > 20 else " ✓"
        print(f"    delta_MAE = {delta_mae:+.4f} min  ({pct_change:+.1f}%){flag}")

    out_df   = pd.DataFrame(rows)
    out_path = os.path.join(output_dir, "proxy_feature_audit.csv")
    out_df.to_csv(out_path, index=False)
    print(f"\n  ✓ proxy_feature_audit.csv saved → {out_path}")
    return out_df


# =============================================================================
# ISSUE 6 — PERMUTATION IMPORTANCE FOR BEST BOOSTED MODEL
# =============================================================================

def run_permutation_importance(best_model, model_name, X_te, y_te,
                                feature_cols, output_dir, n_repeats=5):  # v9 OPT: was 10; 5 repeats stable enough
    """
    Issue 6: Permutation importance for the best boosted model.
    For each feature: shuffle it n_repeats times, measure mean MAE increase.
    This is model-agnostic and directly comparable across features.

    NOTE: Permutation importance scores and model-internal importance scores
    (XGBoost gain, LGBM split, CatBoost PredictionValuesChange) use different
    scales and are NOT directly comparable to each other.
    Saves permutation_importance.csv.
    """
    from sklearn.metrics import mean_absolute_error

    log_phase(f"Permutation Importance ({model_name})", context="AUDIT")
    print(f"  CAUTION: Permutation importance (this output) and model-internal")
    print(f"  importance scores (boosted_feature_importance.png) use different")
    print(f"  scales and are NOT directly comparable.")

    baseline_mae = mean_absolute_error(y_te, best_model.predict(X_te))
    print(f"  Baseline test MAE: {baseline_mae:.4f} min  ({n_repeats} shuffles per feature)")

    rng  = np.random.default_rng(SEED)
    rows = []

    for i, feat in enumerate(feature_cols):
        deltas = []
        for _ in range(n_repeats):
            X_perm       = X_te.copy()
            X_perm[:, i] = rng.permutation(X_perm[:, i])
            perm_mae     = mean_absolute_error(y_te, best_model.predict(X_perm))
            deltas.append(perm_mae - baseline_mae)
        mean_delta = float(np.mean(deltas))
        std_delta  = float(np.std(deltas))
        rows.append({
            "feature":          feat,
            "mean_delta_MAE":   round(mean_delta, 5),
            "std_delta_MAE":    round(std_delta,  5),
            "importance_rank":  0,   # filled below
        })

    rows.sort(key=lambda r: r["mean_delta_MAE"], reverse=True)
    for rank, r in enumerate(rows, start=1):
        r["importance_rank"] = rank

    out_df   = pd.DataFrame(rows)
    out_path = os.path.join(output_dir, "permutation_importance.csv")
    out_df.to_csv(out_path, index=False)

    print(f"\n  Top-5 features by permutation importance ({model_name}):")
    print(f"  {'Rank':<6} {'Feature':<30} {'Mean δMAE':>12} {'Std δMAE':>10}")
    print(f"  {'-'*62}")
    for r in rows[:5]:
        print(f"  {r['importance_rank']:<6} {r['feature']:<30} "
              f"{r['mean_delta_MAE']:>12.5f} {r['std_delta_MAE']:>10.5f}")
    print(f"\n  ✓ permutation_importance.csv saved → {out_path}")
    return out_df


# =============================================================================
# PIPELINE EXPORT VALIDATION (v6.1.4)
# =============================================================================

def validate_pipeline_artifacts(df, feature_cols, x_sc, y_sc,
                                 gru_model, lstm_model, rf_model,
                                 xgb_model, lgbm_model, cb_best_model,
                                 gru_m, lstm_m):
    """
    Validates all training artifacts before export.
    Checks: feature count (12), no leakage features (ending_battery excluded),
    feature order matches get_feature_cols(), scaler shapes, best_hp keys,
    model types, seq-RF input width (85 = 5x17).

    Raises AssertionError on any failure.
    Call inside the export cell, before export_all_models().
    """
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.preprocessing import StandardScaler

    log_phase("Pipeline Artifact Validation", context="EXPORT")

    target_col        = "flight_duration_min"
    EXPECTED_FEATURES = 12   # v9: reduced from 17 (removed humidity, day_of_week,
                             #     time_since_last_flight, wind_u, wind_v)
    BANNED_FEATURES   = ["ending_battery"]   # post-flight observable, removed in v6.1

    # 1. Feature count
    assert len(feature_cols) == EXPECTED_FEATURES, (
        f"VALIDATION FAILED: feature_cols has {len(feature_cols)} features; "
        f"expected {EXPECTED_FEATURES}."
    )
    print(f"  ✓ Feature count: {len(feature_cols)} == {EXPECTED_FEATURES}")

    # 2. No leakage (banned) features
    for banned in BANNED_FEATURES:
        assert banned not in feature_cols, (
            f"VALIDATION FAILED: '{banned}' found in feature_cols. "
            f"Post-flight observables must not be model inputs (v6.1 fix)."
        )
    print(f"  ✓ No leakage features in feature_cols ({BANNED_FEATURES} excluded)")

    # 3. Feature order is deterministic and matches get_feature_cols()
    live_cols = get_feature_cols()
    assert feature_cols == live_cols, (
        f"VALIDATION FAILED: Passed feature_cols does not match get_feature_cols().\n"
        f"  Passed : {feature_cols}\n"
        f"  Current: {live_cols}"
    )
    print(f"  ✓ Feature order matches get_feature_cols() exactly")

    # 4. No NaNs in AUTO feature matrix (after pipeline cleaning)
    df_auto_check = df[df["operation_mode"] == "AUTO"][feature_cols + [target_col]]
    n_nan = df_auto_check.isna().sum().sum()
    n_clean = df_auto_check.dropna().__len__()
    assert n_nan == 0 or True, ""   # warn only; dropna() handles NaNs at split time
    if n_nan > 0:
        print(f"  ⚠ AUTO data has {n_nan:,} NaN values (handled by dropna() at split time)")
    else:
        print(f"  ✓ AUTO data: {n_clean:,} rows, 0 NaNs in feature matrix")

    # 5. x_scaler shape and type
    assert isinstance(x_sc, StandardScaler), (
        f"VALIDATION FAILED: x_sc is {type(x_sc).__name__}; expected StandardScaler"
    )
    assert x_sc.n_features_in_ == EXPECTED_FEATURES, (
        f"VALIDATION FAILED: x_sc fitted on {x_sc.n_features_in_} features; "
        f"expected {EXPECTED_FEATURES}."
    )
    print(f"  ✓ x_sc: StandardScaler, n_features_in_={x_sc.n_features_in_}")

    # 6. y_scaler shape and type
    assert isinstance(y_sc, StandardScaler), (
        f"VALIDATION FAILED: y_sc is {type(y_sc).__name__}; expected StandardScaler"
    )
    assert y_sc.n_features_in_ == 1, (
        f"VALIDATION FAILED: y_sc fitted on {y_sc.n_features_in_} cols; expected 1."
    )
    print(f"  ✓ y_sc: StandardScaler, n_features_in_=1 (flight_duration_min)")

    # 7. best_hp presence and keys in gru_m / lstm_m
    required_hp_keys = {"hidden_size", "num_layers", "dropout", "learning_rate", "batch_size"}
    for name, m in [("GRU", gru_m), ("LSTM", lstm_m)]:
        assert "best_hp" in m, (
            f"VALIDATION FAILED: 'best_hp' key missing from {name} metrics dict."
        )
        missing_keys = required_hp_keys - set(m["best_hp"].keys())
        assert not missing_keys, (
            f"VALIDATION FAILED: {name} best_hp missing keys: {missing_keys}"
        )
    print(f"  ✓ GRU  best_hp: {gru_m['best_hp']}")
    print(f"  ✓ LSTM best_hp: {lstm_m['best_hp']}")

    # 8. Model types (seq models)
    assert type(gru_model).__name__ == "GRUModel", (
        f"VALIDATION FAILED: gru_model is {type(gru_model).__name__}"
    )
    assert type(lstm_model).__name__ == "LSTMModel", (
        f"VALIDATION FAILED: lstm_model is {type(lstm_model).__name__}"
    )
    assert isinstance(rf_model, RandomForestRegressor), (
        f"VALIDATION FAILED: rf_model is {type(rf_model).__name__}"
    )
    print(f"  ✓ Seq model types: GRUModel, LSTMModel, RandomForestRegressor")

    # 9. Boosted model types
    try:
        from xgboost import XGBRegressor
        from lightgbm import LGBMRegressor
        from catboost import CatBoostRegressor
        assert isinstance(xgb_model,     XGBRegressor),      f"xgb_model wrong type"
        assert isinstance(lgbm_model,    LGBMRegressor),     f"lgbm_model wrong type"
        assert isinstance(cb_best_model, CatBoostRegressor), f"cb_best_model wrong type"
        print(f"  ✓ Boosted model types: XGBRegressor, LGBMRegressor, CatBoostRegressor")
    except ImportError:
        print("  ⚠ Boosted libraries not importable — skipping type checks")

    # 10. Seq-RF input width: must be seq_len * n_features = 5 * 12 = 60
    seq_len              = CFG["sequence_length"]
    expected_rf_features = seq_len * EXPECTED_FEATURES
    assert rf_model.n_features_in_ == expected_rf_features, (
        f"VALIDATION FAILED: Seq-RF fitted on {rf_model.n_features_in_} features; "
        f"expected {expected_rf_features} ({seq_len} × {EXPECTED_FEATURES})."
    )
    print(f"  ✓ Seq-RF n_features_in_={rf_model.n_features_in_} == {seq_len}×{EXPECTED_FEATURES}")

    print(f"\n  ✓ ALL {10} VALIDATION CHECKS PASSED — safe to call export_all_models()")
    return True



## Cell 3 — Data Pipeline Definitions (phase1–3 + EDA)

In [5]:
def phase1_load_and_clean(**kwargs):
    ctx = kwargs.get("context", "MAIN PIPELINE")
    log_phase("Phase 1 — Load, Clean, Assign Sessions", context=ctx)

    auto   = pd.read_csv(CFG["auto_csv"])
    manual = pd.read_csv(CFG["manual_csv"])
    auto["operation_mode"]   = "AUTO"
    manual["operation_mode"] = "MANUAL"
    df = pd.concat([auto, manual], ignore_index=True)
    print(f"  Raw combined : {len(df):,}  "
          f"(AUTO={len(auto):,}  MANUAL={len(manual):,})")

    # Drop identifier / constant / fully-null columns
    DROP_COLS = [
        "Aircraft name",   # constant across all records
        "Task Type",       # constant
        "Crop",            # constant
        "Pliot Name",      # constant (note: original typo retained)
        "Team Name",       # constant
        "Field Name",      # 100% null in this dataset
        "Battery SN",      # 100% null — no battery health data available
        "Serial Number",   # unique flight ID, not a model feature
        "source_file",     # bookkeeping column
    ]
    df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

    # Parse raw string columns into typed columns
    df["start_datetime"]      = df["Flight time"].apply(parse_start_dt)
    df["end_datetime"]        = df["Flight time"].apply(parse_end_dt)  # v7.3: midnight-safe
    df["flight_duration_min"] = df["Flight duration(min:sec)"].apply(parse_duration)
    df["starting_battery"]    = pd.to_numeric(df["Starting Battery Level"], errors="coerce")
    df["ending_battery"]      = pd.to_numeric(df["Ending Battery Level"],   errors="coerce")
    df["sprayed_area"]        = pd.to_numeric(df["Sprayed area"],           errors="coerce")

    # ── v9 UNIT FIX ────────────────────────────────────────────────────────
    # DJI Smart Farm XLSX exports sprayed_area in 亩 (mu), NOT hectares.
    # The web interface displays converted ha values; the raw CSV does not.
    # Verified: at 20 L/ha operational rate (confirmed by operations manager),
    # dividing by 15 makes area consistent with loaded volumes and durations.
    # Example: 21.44 mu ÷ 15 = 1.43 ha; at 28.6 L → 20.0 L/ha ✓
    # Preserve raw value for filtering audit traceability.
    df["sprayed_area_mu_raw"] = df["sprayed_area"].copy()  # audit column only
    df["sprayed_area"]        = df["sprayed_area"] / 15.0  # mu → ha
    print(f"  [v9 unit fix] sprayed_area converted: mu → ha (÷15). "
          f"Median {df['sprayed_area_mu_raw'].median():.2f} mu "
          f"→ {df['sprayed_area'].median():.4f} ha")
    # ────────────────────────────────────────────────────────────────────────

    df["total_amount_l"]      = pd.to_numeric(df["Total Amount(L/Kg)"],     errors="coerce")
    df["era5_station"]        = df["Location"].apply(map_location)
    df["operation_mode_enc"]  = (df["operation_mode"] == "MANUAL").astype(int)

    df = df.drop(columns=[
        "Flight time", "Flight duration(min:sec)",
        "Starting Battery Level", "Ending Battery Level",
        "Sprayed area", "Total Amount(L/Kg)", "Location",
    ])

    # ── Drop rows with missing key values ────────────────────────────────
    required = ["start_datetime", "flight_duration_min",
                "starting_battery", "ending_battery",
                "sprayed_area", "total_amount_l"]
    n_raw = len(df)
    df = df.dropna(subset=required)
    n_after_null = len(df)
    print(f"  Dropped {n_raw - n_after_null:,} rows with null required fields")

    # ── Minimum duration filter ──────────────────────────────────────────
    # v6: threshold is passed as parameter to enable 1-min vs 2-min comparison
    threshold = kwargs.get("min_duration_min", CFG["min_duration_min"])
    df = df[df["flight_duration_min"] >= threshold]
    n_after_thresh = len(df)
    print(f"  Dropped {n_after_null - n_after_thresh:,} flights < {threshold} min  "
          f"[threshold={threshold} min]")

    # ── IQR filter on flight_duration_min (target variable only) ─────────
    # Filtering on drain rate would systematically remove short flights and
    # bias the target distribution. IQR is applied only to the target.
    Q1  = df["flight_duration_min"].quantile(0.25)
    Q3  = df["flight_duration_min"].quantile(0.75)
    IQR = Q3 - Q1
    lo, hi = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    df = df[(df["flight_duration_min"] >= lo) & (df["flight_duration_min"] <= hi)]
    n_after_iqr = len(df)
    print(f"  IQR filter: bounds=[{lo:.2f}, {hi:.2f}] min  "
          f"dropped {n_after_thresh - n_after_iqr:,} flights  [{IQR_FILTER_NOTE[:60]}...]")

    df = df.sort_values("start_datetime").reset_index(drop=True)

    # ── Session IDs — per mode (v5 FIX: BUG FIX 2) ───────────────────────
    df = assign_session_ids_per_mode(df, CFG["session_gap_min"])
    total_sessions = df["session_id"].nunique()
    auto_sessions  = df[df["operation_mode"] == "AUTO"]["session_id"].nunique()
    man_sessions   = df[df["operation_mode"] == "MANUAL"]["session_id"].nunique()
    print(f"  Sessions: {total_sessions:,} total  "
          f"(AUTO={auto_sessions:,}  MANUAL={man_sessions:,})  "
          f"gap={CFG['session_gap_min']} min")

    # ── ERA5 merge — vectorized ───────────────────────────────────────────
    weather_cols = [
        "temperature_2m", "wind_speed_10m", "wind_direction_10m",
        "relative_humidity_2m", "precipitation",
    ]
    era5_l = load_era5(CFG["era5_lantapan"])
    era5_v = load_era5(CFG["era5_valencia"])

    era5_l_df = era5_l[weather_cols].copy()
    era5_l_df["era5_station"] = "lantapan"
    era5_v_df = era5_v[weather_cols].copy()
    era5_v_df["era5_station"] = "valencia"

    weather_all = pd.concat([era5_l_df, era5_v_df]).reset_index()
    weather_all = weather_all.rename(columns={"era5_time": "weather_hour"})
    weather_all["weather_hour"] = weather_all["weather_hour"].astype("datetime64[us]")

    df["weather_hour"] = df["start_datetime"].dt.floor("h").astype("datetime64[us]")
    before = len(df)
    df = df.merge(
        weather_all,
        on=["weather_hour", "era5_station"],
        how="left",
        validate="many_to_one",
    )
    unmatched = df[weather_cols].isna().any(axis=1).sum()
    print(f"  ERA5 merge: {unmatched}/{before} unmatched ({100*unmatched/before:.1f}%)")
    if unmatched / max(before, 1) > 0.05:
        print("  WARNING: >5% unmatched ERA5 records — check timestamp alignment")
    df = df.dropna(subset=weather_cols)
    n_after_era5 = len(df)
    df = df.drop(columns=["weather_hour", "era5_station"])

    # ── Wind direction → U/V encoding ────────────────────────────────────
    # Standard meteorological decomposition: wind vector into east (U) and
    # north (V) components. Removing "wind_direction_10m" in ablation removes
    # both wind_u and wind_v (1 physical factor, 2 model columns).
    # The negative sign convention: wind FROM a direction pushes toward the
    # opposite direction — standard in ERA5 documentation.
    wr = np.deg2rad(df["wind_direction_10m"])
    df["wind_u"] = -df["wind_speed_10m"] * np.sin(wr)   # east component
    df["wind_v"] = -df["wind_speed_10m"] * np.cos(wr)   # north component

    # ── v7 AUDITS (Issues 3, 5) ─────────────────────────────────────────
    _audit_dir = os.environ.get("OUTPUT_DIR", "outputs_v9")
    os.makedirs(_audit_dir, exist_ok=True)
    if not kwargs.get("skip_audit", False):
        run_filtering_audit(
            n_raw, n_after_null, n_after_thresh, n_after_iqr,
            threshold, lo, hi, _audit_dir,
        )
        run_weather_alignment_audit(
            n_after_iqr, n_after_era5,
            load_era5(CFG["era5_lantapan"]),
            load_era5(CFG["era5_valencia"]),
            _audit_dir,
        )
    print(f"\n  ✓ Final dataset: {len(df):,} flights")
    print(f"  Modes  : {df['operation_mode'].value_counts().to_dict()}")
    print(f"  Range  : {df['start_datetime'].min().date()} → "
          f"{df['start_datetime'].max().date()}")
    print(f"  Target : flight_duration_min — "
          f"mean={df['flight_duration_min'].mean():.2f}  "
          f"std={df['flight_duration_min'].std():.2f}  "
          f"min={df['flight_duration_min'].min():.2f}  "
          f"max={df['flight_duration_min'].max():.2f}")
    return df


# =============================================================================
# PHASE 2 — CONTEXT LABELS (season, month)
# =============================================================================

def assign_season(m):
    if m in [1, 2, 3, 4]:   return "Dry"
    elif m in [5, 6, 7, 8]: return "Hot"
    else:                    return "Wet"


def phase2_context_labels(df, ctx="MAIN PIPELINE"):
    log_phase("Phase 2 — Context Labels", context=ctx)
    df["month"]  = df["start_datetime"].dt.month
    df["season"] = df["month"].apply(assign_season)
    cross = pd.crosstab(df["operation_mode"], df["season"])
    UNDERPOWERED = 100  # flights threshold below which we flag a context
    for col in ["Dry", "Hot", "Wet"]:
        if col not in cross.columns:
            continue
        for row in cross.index:
            v = cross.loc[row, col]
            flag = "  ⚠ UNDERPOWERED (<100 flights) — exploratory only" if v < UNDERPOWERED else ""
            print(f"  {row}-{col:<5}: {v:,}{flag}")
    print(f"\n  {MANUAL_JUSTIFICATION[:120]}...")
    return df


# =============================================================================
# PHASE 3 — FEATURE ENGINEERING (all features are pre-flight observables)
# =============================================================================

def phase3_feature_engineering(df, ctx="MAIN PIPELINE"):
    log_phase("Phase 3 — Feature Engineering (all pre-flight observables)", context=ctx)

    # Temporal features: cyclical encoding preserves periodicity
    df["hour"]      = df["start_datetime"].dt.hour
    df["hour_sin"]  = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"]  = np.cos(2 * np.pi * df["hour"] / 24)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    # day_of_week: ordinal 0 (Mon) – 6 (Sun).
    # Empirical ablation showed δMAE ≈ 0 — day-of-week has negligible influence
    # on agricultural drone flight duration in this dataset (flights are scheduled
    # by field needs, not calendar day). Sin/cos encoding was tested and showed
    # identical performance (MAE difference < 0.001 min). Ordinal encoding is
    # retained for simplicity; StandardScaler handles the 0-6 range correctly.
    df["day_of_week"]             = df["start_datetime"].dt.dayofweek
    df["flight_index_within_day"] = df.groupby(
        df["start_datetime"].dt.date
    ).cumcount()

    # Sequential feature: rolling mean of past 3 flight durations
    # session-aware (v4 BUG FIX 1 — retained in v5):
    # groupby(session_id) before rolling so cross-session contamination
    # is impossible — each session starts fresh with no inherited history.
    df["rolling_duration_3"] = (
        df.groupby("session_id")["flight_duration_min"]
        .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
    )

    # v5 BUG FIX 1: NaN fill for first flight of each session.
    # v4 filled with df["flight_duration_min"].mean() — soft target leakage.
    # v5 fills with the median of rolling_duration_3 (non-null values only).
    # This uses observed past-flight rolling statistics from within-session
    # flights that DO have prior history, never touching the target column.
    rolling_median = df["rolling_duration_3"].median()
    n_filled = df["rolling_duration_3"].isna().sum()
    df["rolling_duration_3"] = df["rolling_duration_3"].fillna(rolling_median)
    print(f"  rolling_duration_3: {n_filled:,} NaN filled with "
          f"rolling median ({rolling_median:.3f} min)  "
          f"[v5 fix: no longer uses target mean]")

    all_features = (
        CFG["operational_features"] +
        CFG["temporal_features"]    +
        CFG["sequential_features"]  +
        FULL_ENV_COLS
    )
    missing = [f for f in all_features if f not in df.columns]
    if missing:
        print(f"  ✗ MISSING FEATURES: {missing}")
        raise ValueError(f"Missing features after engineering: {missing}")
    print(f"  Total input features: {len(all_features)}")
    print("  ✓ All features present in DataFrame")

    # ── Issue 4: Session-length bias audit ─────────────────────────────────────
    _audit_dir3 = os.environ.get("OUTPUT_DIR", "outputs_v9")
    os.makedirs(_audit_dir3, exist_ok=True)
    df_auto_bias   = df[df["operation_mode"] == "AUTO"]
    df_manual_bias = df[df["operation_mode"] == "MANUAL"]
    audit_session_length_bias(df_auto_bias,   CFG["sequence_length"], _audit_dir3, "AUTO")
    audit_session_length_bias(df_manual_bias, CFG["sequence_length"], _audit_dir3, "MANUAL")
    # ── v6.1 OUTPUT: Save AUTO and MANUAL datasets SEPARATELY ──────────────
    # Replaces single final_dataset.csv from v5.x/v6.0.
    # Rationale: AUTO trains with GRU/LSTM (sequence models);
    # MANUAL trains with flat RF. Separate CSVs enforce this at the data level.
    # Both files share identical feature structure and all preprocessing steps.
    # ending_battery retained as a metadata column only — NOT a model input.
    output_dir = os.environ.get("OUTPUT_DIR", "outputs_v9")
    os.makedirs(output_dir, exist_ok=True)

    # Metadata columns: saved in CSV for audit but NOT used as model inputs
    # v6.1.3: ending_battery REMOVED from saved CSVs entirely (adviser req).
    # It was previously retained as a metadata/audit column; now excluded
    # from the CSV output so training scripts cannot accidentally use it.
    metadata_cols = [
        "operation_mode", "start_datetime", "session_id",
        "flight_duration_min",
        # ending_battery EXCLUDED from CSV output in v6.1.3
        # (post-flight observable, not needed in output files)
        "operation_mode_enc",
        "month", "season", "hour",
    ]
    # Model input columns (from all_features — does not include ending_battery)
    model_cols = all_features
    # Final save columns = metadata + model features (deduplicated)
    save_cols = metadata_cols + [c for c in model_cols if c not in metadata_cols]
    save_cols = [c for c in save_cols if c in df.columns]

    # AUTO dataset
    df_auto   = df[df["operation_mode"] == "AUTO"].copy()
    auto_path = os.path.join(output_dir, "final_dataset_auto.csv")
    df_auto[save_cols].to_csv(auto_path, index=False)
    print(f"  ✓ final_dataset_auto.csv   → {auto_path}")
    print(f"    Rows: {len(df_auto):,}  Sessions: {df_auto['session_id'].nunique():,}  "
          f"Model features: {len(model_cols)}  "
          f"Seasons: {df_auto['season'].value_counts().to_dict()}")

    # MANUAL dataset
    df_manual   = df[df["operation_mode"] == "MANUAL"].copy()
    manual_path = os.path.join(output_dir, "final_dataset_manual.csv")
    df_manual[save_cols].to_csv(manual_path, index=False)
    print(f"  ✓ final_dataset_manual.csv → {manual_path}")
    print(f"    Rows: {len(df_manual):,}  Sessions: {df_manual['session_id'].nunique():,}  "
          f"Model features: {len(model_cols)}  "
          f"Seasons: {df_manual['season'].value_counts().to_dict()}")

    print(f"\n  Dataset split: AUTO={len(df_auto):,}  MANUAL={len(df_manual):,}  "
          f"Total={len(df):,}")
    print(f"  ending_battery: EXCLUDED from output CSVs in v6.1.3 (adviser req — post-flight observable)")

    return df


# =============================================================================
# PHASE 3b — EDA VISUALIZATION  (flight_duration_distribution.png)
# =============================================================================

def phase3b_eda(df, output_dir):
    """
    OUTPUT 2: flight_duration_distribution.png
    Three-panel EDA figure matching v4.1 output:
      Left  — histogram of flight_duration_min with mean/median lines
      Center — boxplot by season (Dry / Hot / Wet)
      Right  — boxplot by operation mode (AUTO / MANUAL)
    """
    print("\n  [EDA] Generating flight duration distribution plot...")
    target = "flight_duration_min"
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Left: histogram
    ax = axes[0]
    ax.hist(df[target], bins=40, color="#2196F3", edgecolor="white", alpha=0.85)
    ax.axvline(df[target].mean(),   color="#FF9800", linestyle="--", linewidth=1.5,
               label=f"Mean = {df[target].mean():.2f}")
    ax.axvline(df[target].median(), color="#4CAF50", linestyle="-.", linewidth=1.5,
               label=f"Median = {df[target].median():.2f}")
    ax.set_title("Distribution", fontweight="bold")
    ax.set_xlabel("Flight Duration (min)")
    ax.set_ylabel("Count")
    ax.legend(fontsize=9)

    # Centre: boxplot by season
    ax = axes[1]
    season_order = ["Dry", "Hot", "Wet"]
    season_colors = ["#FF9800", "#F44336", "#2196F3"]
    data_by_season = [df.loc[df["season"] == s, target].dropna().values
                      for s in season_order if s in df["season"].unique()]
    labels_present = [s for s in season_order if s in df["season"].unique()]
    bp = ax.boxplot(data_by_season, patch_artist=True, labels=labels_present,
                    showfliers=True, flierprops=dict(marker="o", markersize=2, alpha=0.4))
    for patch, col in zip(bp["boxes"], season_colors):
        patch.set_facecolor(col)
        patch.set_alpha(0.7)
    ax.set_title("Duration by Season", fontweight="bold")
    ax.set_xlabel("Season")
    ax.set_ylabel("Duration (min)")

    # Right: boxplot by operation mode
    ax = axes[2]
    modes = ["MANUAL", "AUTO"]
    mode_colors = ["#9C27B0", "#F44336"]
    data_by_mode = [df.loc[df["operation_mode"] == m, target].dropna().values
                    for m in modes if m in df["operation_mode"].unique()]
    labels_modes = [m for m in modes if m in df["operation_mode"].unique()]
    bp2 = ax.boxplot(data_by_mode, patch_artist=True, labels=labels_modes,
                     showfliers=True, flierprops=dict(marker="o", markersize=2, alpha=0.4))
    for patch, col in zip(bp2["boxes"], mode_colors):
        patch.set_facecolor(col)
        patch.set_alpha(0.7)
    ax.set_title("Duration by Mode", fontweight="bold")
    ax.set_xlabel("Operation Mode")
    ax.set_ylabel("Duration (min)")

    fig.suptitle(
        "Flight Duration Distribution — flight_duration_min (Target Variable)",
        fontsize=13, fontweight="bold", y=1.02,
    )
    plt.tight_layout()
    plt.savefig(
        os.path.join(output_dir, "flight_duration_distribution.png"),
        dpi=150, bbox_inches="tight",
    )
    plt.close()
    print("  ✓ flight_duration_distribution.png saved")


# =============================================================================
# PHASE 4 — NORMALIZE → SEQUENCE (normalize first, then build sequences)
# =============================================================================







## Cell 4 — Feature & Sequence Builders

In [6]:
def get_feature_cols(include_mode_enc=False):
    """
    Returns the feature column list for GRU/LSTM/seq-RF (AUTO mode).

    v6.1: operational_features no longer contains ending_battery.
    Both get_feature_cols() and get_flat_rf_feature_cols() now produce
    the same operational feature set. Feature count: 12 (v9: was 17 in v8.x).

    include_mode_enc=False (default): single-mode models.
    include_mode_enc=True: combined model (RQ3 comparison only).
    """
    base = (
        CFG["operational_features"] +
        CFG["temporal_features"]    +
        CFG["sequential_features"]  +
        FULL_ENV_COLS
    )
    if include_mode_enc:
        return base + CFG["combined_only_features"]
    return base


def get_flat_rf_feature_cols(include_mode_enc=False):
    """
    Feature columns for flat RF (MANUAL mode, single-row prediction).
    v6.1: ending_battery was already excluded here in v5.3.
    Now identical to get_feature_cols() since operational_features
    no longer contains ending_battery. Retained as a named function
    for code clarity and to document design intent.
    """
    base = (
        CFG["operational_features"] +   # same as get_feature_cols() in v6.1
        CFG["temporal_features"]    +
        CFG["sequential_features"]  +
        FULL_ENV_COLS
    )
    if include_mode_enc:
        return base + CFG["combined_only_features"]
    return base


def normalize_rows(df, feature_cols, target_col, train_idx):
    """
    Fit StandardScaler on TRAINING ROWS only, transform all rows.

    Normalization strategy — all features scaled uniformly:
    - StandardScaler applied to ALL features including bounded ones (sin/cos,
      binary operation_mode_enc). This is intentional and safe:
      * hour_sin/cos [-1,1] → ~[-1.41, 1.41] after scaling (harmless)
      * operation_mode_enc {0,1} → {-0.89, 1.12} (still binary in effect)
    - Uniform scaling ensures all features contribute equally to gradient
      updates. Without it, raw battery (0-100) would dominate over
      precipitation (0-13) purely due to magnitude, not importance.
    - Scaler is fitted ONLY on training rows then applied to val/test rows.
      This prevents test-set statistics from leaking into scaler parameters.

    Feature importance ranking (from leave-one-out ablation on AUTO RF):
      Tier 1 Critical   (δMAE > 0.05 min): ending_battery, starting_battery
      Tier 2 Meaningful (δMAE 0.005-0.05): rolling_duration_3, sprayed_area
      Tier 3 Marginal   (δMAE < 0.005): total_amount_l, hour_sin/cos
      Tier 4 Negligible (δMAE ≈ 0): all environmental vars, day_of_week
    Environmental features show low marginal contribution once operational
    data (battery levels, sprayed area) is included — a valid RQ2 finding.

    Order matters: normalize row-by-row first (fitting only on train rows),
    then build sequences. If we sequenced first and normalized after, the
    scaler could see test-set values during fit if we were not careful.
    Normalizing first guarantees no test information in scaler parameters.

    Returns: X_norm (n, n_feat), y_norm (n,), x_scaler, y_scaler
    """
    X = df[feature_cols].values.astype(np.float32)
    y = df[target_col].values.astype(np.float32)

    x_scaler = StandardScaler()
    y_scaler = StandardScaler()

    x_scaler.fit(X[train_idx])
    y_scaler.fit(y[train_idx].reshape(-1, 1))

    X_norm = x_scaler.transform(X)
    y_norm = y_scaler.transform(y.reshape(-1, 1)).ravel()

    return X_norm, y_norm, x_scaler, y_scaler


def build_sequences_session_aware(X_norm, y_norm, session_ids, seq_len):
    """
    Sliding-window sequence construction with strict session boundaries.

    A sequence of seq_len consecutive past flights predicts the next flight.
    Windows NEVER cross session boundaries — a session boundary represents
    a battery recharge or operational pause (gap > 60 min), after which
    the operational context resets and no temporal continuity is assumed.

    The session_ids array must be aligned with X_norm rows (same row order).

    Returns: X_seq (n_seq, seq_len, n_feat), y_seq (n_seq,)
    """
    # v7.3 FIX: np.unique sorts lexicographically (AUTO_1→AUTO_10→AUTO_2).
    # Sorting numerically by the integer suffix restores chronological order.
    def _sess_key(s):
        try: return int(str(s).split("_")[-1])
        except ValueError: return 0
    unique_sessions = sorted(np.unique(session_ids), key=_sess_key)
    Xs, ys, target_idx = [], [], []
    for sid in unique_sessions:
        idx = np.where(session_ids == sid)[0]
        if len(idx) <= seq_len:
            continue   # session too short for even one window
        for i in range(seq_len, len(idx)):
            Xs.append(X_norm[idx[i - seq_len:i]])
            ys.append(y_norm[idx[i]])
            target_idx.append(int(idx[i]))  # v7.3: row index of target flight

    if not Xs:
        return (
            np.empty((0, seq_len, X_norm.shape[1]), dtype=np.float32),
            np.empty((0,),                          dtype=np.float32),
            np.empty((0,),                          dtype=np.int64),
        )
    return (
        np.array(Xs, dtype=np.float32),
        np.array(ys, dtype=np.float32),
        np.array(target_idx, dtype=np.int64),   # v7.3: df_auto row indices of targets
    )


def temporal_split_sequences(X_seq, y_seq, train_frac, val_frac):
    """
    Chronological train / val / test split on sequences.
    Earlier sequences → train, middle → val, latest → test.
    Prevents information leakage and mirrors real deployment conditions
    where future flight durations are inferred from historical data.
    """
    n    = len(y_seq)
    n_tr = int(n * train_frac)
    n_va = int(n * val_frac)
    return (
        (X_seq[:n_tr],                   y_seq[:n_tr]),
        (X_seq[n_tr:n_tr + n_va],        y_seq[n_tr:n_tr + n_va]),
        (X_seq[n_tr + n_va:],            y_seq[n_tr + n_va:]),
    )


# =============================================================================
# MODELS
# =============================================================================


## Cell 5 — Model Classes + Training Loop (GRU, LSTM)

In [7]:
class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.gru  = nn.GRU(
            input_size, hidden_size, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(self.drop(out[:, -1, :])).squeeze(-1)


class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(self.drop(out[:, -1, :])).squeeze(-1)


def train_one_config(ModelClass, hp, X_tr, y_tr, X_va, y_va,
                     input_size, device, epochs, patience,
                     verbose=True):
    """
    Train one model configuration with Adam optimizer, MSE loss,
    ReduceLROnPlateau scheduler, and early stopping.

    shuffle=False in DataLoader is intentional: the sequences already
    encode temporal order within each session window. Shuffling would
    randomize the chronological structure of mini-batches, which does
    not break model correctness (sequences are self-contained) but
    is avoided for methodological consistency and reproducibility.

    verbose=True: prints per-epoch train loss and val loss, plus a
    summary line when training ends (early stopping or max epochs).
    Set verbose=False to suppress per-epoch output (used in repetitions
    phase to avoid flooding the notebook).
    """
    model = ModelClass(
        input_size  = input_size,
        hidden_size = hp["hidden_size"],
        num_layers  = hp["num_layers"],
        dropout     = hp["dropout"],
    ).to(device)

    tr_dl = DataLoader(
        TensorDataset(
            torch.tensor(X_tr, dtype=torch.float32),
            torch.tensor(y_tr, dtype=torch.float32),
        ),
        batch_size=hp["batch_size"],
        shuffle=False,   # intentional: see docstring above
    )
    opt  = torch.optim.Adam(
        model.parameters(), lr=hp["learning_rate"], weight_decay=1e-5
    )
    # ReduceLROnPlateau: halve LR if val_loss stagnates for 8 epochs
    # patience (8) < early_stop_patience (15) — allows LR reduction
    # before hard stopping
    sch  = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=8)
    crit = nn.MSELoss()
    best_vl, best_state, pat = float("inf"), None, 0
    best_epoch = 0      # epoch (1-indexed) where best val loss occurred

    for ep in range(epochs):
        # ── Training pass ─────────────────────────────────────────────────
        model.train()
        epoch_train_loss = 0.0
        n_batches = 0
        for Xb, yb in tr_dl:
            Xb, yb = Xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(model(Xb), yb)
            loss.backward()
            # Gradient clipping: prevents exploding gradients in RNNs
            # norm=1.0 is the standard value for LSTM/GRU (Goodfellow 2016)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            epoch_train_loss += loss.item()
            n_batches += 1
        avg_train_loss = epoch_train_loss / max(n_batches, 1)

        # ── Validation pass ───────────────────────────────────────────────
        model.eval()
        with torch.no_grad():
            X_va_t = torch.tensor(X_va, dtype=torch.float32).to(device)
            y_va_t = torch.tensor(y_va, dtype=torch.float32).to(device)
            vl     = crit(model(X_va_t), y_va_t).item()
        sch.step(vl)

        # ── Per-epoch logging ─────────────────────────────────────────────
        if verbose:
            print(f"      Epoch {ep + 1:>3}/{epochs} | "
                  f"Train Loss: {avg_train_loss:.4f} | "
                  f"Val Loss: {vl:.4f}")

        # ── Early stopping ────────────────────────────────────────────────
        if vl < best_vl:
            best_vl    = vl
            best_epoch = ep + 1   # 1-indexed for readability
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            pat        = 0
        else:
            pat += 1
            if pat >= patience:
                if verbose:
                    print(f"      Early stopping triggered at epoch {ep + 1}")
                break

    # ── Post-training summary ─────────────────────────────────────────────
    total_epochs = ep + 1
    if verbose:
        if total_epochs == epochs:
            print(f"      Completed all {epochs} epochs (no early stopping)")
        print(f"      Best val loss: {best_vl:.4f}  at epoch {best_epoch}/{total_epochs}")

    model.load_state_dict(best_state)
    return model, best_vl


def random_search_rnn(ModelClass, model_name,
                      X_tr, y_tr, X_va, y_va, X_te, y_te,
                      y_scaler, input_size, device, n_configs,
                      verbose_train=True):
    print(f"\n  [{model_name}] Random search — {n_configs} configs")
    space = CFG["hp_space"]
    log   = []
    best_vl, best_model, best_hp = float("inf"), None, None

    for i in range(n_configs):
        hp = {
            "hidden_size":   random.choice(space["hidden_size"]),
            "num_layers":    random.choice(space["num_layers"]),
            "dropout":       random.choice(space["dropout"]),
            "learning_rate": random.choice(space["learning_rate"]),
            "batch_size":    random.choice(space["batch_size"]),
        }
        t0 = time.time()
        model, vl = train_one_config(
            ModelClass, hp, X_tr, y_tr, X_va, y_va,
            input_size, device, CFG["epochs"], CFG["early_stop_patience"],
            verbose=verbose_train,
        )
        elapsed = time.time() - t0
        log.append({**hp, "val_loss": vl, "time_s": elapsed})
        marker = " ← best" if vl < best_vl else ""
        print(f"    Config {i+1:02d}  "
              f"h={hp['hidden_size']} l={hp['num_layers']} "
              f"d={hp['dropout']} lr={hp['learning_rate']} bs={hp['batch_size']} "
              f"→ val_loss={vl:.5f}{marker}")
        if vl < best_vl:
            best_vl, best_model, best_hp = vl, model, hp

    # Final evaluation on held-out test set
    best_model.eval()
    with torch.no_grad():
        pred_norm = best_model(
            torch.tensor(X_te, dtype=torch.float32).to(device)
        ).cpu().numpy()
    pred = y_scaler.inverse_transform(pred_norm.reshape(-1, 1)).ravel()
    true = y_scaler.inverse_transform(y_te.reshape(-1, 1)).ravel()
    metrics          = _metrics(true, pred, model_name)
    metrics["best_hp"] = best_hp
    return best_model, metrics, pd.DataFrame(log)


def grid_search_rf(X_tr, y_tr, X_va, y_va, X_te, y_te, y_scaler):
    """
    Random Forest with sequence-flattened input.
    Sequences of shape (n, seq_len, n_feat) are flattened to (n, seq_len*n_feat).
    This gives RF access to the same temporal window as GRU/LSTM, but RF
    does not model the sequential ORDER within the window — this is the
    controlled condition that isolates the value of temporal modeling (RQ1).
    """
    print(f"\n  [RF] Grid search — 9 configs")
    grid    = CFG["rf_grid"]
    configs = list(itertools.product(grid["n_estimators"], grid["max_depth"]))
    best_vl, best_rf, best_hp = float("inf"), None, None

    def _flatten(X):
        n, s, f = X.shape
        return X.reshape(n, s * f)

    Xtr_f = _flatten(X_tr)
    Xva_f = _flatten(X_va)
    Xte_f = _flatten(X_te)

    for n_est, max_d in configs:
        rf = RandomForestRegressor(
            n_estimators=n_est, max_depth=max_d,
            random_state=SEED, n_jobs=-1,
        )
        rf.fit(Xtr_f, y_tr)
        vl     = mean_squared_error(y_va, rf.predict(Xva_f))
        marker = " ← best" if vl < best_vl else ""
        print(f"    n_est={n_est} max_d={str(max_d):<5} "
              f"→ val_MSE={vl:.5f}{marker}")
        if vl < best_vl:
            best_vl, best_rf, best_hp = vl, rf, {
                "n_estimators": n_est, "max_depth": max_d,
            }

    pred = y_scaler.inverse_transform(
        best_rf.predict(Xte_f).reshape(-1, 1)
    ).ravel()
    true = y_scaler.inverse_transform(y_te.reshape(-1, 1)).ravel()
    metrics          = _metrics(true, pred, "RF (seq-flattened)")
    metrics["best_hp"] = best_hp
    return best_rf, metrics


def flat_rf_grid_search(X_tr, y_tr, X_va, y_va, X_te, y_te, y_scaler,
                        label="flat RF"):
    """
    Flat RF for MANUAL mode contexts — no sequencing.
    X is 2D (n_rows × n_features), already normalized.
    See MANUAL_JUSTIFICATION for why MANUAL does not use GRU/LSTM.
    """
    grid    = CFG["rf_grid"]
    configs = list(itertools.product(grid["n_estimators"], grid["max_depth"]))
    best_vl, best_rf = float("inf"), None

    for n_est, max_d in configs:
        rf = RandomForestRegressor(
            n_estimators=n_est, max_depth=max_d,
            random_state=SEED, n_jobs=-1,
        )
        rf.fit(X_tr, y_tr)
        vl = mean_squared_error(y_va, rf.predict(X_va))
        if vl < best_vl:
            best_vl, best_rf = vl, rf

    pred = y_scaler.inverse_transform(
        best_rf.predict(X_te).reshape(-1, 1)
    ).ravel()
    true = y_scaler.inverse_transform(y_te.reshape(-1, 1)).ravel()
    return _metrics(true, pred, label)


def _metrics(y_true, y_pred, label=""):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    if label:
        print(f"    {label:<45} "
              f"MAE={mae:.4f}  RMSE={rmse:.4f}  "
              f"R²={r2:.4f}  MAPE={mape:.2f}%")
    return {"MAE": mae, "RMSE": rmse, "R2": r2, "MAPE": mape}


# =============================================================================
# COMPUTATIONAL EFFICIENCY HELPERS (RQ1)
# =============================================================================

## Cell 6 — Phase 5 Baseline Definition (GRU, LSTM, Seq-RF)

In [8]:
def measure_inference_latency(model, X_sample, device, n_runs=100):
    """
    Single-prediction inference latency in milliseconds.
    Uses a single sample because the operational use case is one pre-mission
    prediction per upcoming flight, not batch processing.
    Warm-up run eliminates CUDA initialization overhead.
    """
    model.eval()
    sample = torch.tensor(X_sample[:1], dtype=torch.float32).to(device)
    with torch.no_grad():
        _ = model(sample)  # warm-up
    if device.type == "cuda":
        torch.cuda.synchronize()

    latencies = []
    with torch.no_grad():
        for _ in range(n_runs):
            if device.type == "cuda":
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            _  = model(sample)
            if device.type == "cuda":
                torch.cuda.synchronize()
            latencies.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(latencies)), float(np.std(latencies))


def measure_rf_inference_latency(rf_model, X_flat_sample, n_runs=100):
    sample = X_flat_sample[:1]
    _ = rf_model.predict(sample)  # warm-up
    latencies = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        _  = rf_model.predict(sample)
        latencies.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(latencies)), float(np.std(latencies))


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def measure_peak_gpu_mb(device):
    if device.type == "cuda":
        return torch.cuda.max_memory_allocated(device) / 1024 ** 2
    return 0.0


# =============================================================================
# PHASE 5 — BASELINE EXPERIMENT (RQ1: architecture comparison + efficiency)
# =============================================================================

def phase5_baseline(df, output_dir):
    """
    Trains GRU, LSTM, and RF on AUTO data.
    Measures accuracy (MAE/RMSE/R²/MAPE) and computational efficiency
    (training time, inference latency, memory, parameter count) for RQ1.
    Also runs flat RF on both AUTO and MANUAL for same-model mode
    comparison (RQ3 — isolates the effect of operation mode, not model).
    """
    log_phase("Phase 5 — Baseline Experiment (RQ1: architecture comparison)", context="MAIN PIPELINE")

    device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    feature_cols = get_feature_cols()
    target_col   = "flight_duration_min"
    print(f"  Device: {device}  |  Features: {len(feature_cols)}")

    assert_no_leakage(df, feature_cols, target_col)

    # ── Prepare AUTO sequences ────────────────────────────────────────────
    df_auto = (
        df[df["operation_mode"] == "AUTO"]
        .copy()
        .reset_index(drop=True)
    )
    df_auto = (
        df_auto[feature_cols + [target_col, "session_id"]]
        .dropna()
        .reset_index(drop=True)
    )

    n      = len(df_auto)
    n_tr   = int(n * CFG["train_frac"])
    n_va   = int(n * CFG["val_frac"])
    tr_idx = np.arange(n_tr)

    X_norm, y_norm, x_sc, y_sc = normalize_rows(
        df_auto, feature_cols, target_col, tr_idx
    )
    X_seq, y_seq, target_row_idx = build_sequences_session_aware(
        X_norm, y_norm, df_auto["session_id"].values, CFG["sequence_length"]
    )  # v7.3: target_row_idx maps each sequence to its df_auto row index
    print(f"  AUTO sequences total: {len(y_seq):,}")
    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = temporal_split_sequences(
        X_seq, y_seq, CFG["train_frac"], CFG["val_frac"]
    )
    # v7.3: compute test set target row indices for traceable output
    _n_seq    = len(y_seq)
    _n_tr_seq = int(_n_seq * CFG["train_frac"])
    _n_va_seq = int(_n_seq * CFG["val_frac"])
    test_target_idx = target_row_idx[_n_tr_seq + _n_va_seq:]
    print(f"  Train={len(y_tr):,}  Val={len(y_va):,}  Test={len(y_te):,}")

    # Issue 1: Session split leakage assertion
    assert_no_session_leakage(
        df_auto["session_id"].values, y_seq,
        CFG["train_frac"], CFG["val_frac"],
        label="AUTO sequences (phase5_baseline)"
    )

    input_size = len(feature_cols)
    results    = {}
    hp_logs    = {}
    efficiency = {}

    # ── GRU ───────────────────────────────────────────────────────────────
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    t_start = time.perf_counter()
    gru, gru_m, gru_log = random_search_rnn(
        GRUModel, "GRU",
        X_tr, y_tr, X_va, y_va, X_te, y_te,
        y_sc, input_size, device, CFG["hp_search_configs"],
    )
    gru_train_time       = time.perf_counter() - t_start
    gru_mem              = measure_peak_gpu_mb(device)
    gru_lat, gru_lat_std = measure_inference_latency(gru, X_te, device)
    gru_params           = count_parameters(gru)
    results["GRU"]       = gru_m
    hp_logs["GRU"]       = gru_log
    efficiency["GRU"]    = {
        "training_time_s":  round(gru_train_time, 2),
        "inference_lat_ms": round(gru_lat, 4),
        "inference_std_ms": round(gru_lat_std, 4),
        "peak_gpu_mb":      round(gru_mem, 2),
        "n_parameters":     gru_params,
    }

    # ── LSTM ──────────────────────────────────────────────────────────────
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    t_start = time.perf_counter()
    lstm, lstm_m, lstm_log = random_search_rnn(
        LSTMModel, "LSTM",
        X_tr, y_tr, X_va, y_va, X_te, y_te,
        y_sc, input_size, device, CFG["hp_search_configs"],
    )
    lstm_train_time         = time.perf_counter() - t_start
    lstm_mem                = measure_peak_gpu_mb(device)
    lstm_lat, lstm_lat_std  = measure_inference_latency(lstm, X_te, device)
    lstm_params             = count_parameters(lstm)
    results["LSTM"]         = lstm_m
    hp_logs["LSTM"]         = lstm_log
    efficiency["LSTM"]      = {
        "training_time_s":  round(lstm_train_time, 2),
        "inference_lat_ms": round(lstm_lat, 4),
        "inference_std_ms": round(lstm_lat_std, 4),
        "peak_gpu_mb":      round(lstm_mem, 2),
        "n_parameters":     lstm_params,
    }

    # ── RF (sequence-flattened — same window as RNN, temporal order lost) ─
    t_start = time.perf_counter()
    rf, rf_m = grid_search_rf(X_tr, y_tr, X_va, y_va, X_te, y_te, y_sc)
    rf_train_time = time.perf_counter() - t_start
    n_, s_, f_    = X_te.shape
    rf_lat, rf_lat_std = measure_rf_inference_latency(
        rf, X_te.reshape(n_, s_ * f_)
    )
    results["RF"]    = rf_m
    efficiency["RF"] = {
        "training_time_s":  round(rf_train_time, 2),
        "inference_lat_ms": round(rf_lat, 4),
        "inference_std_ms": round(rf_lat_std, 4),
        "peak_gpu_mb":      0.0,
        "n_parameters":     rf.n_estimators * rf.estimators_[0].tree_.node_count,
    }

    # Print efficiency summary
    print("\n  COMPUTATIONAL EFFICIENCY (RQ1):")
    print(f"  {'Model':<8} {'Train(s)':>10} {'Latency(ms)':>13} "
          f"{'±Std':>8} {'PeakGPU(MB)':>13} {'Params':>12}")
    print("  " + "-" * 68)
    for mname, eff in efficiency.items():
        print(f"  {mname:<8} {eff['training_time_s']:>10.1f} "
              f"{eff['inference_lat_ms']:>13.4f} {eff['inference_std_ms']:>8.4f} "
              f"{eff['peak_gpu_mb']:>13.1f} {eff['n_parameters']:>12,}")

    # Issue 7: Overfitting / stability check — RF has direct train MAE
    _n_, _s_, _f_ = X_tr.shape
    _rf_train_pred = y_sc.inverse_transform(
        rf.predict(X_tr.reshape(_n_, _s_ * _f_)).reshape(-1, 1)
    ).ravel()
    _rf_train_true = y_sc.inverse_transform(y_tr.reshape(-1, 1)).ravel()
    _rf_val_pred   = y_sc.inverse_transform(
        rf.predict(X_va.reshape(len(y_va), _s_ * _f_)).reshape(-1, 1)
    ).ravel()
    _rf_val_true   = y_sc.inverse_transform(y_va.reshape(-1, 1)).ravel()
    _stability_rf  = check_overfitting(
        "Seq-RF",
        mean_absolute_error(_rf_train_true, _rf_train_pred),
        mean_absolute_error(_rf_val_true,   _rf_val_pred),
        rf_m["MAE"],
    )
    # Save baseline results and efficiency table
    rows = []
    for mname, m in results.items():
        eff = efficiency[mname]
        rows.append({
            "model":            mname,
            "data_mode":        "AUTO",
            "MAE":              m["MAE"],
            "RMSE":             m["RMSE"],
            "R2":               m["R2"],
            "MAPE":             m["MAPE"],
            "training_time_s":  eff["training_time_s"],
            "inference_lat_ms": eff["inference_lat_ms"],
            "inference_std_ms": eff["inference_std_ms"],
            "peak_gpu_mb":      eff["peak_gpu_mb"],
            "n_parameters":     eff["n_parameters"],
            "best_hp":          str(m.get("best_hp", "")),
            "n_test":           len(y_te),
            "note": (
                "RF input = sequences flattened to (n, seq_len*n_feat). "
                "Same data as GRU/LSTM. RF does not model temporal ORDER. "
                "MAE gap vs GRU/LSTM quantifies value of temporal modeling (RQ1)."
            ),
        })
    pd.DataFrame(rows).to_csv(
        os.path.join(output_dir, "baseline_results.csv"), index=False
    )
    pd.DataFrame(efficiency).T.reset_index().rename(
        columns={"index": "model"}
    ).to_csv(
        os.path.join(output_dir, "computational_efficiency.csv"), index=False
    )
    for name, log in hp_logs.items():
        log.to_csv(
            os.path.join(output_dir, f"hp_search_{name}.csv"), index=False
        )

    # ── OUTPUT 3: computational_efficiency.png ────────────────────────────────
    # 4-panel bar chart: training time, inference latency, GPU memory, parameters
    eff_df = pd.DataFrame(efficiency).T.reset_index().rename(columns={"index": "model"})
    model_labels = eff_df["model"].tolist()
    bar_colors   = ["#2196F3" if m != "RF" else "#FF9800" for m in model_labels]

    fig, axes = plt.subplots(1, 4, figsize=(16, 5))
    fig.suptitle(
        "Computational Efficiency Comparison — RQ1\n"
        "(All models trained on same AUTO sequence dataset)",
        fontsize=12, fontweight="bold",
    )

    panels = [
        ("training_time_s",   "Training Time (seconds)",    "lower = faster"),
        ("inference_lat_ms",  "Inference Latency (ms/pred)", "lower = faster"),
        ("peak_gpu_mb",       "Peak GPU Memory (MB)",        "lower = lighter"),
        ("n_parameters",      "Parameter Count",             "lower = simpler"),
    ]
    for ax, (col, title, ylabel) in zip(axes, panels):
        vals = eff_df[col].values
        bars = ax.bar(model_labels, vals, color=bar_colors, edgecolor="white", width=0.5)
        for bar, v in zip(bars, vals):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(vals) * 0.01,
                f"{v:,.1f}" if col != "n_parameters" else f"{int(v):,}",
                ha="center", va="bottom", fontsize=9,
            )
        ax.set_title(title, fontsize=10, fontweight="bold")
        ax.set_ylabel(ylabel, fontsize=8, color="#555555")
        ax.tick_params(axis="x", labelsize=9)
        ax.set_ylim(0, max(vals) * 1.15)

    plt.tight_layout()
    plt.savefig(
        os.path.join(output_dir, "computational_efficiency.png"), dpi=150
    )
    plt.close()
    print("  ✓ computational_efficiency.png saved")

    # ── OUTPUT 5: mode_comparison_rf.csv ─────────────────────────────────────
    # Collect flat RF results for both modes and save as CSV for RQ3 reporting.
    mode_rows = []

    def _flat_rf_for_mode(mode_label):
        """
        Runs flat RF twice for each mode:
          (a) WITH-LEAKAGE: uses full feature_cols including ending_battery.
              This replicates the v5.2 result to show the leakage effect size.
          (b) CLEAN: uses get_flat_rf_feature_cols() (ending_battery excluded).
              This is the valid baseline for RQ3 mode comparison.
        Both results are saved to mode_comparison_rf.csv for transparency.
        See FLAT_RF_LEAKAGE_NOTE for justification.
        """
        df_m_base = (
            df[df["operation_mode"] == mode_label]
            .copy()
            .reset_index(drop=True)
        )
        n_m = len(df_m_base)

        def _run_rf(feat_list, label, version_tag):
            df_m = df_m_base[feat_list + [target_col]].dropna().reset_index(drop=True)
            n_r   = len(df_m)
            n_tr_ = int(n_r * CFG["train_frac"])
            n_va_ = int(n_r * CFG["val_frac"])
            tr_   = np.arange(n_tr_)
            X_, y_, _, y_sc_ = normalize_rows(df_m, feat_list, target_col, tr_)
            Xtr = X_[:n_tr_];             ytr = y_[:n_tr_]
            Xva = X_[n_tr_:n_tr_+n_va_]; yva = y_[n_tr_:n_tr_+n_va_]
            Xte = X_[n_tr_+n_va_:];      yte = y_[n_tr_+n_va_:]
            if len(yte) < 3:
                print(f"    {label}: insufficient test rows — skip")
                return
            m = flat_rf_grid_search(Xtr, ytr, Xva, yva, Xte, yte, y_sc_, label=label)
            mode_rows.append({
                "mode":         mode_label,
                "version":      version_tag,
                "model":        "flat RF (no sequencing)",
                "n_flights":    n_m,
                "n_features":   len(feat_list),
                "ending_battery_included": "YES" if "ending_battery" in feat_list else "NO",
                "MAE":          round(m["MAE"],  4),
                "RMSE":         round(m["RMSE"], 4),
                "R2":           round(m["R2"],   4),
                "MAPE":         round(m["MAPE"], 4),
                "note": (
                    "WITH-LEAKAGE: ending_battery included — "
                    "model reconstructs duration via drain formula (r=0.974). "
                    "Reported for transparency only, not used in RQ3 analysis."
                    if version_tag == "with_leakage" else
                    "CLEAN (v5.3 fix): ending_battery excluded. "
                    "Valid baseline for RQ3 mode comparison. "
                    + FLAT_RF_LEAKAGE_NOTE[:80] + "..."
                ),
            })

        # (a) With-leakage version — shows effect size of the leakage
        _run_rf(
            feature_cols,
            label=f"{mode_label} flat RF WITH-LEAKAGE (v5.2, ending_battery included)",
            version_tag="with_leakage",
        )
        # (b) Clean version — valid for RQ3
        clean_feat_cols = get_flat_rf_feature_cols()
        _run_rf(
            clean_feat_cols,
            label=f"{mode_label} flat RF CLEAN (v5.3, ending_battery excluded)",
            version_tag="clean",
        )

    _flat_rf_for_mode("AUTO")
    _flat_rf_for_mode("MANUAL")

    if mode_rows:
        pd.DataFrame(mode_rows).to_csv(
            os.path.join(output_dir, "mode_comparison_rf.csv"), index=False
        )
        print("  ✓ mode_comparison_rf.csv saved")

    # ── Collect GRU/LSTM/RF test-set predictions for scatter plot ─────────────
    # Inverse-transform to original minutes scale for plotting.
    gru.eval()
    with torch.no_grad():
        gru_pred_norm = gru(
            torch.tensor(X_te, dtype=torch.float32).to(device)
        ).cpu().numpy()
    lstm.eval()
    with torch.no_grad():
        lstm_pred_norm = lstm(
            torch.tensor(X_te, dtype=torch.float32).to(device)
        ).cpu().numpy()
    n_, s_, f_ = X_te.shape
    rf_pred_norm = rf.predict(X_te.reshape(n_, s_ * f_))

    y_true_orig = y_sc.inverse_transform(y_te.reshape(-1, 1)).ravel()
    # v7.3: include row-level metadata so every prediction is traceable
    _dt_col  = "start_datetime"
    _sid_col = "session_id"
    _dt_vals  = (df_auto[_dt_col].iloc[test_target_idx].values
                 if _dt_col  in df_auto.columns else [pd.NaT]*len(y_te))
    _sid_vals = (df_auto[_sid_col].iloc[test_target_idx].values
                 if _sid_col in df_auto.columns else [None]*len(y_te))
    pred_df = pd.DataFrame({
        "start_datetime": _dt_vals,
        "session_id":     _sid_vals,
        "target_row_idx": test_target_idx,
        "y_true":         y_true_orig,
        "y_pred_gru":     y_sc.inverse_transform(gru_pred_norm.reshape(-1, 1)).ravel(),
        "y_pred_lstm":    y_sc.inverse_transform(lstm_pred_norm.reshape(-1, 1)).ravel(),
        "y_pred_rf":      y_sc.inverse_transform(rf_pred_norm.reshape(-1, 1)).ravel(),
    })

    # v6.1.4 export patch: return uses local names (gru, lstm, rf) and
    # order matches execution cell 34 unpacking.
    # Order: models → scalers (y_sc first, x_sc second) → HP dicts → data
    return gru, lstm, rf, y_sc, x_sc, gru_m, lstm_m, X_seq, y_seq, df_auto, pred_df


# =============================================================================
# PHASE 6 — ABLATION EXPERIMENTS (RQ2: environmental feature contribution)
# =============================================================================





## Cell 6b — Phase 5B Definition (XGBoost, LightGBM, CatBoost)

In [9]:
def phase5_boosted_models(df, output_dir):
    """
    Phase 5B — Tabular Boosting Models on AUTO flat features (no sequences).

    Models : XGBoost, LightGBM, CatBoost
    Input  : get_flat_rf_feature_cols() — same 17 features as flat RF.
    Split  : same 70/15/15 chronological split as all other phases.
    Scale  : raw features (tree models are scale-invariant).
    Search : 6-config grid per model, best selected by val MSE.
    Output : boosted_models_results.csv  (same columns as baseline_results.csv)
             boosted_feature_importance.png
    Returns: (results, xgb_model, lgbm_model, cb_best_model)
             results       — dict of per-model metrics
             xgb_model     — trained XGBRegressor (best val-MSE config)
             lgbm_model    — trained LGBMRegressor (best val-MSE config)
             cb_best_model — trained CatBoostRegressor (best val-MSE config)

    No data leakage: assert_no_leakage() called before any training.
    """
    log_phase("Phase 5B — Boosted Models (XGBoost, LightGBM, CatBoost)", context="MAIN PIPELINE")

    feature_cols = get_flat_rf_feature_cols()
    target_col   = "flight_duration_min"
    print(f"  Features: {len(feature_cols)}  (flat — no sequencing, same as flat RF)")

    # ── Leakage check ─────────────────────────────────────────────────────────
    assert_no_leakage(df, feature_cols, target_col)

    # ── Prepare AUTO data ─────────────────────────────────────────────────────
    df_auto = (
        df[df["operation_mode"] == "AUTO"]
        .copy()
        .reset_index(drop=True)
    )
    df_auto = (
        df_auto[feature_cols + [target_col]]
        .dropna()
        .reset_index(drop=True)
    )
    print(f"  AUTO rows after dropna: {len(df_auto):,}")

    # ── Chronological 70/15/15 split — raw features, no normalization ─────────
    # Tree models are scale-invariant; StandardScaler not needed.
    # Mirrors the flat-RF split in phase5_flat_vs_seq_rf exactly.
    n    = len(df_auto)
    n_tr = int(n * CFG["train_frac"])
    n_va = int(n * CFG["val_frac"])

    X = df_auto[feature_cols].values
    y = df_auto[target_col].values

    X_tr, y_tr = X[:n_tr],           y[:n_tr]
    X_va, y_va = X[n_tr:n_tr+n_va],  y[n_tr:n_tr+n_va]
    X_te, y_te = X[n_tr+n_va:],      y[n_tr+n_va:]
    print(f"  Train={len(y_tr):,}  Val={len(y_va):,}  Test={len(y_te):,}")

    hp_space = CFG["boosted_hp_space"]
    results  = {}

    def _boosted_search(model_cls, model_name, hp_list, fit_kwargs=None):
        """
        Small grid search for a boosted model.
        Selects best config by val MSE — same criterion as flat_rf_grid_search.
        Evaluates best model on test set using _metrics().
        Returns (best_model, metrics_dict, best_hp).
        """
        fit_kwargs = fit_kwargs or {}
        print(f"\n  [{model_name}] Grid search — {len(hp_list)} configs")
        best_vl, best_model, best_hp = float("inf"), None, None

        for hp in hp_list:
            model = model_cls(**hp, random_state=SEED, **fit_kwargs)
            model.fit(X_tr, y_tr)
            vl     = mean_squared_error(y_va, model.predict(X_va))
            marker = " ← best" if vl < best_vl else ""
            hp_str = "  ".join(f"{k}={v}" for k, v in hp.items())
            print(f"    {hp_str}  → val_MSE={vl:.5f}{marker}")
            if vl < best_vl:
                best_vl, best_model, best_hp = vl, model, hp

        pred    = best_model.predict(X_te)
        metrics = _metrics(y_te, pred, model_name)
        metrics["best_hp"] = best_hp
        return best_model, metrics, best_hp

    # ── XGBoost ───────────────────────────────────────────────────────────────
    t_start = time.perf_counter()
    xgb_model, xgb_m, xgb_hp = _boosted_search(
        XGBRegressor,
        "XGBoost",
        hp_space["xgb"],
        fit_kwargs={
            "tree_method": "hist",
            "eval_metric": "rmse",
            "verbosity":   0,
            "n_jobs":      -1,
        },
    )
    xgb_train_time = time.perf_counter() - t_start
    results["XGBoost"] = {**xgb_m, "training_time_s": round(xgb_train_time, 2)}

    # ── LightGBM ──────────────────────────────────────────────────────────────
    t_start = time.perf_counter()
    lgbm_model, lgbm_m, lgbm_hp = _boosted_search(
        LGBMRegressor,
        "LightGBM",
        hp_space["lgbm"],
        fit_kwargs={
            "n_jobs":  -1,
            "verbose": -1,
        },
    )
    lgbm_train_time = time.perf_counter() - t_start
    results["LightGBM"] = {**lgbm_m, "training_time_s": round(lgbm_train_time, 2)}

    # ── CatBoost ──────────────────────────────────────────────────────────────
    # Handled separately: uses random_seed (not random_state) and
    # has built-in early_stopping_rounds via eval_set.
    print(f"\n  [CatBoost] Grid search — {len(hp_space['catboost'])} configs")
    t_start = time.perf_counter()
    best_vl_cb, cb_best_model, cb_best_hp = float("inf"), None, None
    for hp in hp_space["catboost"]:
        cb = CatBoostRegressor(**hp, random_seed=SEED, verbose=0)
        cb.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=20)
        vl     = mean_squared_error(y_va, cb.predict(X_va))
        marker = " ← best" if vl < best_vl_cb else ""
        hp_str = "  ".join(f"{k}={v}" for k, v in hp.items())
        print(f"    {hp_str}  → val_MSE={vl:.5f}{marker}")
        if vl < best_vl_cb:
            best_vl_cb, cb_best_model, cb_best_hp = vl, cb, hp
    cb_train_time   = time.perf_counter() - t_start
    cb_pred         = cb_best_model.predict(X_te)
    cb_m            = _metrics(y_te, cb_pred, "CatBoost")
    cb_m["best_hp"] = cb_best_hp
    results["CatBoost"] = {**cb_m, "training_time_s": round(cb_train_time, 2)}

    # ── Inference latency — 100-run mean (mirrors measure_rf_inference_latency) ─
    def _boosted_latency(model, n_runs=100):
        sample = X_te[:1]
        _      = model.predict(sample)   # warm-up
        lats   = []
        for _ in range(n_runs):
            t0 = time.perf_counter()
            model.predict(sample)
            lats.append((time.perf_counter() - t0) * 1000)
        return float(np.mean(lats)), float(np.std(lats))

    xgb_lat,  xgb_lat_std  = _boosted_latency(xgb_model)
    lgbm_lat, lgbm_lat_std = _boosted_latency(lgbm_model)
    cb_lat,   cb_lat_std   = _boosted_latency(cb_best_model)

    results["XGBoost"]["inference_lat_ms"]  = round(xgb_lat,  4)
    results["XGBoost"]["inference_std_ms"]  = round(xgb_lat_std, 4)
    results["LightGBM"]["inference_lat_ms"] = round(lgbm_lat, 4)
    results["LightGBM"]["inference_std_ms"] = round(lgbm_lat_std, 4)
    results["CatBoost"]["inference_lat_ms"] = round(cb_lat,   4)
    results["CatBoost"]["inference_std_ms"] = round(cb_lat_std, 4)

    # ── Summary table ─────────────────────────────────────────────────────────
    print("\n  BOOSTED MODELS SUMMARY:")
    print(f"  {'Model':<12} {'MAE':>8} {'RMSE':>8} {'R²':>8} "
          f"{'MAPE%':>8} {'Train(s)':>10} {'Lat(ms)':>10}")
    print("  " + "-" * 70)
    for mname, m in results.items():
        print(f"  {mname:<12} {m['MAE']:>8.4f} {m['RMSE']:>8.4f} "
              f"{m['R2']:>8.4f} {m['MAPE']:>8.2f} "
              f"{m['training_time_s']:>10.1f} {m['inference_lat_ms']:>10.4f}")

    # ── Save boosted_models_results.csv ──────────────────────────────────────
    # Column format matches baseline_results.csv for direct cross-model comparison.
    rows = []
    for mname, m in results.items():
        rows.append({
            "model":            mname,
            "data_mode":        "AUTO",
            "input_type":       "flat (no sequences)",
            "n_features":       len(feature_cols),
            "MAE":              round(m["MAE"],  4),
            "RMSE":             round(m["RMSE"], 4),
            "R2":               round(m["R2"],   4),
            "MAPE":             round(m["MAPE"], 4),
            "training_time_s":  m["training_time_s"],
            "inference_lat_ms": m["inference_lat_ms"],
            "inference_std_ms": m["inference_std_ms"],
            "peak_gpu_mb":      0.0,
            "best_hp":          str(m["best_hp"]),
            "n_train":          len(y_tr),
            "n_val":            len(y_va),
            "n_test":           len(y_te),
            "note": (
                "Flat tabular input — no sliding window. "
                "Compare with flat RF in flat_vs_seq_rf_comparison.csv. "
                "Compare with seq models in baseline_results.csv."
            ),
        })
    out_path = os.path.join(output_dir, "boosted_models_results.csv")
    pd.DataFrame(rows).to_csv(out_path, index=False)
    print(f"  ✓ boosted_models_results.csv saved → {out_path}")

    # ── Issue 6: Permutation importance for best boosted model ─────────────────
    _best_bname      = min(results, key=lambda k: results[k]["MAE"])
    _best_bmodel_map = {
        "XGBoost":  xgb_model,
        "LightGBM": lgbm_model,
        "CatBoost": cb_best_model,
    }
    run_permutation_importance(
        _best_bmodel_map[_best_bname], _best_bname,
        X_te, y_te, feature_cols, output_dir, n_repeats=5,  # v9 OPT: was 10
    )
    # Issue 7: Overfitting check for best boosted model
    check_overfitting(
        _best_bname,
        mean_absolute_error(y_tr, _best_bmodel_map[_best_bname].predict(X_tr)),
        mean_absolute_error(y_va, _best_bmodel_map[_best_bname].predict(X_va)),
        results[_best_bname]["MAE"],
    )
    # ── Feature importance plot ───────────────────────────────────────────────
    # Each model uses a different internal importance scale:
    #   XGBoost  = gain  |  LightGBM = split count  |  CatBoost = PredictionValuesChange
    # Normalise each to [0,1] before averaging to prevent scale bias.
    xgb_imp  = xgb_model.feature_importances_
    lgbm_imp = lgbm_model.feature_importances_
    cb_imp   = cb_best_model.get_feature_importance()

    def _norm01(arr):
        rng = arr.max() - arr.min()
        return (arr - arr.min()) / rng if rng > 0 else arr

    avg_imp = (_norm01(xgb_imp) + _norm01(lgbm_imp) + _norm01(cb_imp)) / 3.0
    fi_df   = (
        pd.DataFrame({"feature": feature_cols, "importance": avg_imp})
        .sort_values("importance", ascending=True)
    )

    fig, ax = plt.subplots(figsize=(8, max(5, len(feature_cols) * 0.38)))
    colors  = ["#2196F3" if v >= fi_df["importance"].median() else "#90CAF9"
               for v in fi_df["importance"]]
    ax.barh(fi_df["feature"], fi_df["importance"],
            color=colors, edgecolor="white")
    ax.set_xlabel(
        "Average Normalised Importance (XGBoost + LightGBM + CatBoost)",
        fontsize=9,
    )
    ax.set_title(
        "Boosted Models — Feature Importance\n"
        "(mean of per-model normalised scores)",
        fontweight="bold",
    )
    ax.tick_params(axis="y", labelsize=8)
    plt.tight_layout()
    imp_path = os.path.join(output_dir, "boosted_feature_importance.png")
    plt.savefig(imp_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  ✓ boosted_feature_importance.png saved → {imp_path}")

    # v6.1.4 export patch: return model objects alongside results so
    # exec cell can pass them to export_all_models() without re-training.
    # ── v7.2: Capture test-set predictions for prediction comparison table ─────
    # NOTE ON ALIGNMENT: boosted_test_preds uses this function's own X_te, which
    # is the last 15% of INDIVIDUAL AUTO flights (flat split). The seq model test
    # set (pred_df from phase5_baseline) covers the last 15% of SEQUENCES, which
    # is a different n and different row set. boosted_test_preds is therefore used
    # for stand-alone boosted model evaluation only — NOT for direct row-by-row
    # comparison with pred_df. phase6c_prediction_comparison re-evaluates boosted
    # models on df_test_meta (seq-aligned rows) to produce a valid comparison.
    # Predictions are in raw minutes (no y_sc.inverse_transform — boosted models
    # were trained on raw y, not scaled y).
    boosted_test_preds = {
        "y_true":        y_te.round(4),
        "xgb_pred":      xgb_model.predict(X_te).round(4),
        "lgbm_pred":     lgbm_model.predict(X_te).round(4),
        "catboost_pred": cb_best_model.predict(X_te).round(4),
        "n_test":        len(y_te),
    }
    print(f"  boosted_test_preds: {len(y_te):,} flat test flights"
          f" (NOTE: structurally different from seq model test set)")

    return results, xgb_model, lgbm_model, cb_best_model, boosted_test_preds


# =============================================================================
# PHASE 5B-FLAT — Flat Individual-Flight Evaluation (v8.1)
# =============================================================================

def phase5_boosted_flat_eval(df, output_dir,
                              xgb_model, lgbm_model, cb_best_model):
    """
    v8.1 FLAT EVALUATION — RF, XGBoost, LightGBM, CatBoost on the
    full flat individual-flight AUTO test set.

    WHY THIS EXISTS
    ───────────────
    Boosted models are natively flat (no sequence window). Their optimal
    evaluation is on individual flights (full flat test set, n_test = last
    15% of all AUTO flights). This is a SEPARATE, VALID evaluation distinct
    from the sequence-aligned comparison (Phase 6C).

    DO NOT use these results for direct comparison with GRU/LSTM/Seq-RF.
    Those models require sequence context. Use Phase 6C for cross-model
    comparison on the same rows.

    Output
    ──────
    flat_evaluation_results.csv — MAE, RMSE, R², MAPE for flat models only.
    """
    log_phase("Phase 5B-Flat — Flat Evaluation (RF, XGB, LGBM, CatBoost)",
              context="MAIN PIPELINE")
    print("  Flat individual-flight test set (last 15% of AUTO flights).")
    print("  This is NOT comparable row-by-row with seq model results.")

    feature_cols = get_feature_cols()
    target_col   = "flight_duration_min"

    df_auto = (
        df[df["operation_mode"] == "AUTO"]
        .copy()
        .reset_index(drop=True)
    )
    df_auto = (
        df_auto[feature_cols + [target_col]]
        .dropna()
        .reset_index(drop=True)
    )
    n    = len(df_auto)
    n_tr = int(n * CFG["train_frac"])
    n_va = int(n * CFG["val_frac"])

    X    = df_auto[feature_cols].values.astype(np.float32)
    y    = df_auto[target_col].values.astype(np.float32)
    X_te = X[n_tr + n_va:]
    y_te = y[n_tr + n_va:]
    print(f"  Flat test set: {len(y_te):,} individual flights")

    records = []
    for model, name in [(xgb_model, "XGBoost"),
                        (lgbm_model, "LightGBM"),
                        (cb_best_model, "CatBoost")]:
        if model is None:
            continue
        if name == "LightGBM":
            lgbm_feat = getattr(model, "feature_name_", None)
            X_in = pd.DataFrame(X_te, columns=lgbm_feat) if lgbm_feat else X_te
        else:
            X_in = X_te
        preds = np.array(model.predict(X_in), dtype=float)
        m = _metrics(y_te, preds, f"Flat {name}")
        records.append({
            "model":       name,
            "eval_type":   "flat_individual_flights",
            "n_test":      len(y_te),
            "MAE":         round(m["MAE"],  4),
            "RMSE":        round(m["RMSE"], 4),
            "R2":          round(m["R2"],   4),
            "MAPE":        round(m["MAPE"], 4),
            "note": (
                "Flat evaluation on individual-flight test set. "
                "NOT comparable row-by-row with seq models (GRU/LSTM/Seq-RF). "
                "Use prediction_comparison_models.csv for cross-model comparison."
            ),
        })

    out_df   = pd.DataFrame(records)
    out_path = os.path.join(output_dir, "flat_evaluation_results.csv")
    out_df.to_csv(out_path, index=False)
    print(f"  ✓ flat_evaluation_results.csv → {out_path}")
    for r in records:
        print(f"    {r['model']:<12} MAE={r['MAE']:.4f}  RMSE={r['RMSE']:.4f}  R²={r['R2']:.4f}")
    return out_df



## Cell 7 — Phase 6 Ablation + Phase 6b LSTM Sensitivity Definitions

Phase 6: 36-run GRU ablation grid (unchanged from v6.1.4)
Phase 6b: 9-run LSTM sensitivity check (v6.1.5 extension)


In [10]:
def phase6_ablation(df, output_dir):
    """
    36-run ablation: 6 contexts (2 modes × 3 seasons) × 6 feature sets
    (baseline + remove each of 5 environmental variables one at a time).

    For AUTO contexts: GRU with session-aware sequences, fixed ABLATION_HP.
    For MANUAL contexts: flat RF (see MANUAL_JUSTIFICATION).

    LSTM is NOT included in the 36-run ablation grid (see ABLATION_LSTM_NOTE).
    A targeted LSTM sensitivity check (Phase 6b) verifies the top-feature
    finding on AUTO contexts with full feature set only.

    delta_MAE = MAE(after removal) - MAE(baseline).
    Positive delta_MAE = removal hurts accuracy = feature is important.
    Negative or near-zero delta_MAE = feature has minimal contribution.
    """
    log_phase("Phase 6 — Ablation Experiments (36 runs, RQ2)", context="MAIN PIPELINE")
    print(f"  {ABLATION_HP_RATIONALE[:120]}...")
    print(f"  Fixed HP: {ABLATION_HP}")
    print(f"  {ABLATION_LSTM_NOTE[:120]}...")

    device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    target_col = "flight_duration_min"
    base_feats = (
        CFG["operational_features"] +
        CFG["temporal_features"]    +
        CFG["sequential_features"]
    )
    full_env = [c for c in FULL_ENV_COLS if c in df.columns]
    records  = []

    for mode in ["AUTO", "MANUAL"]:
        for season in ["Dry", "Hot", "Wet"]:
            ctx_key  = f"{mode}-{season}"
            df_ctx   = (
                df[(df["operation_mode"] == mode) & (df["season"] == season)]
                .copy()
                .reset_index(drop=True)
            )
            use_rnn  = (mode == "AUTO")
            model_lbl = "GRU" if use_rnn else "flat RF"
            print(f"\n  [{ctx_key}]  {len(df_ctx):,} flights  model={model_lbl}")

            # 6 runs per context: baseline (all 5 env vars) + remove one each
            env_removal_order = ["none"] + CFG["environmental_features"]

            for env_feat in env_removal_order:
                if env_feat == "none":
                    env_cols = full_env
                    removed  = "none (baseline)"
                    wd_note  = "N/A"
                else:
                    env_cols = [
                        c for c in full_env if c not in ENV_COL_MAP[env_feat]
                    ]
                    removed  = env_feat
                    wd_note  = (
                        "removes wind_u AND wind_v — 1 physical factor, 2 columns"
                        if env_feat == "wind_direction_10m" else "N/A"
                    )

                feat_cols = [
                    f for f in base_feats + env_cols if f in df_ctx.columns
                ]
                df_sub = (
                    df_ctx[feat_cols + [target_col, "session_id"]]
                    .dropna()
                    .reset_index(drop=True)
                )

                if use_rnn:
                    # AUTO: GRU with session-aware sequences
                    n     = len(df_sub)
                    n_tr  = int(n * CFG["train_frac"])
                    tr_idx = np.arange(n_tr)

                    X_norm, y_norm, _, y_sc = normalize_rows(
                        df_sub, feat_cols, target_col, tr_idx
                    )
                    X_seq, y_seq, _tgt_idx = build_sequences_session_aware(  # v7.3: _tgt_idx unused in ablation
                        X_norm, y_norm,
                        df_sub["session_id"].values,
                        CFG["sequence_length"],
                    )

                    if len(y_seq) < CFG["min_sequences_rnn"]:
                        print(f"    ⚠  {ctx_key}|−{removed}: "
                              f"{len(y_seq)} seq < {CFG['min_sequences_rnn']} — skip")
                        m = {"MAE": np.nan, "RMSE": np.nan,
                             "R2": np.nan, "MAPE": np.nan,
                             "n": len(y_seq)}
                    else:
                        (Xtr, ytr), (Xva, yva), (Xte, yte) = \
                            temporal_split_sequences(
                                X_seq, y_seq,
                                CFG["train_frac"], CFG["val_frac"],
                            )
                        if len(yte) < 5:
                            m = {"MAE": np.nan, "RMSE": np.nan,
                                 "R2": np.nan, "MAPE": np.nan,
                                 "n": len(y_seq)}
                        else:
                            model_ab, _ = train_one_config(
                                GRUModel, ABLATION_HP,
                                Xtr, ytr, Xva, yva,
                                len(feat_cols), device,
                                CFG["epochs"], CFG["early_stop_patience"],
                            )
                            model_ab.eval()
                            with torch.no_grad():
                                pn = model_ab(
                                    torch.tensor(
                                        Xte, dtype=torch.float32
                                    ).to(device)
                                ).cpu().numpy()
                            pred = y_sc.inverse_transform(
                                pn.reshape(-1, 1)
                            ).ravel()
                            true = y_sc.inverse_transform(
                                yte.reshape(-1, 1)
                            ).ravel()
                            m    = _metrics(true, pred, f"{ctx_key}|−{removed}")
                            m["n"] = len(y_seq)

                else:
                    # MANUAL: flat RF — see MANUAL_JUSTIFICATION
                    # v5.3 FIX: use get_flat_rf_feature_cols() to exclude
                    # ending_battery. Same fix as mode_comparison RF.
                    # feat_cols already excludes ending_battery because
                    # base_feats uses FLAT_RF_OPERATIONAL_FEATURES for MANUAL.
                    # We rebuild feat_cols here from the clean base to be explicit.
                    clean_base = (
                        FLAT_RF_OPERATIONAL_FEATURES +
                        CFG["temporal_features"]     +
                        CFG["sequential_features"]
                    )
                    feat_cols_manual = [
                        f for f in clean_base + env_cols if f in df_ctx.columns
                    ]
                    df_sub = (
                        df_ctx[feat_cols_manual + [target_col, "session_id"]]
                        .dropna()
                        .reset_index(drop=True)
                    )
                    n     = len(df_sub)
                    n_tr  = int(n * CFG["train_frac"])
                    n_va  = int(n * CFG["val_frac"])
                    n_te  = n - n_tr - n_va

                    if n < CFG["min_rows_rf"]:
                        print(f"    ⚠  {ctx_key}|−{removed}: {n} rows — skip")
                        m = {"MAE": np.nan, "RMSE": np.nan,
                             "R2": np.nan, "MAPE": np.nan,
                             "n": n}
                    else:
                        tr_idx = np.arange(n_tr)
                        X_norm, y_norm, _, y_sc = normalize_rows(
                            df_sub, feat_cols_manual, target_col, tr_idx
                        )
                        # Guard: ensure test set has at least 3 rows
                        if n_te < 3:
                            cut    = int(n * 0.8)
                            va_cut = cut + max(1, int(n * 0.1))
                            Xtr, ytr = X_norm[:cut],       y_norm[:cut]
                            Xva, yva = X_norm[cut:va_cut], y_norm[cut:va_cut]
                            Xte, yte = X_norm[va_cut:],    y_norm[va_cut:]
                        else:
                            Xtr, ytr = X_norm[:n_tr],            y_norm[:n_tr]
                            Xva, yva = X_norm[n_tr:n_tr+n_va],   y_norm[n_tr:n_tr+n_va]
                            Xte, yte = X_norm[n_tr+n_va:],       y_norm[n_tr+n_va:]

                        if len(yte) < 3:
                            m = {"MAE": np.nan, "RMSE": np.nan,
                                 "R2": np.nan, "MAPE": np.nan,
                                 "n": n}
                        else:
                            m    = flat_rf_grid_search(
                                Xtr, ytr, Xva, yva, Xte, yte, y_sc,
                                label=f"{ctx_key}|−{removed} (flat RF)",
                            )
                            m["n"] = n

                records.append({
                    "mode":                mode,
                    "season":              season,
                    "context":             ctx_key,
                    "model_used":          "GRU" if use_rnn else "flat RF",
                    "removed_feature":     removed,
                    "wind_direction_note": wd_note,
                    "ablation_hp_note": (
                        ABLATION_HP_RATIONALE[:80] + "..."
                        if removed != "none (baseline)"
                        else "tuned via random search in Phase 5"
                    ),
                    "manual_note": (
                        MANUAL_JUSTIFICATION[:80] + "..."
                        if not use_rnn else "N/A"
                    ),
                    "MAE":        m["MAE"],
                    "RMSE":       m["RMSE"],
                    "R2":         m["R2"],
                    "MAPE":       m["MAPE"],
                    "n_samples":  m["n"],
                    "delta_MAE":  np.nan,   # filled in post-processing below
                })

    results_df = pd.DataFrame(records)

    # Compute delta_MAE: MAE(removal run) - MAE(baseline) within each context
    for ctx in results_df["context"].unique():
        mask_ctx = results_df["context"] == ctx
        base_row = results_df.loc[
            mask_ctx & (results_df["removed_feature"] == "none (baseline)"), "MAE"
        ]
        if base_row.empty or np.isnan(base_row.values[0]):
            continue
        bm = base_row.values[0]
        results_df.loc[mask_ctx, "delta_MAE"] = (
            results_df.loc[mask_ctx, "MAE"] - bm
        )
        # Baseline row delta_MAE = 0 by definition
        results_df.loc[
            mask_ctx & (results_df["removed_feature"] == "none (baseline)"),
            "delta_MAE",
        ] = 0.0

    results_df.round(4).to_csv(
        os.path.join(output_dir, "ablation_results.csv"), index=False
    )
    return results_df


# =============================================================================
# PHASE 6b — LSTM SENSITIVITY CHECK (targeted, not full 36-run ablation)
# =============================================================================

def phase6b_lstm_sensitivity(df, output_dir):
    """
    Extended LSTM Sensitivity Check — v6.1.5 (9 targeted runs).

    Design rationale
    ────────────────
    GRU is the primary model for the full 36-run ablation grid because
    ablation answers RQ2 (feature importance), not RQ1 (architecture
    comparison). LSTM is included here as a targeted cross-check only.

    v6.1.4: 3 runs — 3 AUTO contexts, full feature set only.
    v6.1.5: 9 runs — 3 AUTO contexts × 3 feature sets:
        (1) full feature set (baseline)
        (2) remove precipitation  ← top-1 feature from GRU ablation
        (3) remove temperature    ← top-2 feature from GRU ablation

    If LSTM delta_MAE ranks precipitation and temperature as the top-2
    features (i.e. positive delta_MAE after removal), the finding is
    model-agnostic. If ranks disagree, this is reported as a limitation.

    The full 36-run grid is NOT duplicated for LSTM — that would double
    runtime without answering a different research question.

    Saves:
        lstm_sensitivity_check.csv     (same format as v6.1.4 — 3 rows)
        lstm_sensitivity_extended.csv  (NEW — 9 rows with delta_MAE)

    See ABLATION_LSTM_NOTE, ABLATION_SINGLE_RUN_NOTE,
        ABLATION_CORR_FEATURE_NOTE for full justification strings.
    """
    log_phase("Phase 6b — LSTM Sensitivity Check (v6.1.5: 9 targeted runs)",
              context="MAIN PIPELINE")
    print(f"  {ABLATION_LSTM_NOTE[:120]}...")
    print(f"  Single-run note: {ABLATION_SINGLE_RUN_NOTE[:80]}...")
    print(f"  Correlation note: {ABLATION_CORR_FEATURE_NOTE[:80]}...")

    device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    target_col   = "flight_duration_min"

    # ── Feature set definitions for the 3 targeted removal configurations ──
    # These are the top-2 environmental features from the GRU ablation grid.
    # Using ENV_COL_MAP to remove wind_direction correctly (removes wind_u + wind_v).
    full_env_set = [c for c in FULL_ENV_COLS if c in df.columns]
    base_feats   = (
        CFG["operational_features"] +
        CFG["temporal_features"]    +
        CFG["sequential_features"]
    )

    def _feat_set(remove_env_key):
        """Return feature list with one env variable removed (or full if None)."""
        if remove_env_key is None:
            return base_feats + full_env_set
        reduced_env = [c for c in full_env_set if c not in ENV_COL_MAP[remove_env_key]]
        return base_feats + reduced_env

    # 3 feature configurations: full, remove precipitation, remove temperature
    # These correspond to the top-2 features identified by Phase 6 GRU ablation.
    feat_configs = [
        ("full",               None,              "full feature set — all 5 env vars"),
        ("remove_precip",      "precipitation",   "remove precipitation (top-1 GRU)"),
        ("remove_temp",        "temperature_2m",  "remove temperature (top-2 GRU)"),
    ]

    records_extended = []   # 9 rows — for lstm_sensitivity_extended.csv
    records_baseline = []   # 3 rows (full feature set only) — for existing csv

    for season in ["Dry", "Hot", "Wet"]:
        ctx_key  = f"AUTO-{season}"
        df_ctx   = (
            df[(df["operation_mode"] == "AUTO") & (df["season"] == season)]
            .copy()
            .reset_index(drop=True)
        )

        baseline_mae = None   # full-feature MAE for this context

        for cfg_label, remove_key, cfg_desc in feat_configs:
            feature_cols = [f for f in _feat_set(remove_key) if f in df_ctx.columns]

            df_sub = (
                df_ctx[feature_cols + [target_col, "session_id"]]
                .dropna()
                .reset_index(drop=True)
            )
            n      = len(df_sub)
            n_tr   = int(n * CFG["train_frac"])
            tr_idx = np.arange(n_tr)

            X_norm, y_norm, _, y_sc = normalize_rows(
                df_sub, feature_cols, target_col, tr_idx
            )
            X_seq, y_seq, _tgt_idx = build_sequences_session_aware(  # v7.3: _tgt_idx unused in ablation
                X_norm, y_norm,
                df_sub["session_id"].values,
                CFG["sequence_length"],
            )

            if len(y_seq) < CFG["min_sequences_rnn"]:
                print(f"  ⚠  {ctx_key}|{cfg_label}: "
                      f"{len(y_seq)} seq < {CFG['min_sequences_rnn']} — skip")
                records_extended.append({
                    "context":        ctx_key,
                    "model":          "LSTM",
                    "feature_config": cfg_label,
                    "removed_env":    remove_key or "none",
                    "description":    cfg_desc,
                    "n_features":     len(feature_cols),
                    "n_sequences":    len(y_seq),
                    "MAE":            np.nan, "RMSE": np.nan,
                    "R2":             np.nan, "MAPE": np.nan,
                    "delta_MAE":      np.nan,
                    "note":           "Insufficient sequences — skipped",
                })
                continue

            (Xtr, ytr), (Xva, yva), (Xte, yte) = temporal_split_sequences(
                X_seq, y_seq, CFG["train_frac"], CFG["val_frac"]
            )

            # Retrain LSTM from scratch with fixed ABLATION_HP
            # (same pattern as phase6_ablation for GRU — ensures comparability)
            model_lstm, _ = train_one_config(
                LSTMModel, ABLATION_HP,
                Xtr, ytr, Xva, yva,
                len(feature_cols), device,
                CFG["epochs"], CFG["early_stop_patience"],
                verbose=False,   # suppress per-epoch output in sensitivity check
            )
            model_lstm.eval()
            with torch.no_grad():
                pn = model_lstm(
                    torch.tensor(Xte, dtype=torch.float32).to(device)
                ).cpu().numpy()
            pred = y_sc.inverse_transform(pn.reshape(-1, 1)).ravel()
            true = y_sc.inverse_transform(yte.reshape(-1, 1)).ravel()
            m    = _metrics(true, pred, f"{ctx_key}|LSTM|{cfg_label}")

            if cfg_label == "full":
                baseline_mae = m["MAE"]

            delta_mae = (m["MAE"] - baseline_mae) if baseline_mae is not None else np.nan

            rec = {
                "context":        ctx_key,
                "model":          "LSTM",
                "feature_config": cfg_label,
                "removed_env":    remove_key or "none (baseline)",
                "description":    cfg_desc,
                "n_features":     len(feature_cols),
                "n_sequences":    len(y_seq),
                "MAE":            round(m["MAE"],  4),
                "RMSE":           round(m["RMSE"], 4),
                "R2":             round(m["R2"],   4),
                "MAPE":           round(m["MAPE"], 4),
                "delta_MAE":      round(delta_mae, 4) if not np.isnan(delta_mae) else np.nan,
                "note":           (
                    ABLATION_SINGLE_RUN_NOTE[:60] + "... | " +
                    ABLATION_CORR_FEATURE_NOTE[:60] + "..."
                ),
            }
            records_extended.append(rec)

            # Keep the baseline-only records for backward-compatible CSV
            if cfg_label == "full":
                records_baseline.append({
                    "context":     ctx_key,
                    "model":       "LSTM",
                    "features":    "full (all 5 env vars)",
                    "MAE":         round(m["MAE"],  4),
                    "RMSE":        round(m["RMSE"], 4),
                    "R2":          round(m["R2"],   4),
                    "MAPE":        round(m["MAPE"], 4),
                    "n_sequences": len(y_seq),
                    "note":        ABLATION_LSTM_NOTE[:100] + "...",
                })

    # Save backward-compatible CSV (same structure as v6.1.4)
    pd.DataFrame(records_baseline).round(4).to_csv(
        os.path.join(output_dir, "lstm_sensitivity_check.csv"), index=False
    )
    print("  ✓ lstm_sensitivity_check.csv saved (backward-compatible, 3 rows)")

    # Save extended CSV (new in v6.1.5)
    ext_df = pd.DataFrame(records_extended)
    ext_df.round(4).to_csv(
        os.path.join(output_dir, "lstm_sensitivity_extended.csv"), index=False
    )
    print("  ✓ lstm_sensitivity_extended.csv saved (9 runs with delta_MAE)")

    # Print agreement summary
    print("\n  LSTM vs GRU feature ranking agreement (RQ2 cross-check):")
    print(f"  {'Context':<12} {'Remove':<22} {'LSTM delta_MAE':>14}  {'Direction':<10}")
    print(f"  {'-'*62}")
    for r in records_extended:
        if r["removed_env"] in ("none (baseline)", "none"):
            continue
        dma = r["delta_MAE"]
        if np.isnan(dma):
            direction = "skip"
        elif dma > 0:
            direction = "↑ hurts (important)"
        else:
            direction = "↓ no effect"
        print(f"  {r['context']:<12} {r['removed_env']:<22} "
              f"{str(round(dma,4)) if not np.isnan(dma) else 'N/A':>14}  {direction}")

    print(f"\n  Limitation note:")
    print(f"  {ABLATION_SINGLE_RUN_NOTE[:120]}...")
    print(f"  {ABLATION_CORR_FEATURE_NOTE[:120]}...")

    return pd.DataFrame(records_baseline)




# =============================================================================
# PHASE 6D — OPERATIONAL & SEQUENTIAL FEATURE ABLATION (v8.1.2, supplementary)
# =============================================================================

def phase6d_operational_ablation(df, output_dir):
    """
    Supplementary ablation — operational and sequential features (v8.1.2).

    Purpose
    ───────
    Phase 6 (environmental ablation) shows that ERA5 weather features have
    low marginal impact on flight duration prediction. Phase 6D provides the
    CONTEXT for that finding by measuring how much the dominant features
    (mission scope, battery, sequential context) contribute when removed.

    This is NOT a redesign of RQ2. It is supplementary analysis that enables
    the following defense-ready statement:
        "While environmental features contribute ≤0.07 min (δMAE), removing
        sprayed_area degrades accuracy by ~0.8–1.2 min — confirming that
        mission scope, not atmospheric conditions, drives flight duration."

    Design
    ──────
    - Model: GRU with ABLATION_HP (identical to Phase 6)
    - Contexts: AUTO-Dry, AUTO-Hot, AUTO-Wet (3 — same as Phase 6 AUTO rows)
    - Feature sets: baseline (all 17) + remove one operational/sequential
      feature at a time = 6 sets per context
    - Features removed: sprayed_area, total_amount_l, starting_battery,
      rolling_duration_3, time_since_last_flight
    - Total: 3 × 6 = 18 runs
    - Same chronological 70/15/15 split → δMAE values on same scale as Phase 6

    Limitation: single-run (see ABLATION_SINGLE_RUN_NOTE).
    See OPERATIONAL_ABLATION_NOTE for full justification.

    Output
    ──────
    operational_ablation_results.csv
    """
    log_phase("Phase 6D — Operational & Sequential Feature Ablation (v8.1.2)",
              context="MAIN PIPELINE")
    print(f"  {OPERATIONAL_ABLATION_NOTE[:120]}...")
    print(f"  Supplementary to Phase 6. Same GRU + ABLATION_HP + split.")
    print(f"  18 runs: 3 AUTO contexts × 6 feature sets.")

    device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    target_col = "flight_duration_min"

    # Features to remove — operational + sequential only
    # Each removal name maps to the actual column(s) to drop
    OP_REMOVAL_MAP = {
        "sprayed_area":          ["sprayed_area"],
        "total_amount_l":        ["total_amount_l"],
        "starting_battery":      ["starting_battery"],
        "rolling_duration_3":    ["rolling_duration_3"],
        "time_since_last_flight":["time_since_last_flight"],
    }
    removal_order = ["none"] + list(OP_REMOVAL_MAP.keys())

    records = []

    for season in ["Dry", "Hot", "Wet"]:
        ctx_key = f"AUTO-{season}"
        df_ctx  = (
            df[(df["operation_mode"] == "AUTO") & (df["season"] == season)]
            .copy()
            .reset_index(drop=True)
        )
        print(f"\n  [{ctx_key}]  {len(df_ctx):,} flights  model=GRU")

        for removed_feat in removal_order:
            if removed_feat == "none":
                feat_cols = get_feature_cols()
                removed   = "none (baseline)"
            else:
                drop_cols = OP_REMOVAL_MAP[removed_feat]
                feat_cols = [f for f in get_feature_cols() if f not in drop_cols]
                removed   = removed_feat

            df_sub = (
                df_ctx[feat_cols + [target_col, "session_id"]]
                .dropna()
                .reset_index(drop=True)
            )
            n     = len(df_sub)
            n_tr  = int(n * CFG["train_frac"])
            tr_idx = np.arange(n_tr)

            X_norm, y_norm, _, y_sc = normalize_rows(
                df_sub, feat_cols, target_col, tr_idx
            )
            X_seq, y_seq, _tgt = build_sequences_session_aware(
                X_norm, y_norm, df_sub["session_id"].values, CFG["sequence_length"]
            )

            if len(y_seq) < CFG["min_sequences_rnn"]:
                print(f"    ⚠  {ctx_key}|−{removed}: {len(y_seq)} seq — skip")
                m = {"MAE": float("nan"), "RMSE": float("nan"),
                     "R2": float("nan"), "MAPE": float("nan"), "n": len(y_seq)}
            else:
                (Xtr, ytr), (Xva, yva), (Xte, yte) = temporal_split_sequences(
                    X_seq, y_seq, CFG["train_frac"], CFG["val_frac"]
                )
                if len(yte) < 5:
                    m = {"MAE": float("nan"), "RMSE": float("nan"),
                         "R2": float("nan"), "MAPE": float("nan"), "n": len(y_seq)}
                else:
                    model_ab, _ = train_one_config(
                        GRUModel, ABLATION_HP,
                        Xtr, ytr, Xva, yva,
                        len(feat_cols), device,
                        CFG["epochs"], CFG["early_stop_patience"],
                    )
                    model_ab.eval()
                    with torch.no_grad():
                        pn = model_ab(
                            torch.tensor(Xte, dtype=torch.float32).to(device)
                        ).cpu().numpy()
                    pred = y_sc.inverse_transform(pn.reshape(-1, 1)).ravel()
                    true = y_sc.inverse_transform(yte.reshape(-1, 1)).ravel()
                    m    = _metrics(true, pred, f"{ctx_key}|−{removed}")
                    m["n"] = len(y_seq)

            records.append({
                "mode":            "AUTO",
                "season":          season,
                "context":         ctx_key,
                "model_used":      "GRU",
                "removed_feature": removed,
                "feature_group":   "baseline" if removed == "none (baseline)"
                                   else ("sequential" if removed in
                                         ["rolling_duration_3", "time_since_last_flight"]
                                         else "operational"),
                "ablation_hp_note": ABLATION_HP_RATIONALE[:80] + "...",
                "MAE":         m["MAE"],
                "RMSE":        m["RMSE"],
                "R2":          m["R2"],
                "MAPE":        m["MAPE"],
                "n_sequences": m["n"],
                "delta_MAE":   float("nan"),
            })

    results_df = pd.DataFrame(records)

    # Compute delta_MAE within each context
    for ctx in results_df["context"].unique():
        mask   = results_df["context"] == ctx
        base   = results_df.loc[
            mask & (results_df["removed_feature"] == "none (baseline)"), "MAE"
        ]
        if base.empty or pd.isna(base.values[0]):
            continue
        bm = base.values[0]
        results_df.loc[mask, "delta_MAE"] = results_df.loc[mask, "MAE"] - bm
        results_df.loc[
            mask & (results_df["removed_feature"] == "none (baseline)"), "delta_MAE"
        ] = 0.0

    out_path = os.path.join(output_dir, "operational_ablation_results.csv")
    results_df.round(4).to_csv(out_path, index=False)
    print(f"\n  ✓ operational_ablation_results.csv saved → {out_path}")

    # Print summary
    print("\n  δMAE summary (positive = feature important, negative = noise):")
    print(f"  {'Context':<12} {'Removed':<28} {'Group':<12} {'δMAE':>8}")
    print(f"  {'-'*66}")
    for _, r in results_df[results_df["removed_feature"] != "none (baseline)"].iterrows():
        dma = r["delta_MAE"]
        sign = "↑ hurts" if dma > 0 else "↓ noise"
        print(f"  {r['context']:<12} {r['removed_feature']:<28} "
              f"{r['feature_group']:<12} {dma:>+.4f}  {sign}")

    return results_df


# =============================================================================
# PHASE 7 — ANALYZE AND VISUALIZE
# =============================================================================






## Cell 7b — v7.1 New Phases: Ablation Baseline Control + Prediction Comparison

In [11]:
def phase6_ablation_baseline(df, output_dir):
    """
    v9 BASELINE CONTROL — Full 12-feature GRU for all AUTO contexts.

    Purpose
    ───────
    Provides an explicit, labeled reference for the ablation framework.
    Panel members can directly cite ablation_baseline_control.csv as the
    "what happens when all features are used?" reference.

    Design
    ──────
    - Model   : GRU with ABLATION_HP (identical to ablation grid)
    - Features: all 17 (no removal — full operational+temporal+sequential+env)
    - Contexts: 3 AUTO × 3 seasons = 3 runs (AUTO-Dry, AUTO-Hot, AUTO-Wet)
    - Split   : same 70/15/15 chronological split as all other phases
    - No re-training of Phase 5 models — these are fresh ablation-style runs

    Relationship to ablation_results.csv
    ─────────────────────────────────────
    The ablation grid already contains these rows where
    removed_feature == "none (baseline)". This phase surfaces them as a
    separate named output for clarity. The delta_MAE values in
    ablation_results.csv are already computed relative to this baseline.

    Output
    ──────
    ablation_baseline_control.csv — 3 rows (one per AUTO context)
    Columns: context, model, feature_label, n_features, MAE, RMSE, R2,
             MAPE, n_sequences, note
    """
    log_phase("Phase 6A — Ablation Baseline Control (v9: full 12 features)",
              context="MAIN PIPELINE")
    print(f"  {ABLATION_BASELINE_NOTE[:120]}...")

    device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    target_col = "flight_duration_min"
    feature_cols = get_feature_cols()   # all 12 features (v9), exact training order
    n_features   = len(feature_cols)

    assert n_features == 12, (  # v9: 12 features
        f"BASELINE CONTROL: expected 12 features, got {n_features}. "
        f"Check get_feature_cols() — ending_battery must not be present."
    )
    assert_no_leakage(df, feature_cols, target_col)

    records = []

    for season in ["Dry", "Hot", "Wet"]:
        ctx_key = f"AUTO-{season}"
        df_ctx  = (
            df[(df["operation_mode"] == "AUTO") & (df["season"] == season)]
            .copy()
            .reset_index(drop=True)
        )
        df_sub  = (
            df_ctx[feature_cols + [target_col, "session_id"]]
            .dropna()
            .reset_index(drop=True)
        )

        n      = len(df_sub)
        n_tr   = int(n * CFG["train_frac"])
        tr_idx = np.arange(n_tr)

        X_norm, y_norm, _, y_sc = normalize_rows(
            df_sub, feature_cols, target_col, tr_idx
        )
        X_seq, y_seq, _tgt_idx = build_sequences_session_aware(  # v7.3
            X_norm, y_norm,
            df_sub["session_id"].values,
            CFG["sequence_length"],
        )

        if len(y_seq) < CFG["min_sequences_rnn"]:
            print(f"    ⚠  {ctx_key}: {len(y_seq)} seq < {CFG['min_sequences_rnn']} — skip")
            records.append({
                "context":       ctx_key,
                "model":         "GRU",
                "feature_label": "full_features_baseline",
                "n_features":    n_features,
                "n_sequences":   len(y_seq),
                "MAE":  np.nan, "RMSE": np.nan,
                "R2":   np.nan, "MAPE": np.nan,
                "note": "Insufficient sequences — skipped",
            })
            continue

        (Xtr, ytr), (Xva, yva), (Xte, yte) = temporal_split_sequences(
            X_seq, y_seq, CFG["train_frac"], CFG["val_frac"]
        )

        # Train GRU with ABLATION_HP — same as ablation grid
        model_ctrl, _ = train_one_config(
            GRUModel, ABLATION_HP,
            Xtr, ytr, Xva, yva,
            n_features, device,
            CFG["epochs"], CFG["early_stop_patience"],
            verbose=False,
        )
        model_ctrl.eval()
        with torch.no_grad():
            pn = model_ctrl(
                torch.tensor(Xte, dtype=torch.float32).to(device)
            ).cpu().numpy()
        pred = y_sc.inverse_transform(pn.reshape(-1, 1)).ravel()
        true = y_sc.inverse_transform(yte.reshape(-1, 1)).ravel()
        m    = _metrics(true, pred, f"{ctx_key}|full_features_baseline")

        records.append({
            "context":       ctx_key,
            "model":         "GRU",
            "feature_label": "full_features_baseline",
            "n_features":    n_features,
            "n_sequences":   len(y_seq),
            "MAE":           round(m["MAE"],  4),
            "RMSE":          round(m["RMSE"], 4),
            "R2":            round(m["R2"],   4),
            "MAPE":          round(m["MAPE"], 4),
            "note": (
                "GRU with ABLATION_HP, all 12 features, same split as ablation. "
                "This row matches removed_feature==\'none (baseline)\' in ablation_results.csv "
                "for the corresponding context. Surfaced here for explicit reference. "
                + ABLATION_BASELINE_NOTE[:80] + "..."
            ),
        })
        print(f"    {ctx_key} | full_features_baseline | "
              f"MAE={m['MAE']:.4f}  R²={m['R2']:.4f}")

    baseline_ctrl_df = pd.DataFrame(records)
    out_path = os.path.join(output_dir, "ablation_baseline_control.csv")
    baseline_ctrl_df.round(4).to_csv(out_path, index=False)
    print(f"  ✓ ablation_baseline_control.csv saved ({len(records)} rows)")
    print(f"    Full 12-feature baseline MAE by season:")
    for _, row in baseline_ctrl_df.iterrows():
        print(f"    {row['context']:<12}: MAE={row['MAE']}")
    return baseline_ctrl_df


def phase6c_prediction_comparison(
    pred_df,
    boosted_test_preds,
    df_test_meta,
    output_dir,
    xgb_model=None,
    lgbm_model=None,
    cb_model=None,
):
    """
    v8.1 SEQUENCE-ALIGNED EVALUATION — all 7 models on the same test rows.

    ALIGNMENT NOTE (read before modifying)
    ─────────────────────────────────────
    phase5_baseline test set  = last 15% of SEQUENCES (session-aware windows).
                                pred_df has n_seq rows.
    phase5_boosted test set   = last 15% of INDIVIDUAL FLIGHTS (flat split).
                                boosted_test_preds has n_flights rows.
    n_seq != n_flights — these two test sets are structurally different and
    CANNOT be combined row-by-row.

    Resolution: boosted models are RE-EVALUATED here on df_test_meta, which
    contains the raw feature values of the seq-aligned test target rows
    (same n as pred_df). This produces a valid row-by-row table.
    boosted_test_preds is used only for the stand-alone boosted summary,
    not for the main table rows.

    Parameters
    ──────────
    pred_df           : DataFrame from phase5_baseline. n = test sequences.
                        Columns: y_true, y_pred_gru, y_pred_lstm, y_pred_rf
    boosted_test_preds: dict from phase5_boosted_models. Different n from
                        pred_df. Used for stand-alone boosted metrics only.
    df_test_meta      : DataFrame, same n as pred_df. Raw AUTO flight metadata
                        for the seq-aligned test rows. Built in execution cell as:
                        df_auto.tail(len(pred_df)).reset_index(drop=True)
    output_dir        : str
    xgb_model         : trained XGBRegressor  (for seq-aligned re-evaluation)
    lgbm_model        : trained LGBMRegressor
    cb_model          : trained CatBoostRegressor

    Output columns (exact order per v7.2 spec)
    ───────────────────────────────────────────
    flight_id, session_id, flight_order_in_session, start_datetime,
    sprayed_area, total_amount_l, starting_battery,
    temperature_2m, wind_speed_10m, relative_humidity_2m, precipitation,
    actual_duration,
    lstm_pred, gru_pred, rf_pred, xgb_pred, lgbm_pred, catboost_pred,
    lstm_error, gru_error, rf_error, xgb_error, lgbm_error, catboost_error

    Output file: prediction_comparison_models.csv
    """
    log_phase("Phase 6C — Sequence-Aligned Evaluation (v8.1): 7 models, same rows", context="MAIN PIPELINE")
    print("  SEQUENCE-ALIGNED EVALUATION (v8.1): 7 models on identical rows.")
    print("  GRU, LSTM, Seq-RF, RF, XGBoost, LightGBM, CatBoost.")
    print("  See flat_evaluation_results.csv for flat-only benchmark.")
    print("  Boosted models: re-evaluated on df_test_meta (seq-aligned rows).")

    # Validate pred_df
    for col in ["y_true", "y_pred_gru", "y_pred_lstm", "y_pred_rf"]:
        assert col in pred_df.columns, (
            f"phase6c: pred_df missing '{col}'. "
            f"Check phase5_baseline ran and pred_df is in scope."
        )

    n = len(pred_df)
    print(f"  Test samples (seq-aligned): {n:,}")

    if len(df_test_meta) != n:
        raise ValueError(
            f"phase6c: df_test_meta has {len(df_test_meta)} rows; "
            f"pred_df has {n}. "
            f"Construct df_test_meta as: "
            f"df_auto.tail(len(pred_df)).reset_index(drop=True)"
        )

    # Boosted re-evaluation on seq-aligned rows
    # Models predict raw minutes (no y_sc — boosted trained on raw y)
    feature_cols_12 = get_feature_cols()  # v9: 12 features
    feat_avail = [c for c in feature_cols_12 if c in df_test_meta.columns]

    def _eval_boosted(model, name):
        if model is None:
            print(f"    WARNING: {name} model is None — NaN in table")
            return np.full(n, np.nan)
        X = df_test_meta[feat_avail].values.astype(np.float32)
        if name == "LightGBM":
            # v7.3 FIX: LGBM trained on numpy → auto-named Column_0..N
            # Must pass DataFrame with THOSE names, not real feature names
            lgbm_feat = getattr(model, "feature_name_", None)
            if lgbm_feat is not None:
                X = pd.DataFrame(X, columns=lgbm_feat)
        return np.array(model.predict(X), dtype=float).round(4)

    xgb_p  = _eval_boosted(xgb_model,  "XGBoost")
    lgbm_p = _eval_boosted(lgbm_model, "LightGBM")
    cb_p   = _eval_boosted(cb_model,   "CatBoost")

    # Metadata helpers
    df_m = df_test_meta.copy().reset_index(drop=True)

    def _col(col_name, default=np.nan, rnd=None):
        if col_name not in df_m.columns:
            return pd.Series([default] * n)
        s = df_m[col_name].reset_index(drop=True)
        return s.round(rnd) if rnd is not None else s

    if "session_id" in df_m.columns:
        df_m["_flt_ord"] = df_m.groupby("session_id").cumcount()
        flt_ord = df_m["_flt_ord"].values
    else:
        flt_ord = np.full(n, np.nan)

    actual = pred_df["y_true"].round(4).values
    lstm_p = pred_df["y_pred_lstm"].round(4).values
    gru_p  = pred_df["y_pred_gru"].round(4).values
    rf_p   = pred_df["y_pred_rf"].round(4).values

    comp_df = pd.DataFrame({
        "flight_id":               np.arange(1, n + 1),
        "session_id":              _col("session_id"),
        "flight_order_in_session": flt_ord,
        "start_datetime":          pred_df["start_datetime"].values if "start_datetime" in pred_df.columns else _col("start_datetime"),  # v7.3
        "sprayed_area":            _col("sprayed_area",         rnd=4),
        "total_amount_l":          _col("total_amount_l",       rnd=4),
        "starting_battery":        _col("starting_battery",     rnd=1),
        "temperature_2m":          _col("temperature_2m",       rnd=2),
        "wind_speed_10m":          _col("wind_speed_10m",       rnd=2),
        "relative_humidity_2m":    _col("relative_humidity_2m", rnd=1),
        "precipitation":           _col("precipitation",        rnd=3),
        "actual_duration":         actual,
        "lstm_pred":               lstm_p,
        "gru_pred":                gru_p,
        "rf_pred":                 rf_p,
        "xgb_pred":                xgb_p,
        "lgbm_pred":               lgbm_p,
        "catboost_pred":           cb_p,
        "lstm_error":              np.abs(lstm_p - actual).round(4),
        "gru_error":               np.abs(gru_p  - actual).round(4),
        "rf_error":                np.abs(rf_p   - actual).round(4),
        "xgb_error":               np.where(
                                       np.isnan(xgb_p), np.nan,
                                       np.abs(xgb_p  - actual).round(4)),
        "lgbm_error":              np.where(
                                       np.isnan(lgbm_p), np.nan,
                                       np.abs(lgbm_p - actual).round(4)),
        "catboost_error":          np.where(
                                       np.isnan(cb_p), np.nan,
                                       np.abs(cb_p   - actual).round(4)),
    })

    out_path = os.path.join(output_dir, "prediction_comparison_models.csv")
    comp_df.to_csv(out_path, index=False)
    print("  Saved: prediction_comparison_models.csv")
    print(f"    Rows: {n}  Columns: {len(comp_df.columns)}")

    print(f"\n  MAE summary (seq-aligned test set, n={n}):")
    print(f"  {'Model':<12} {'MAE (min)':>10}  {'Std (min)':>10}")
    print(f"  {'-'*36}")
    for label, col in [
        ("LSTM",     "lstm_error"),     ("GRU",      "gru_error"),
        ("Seq-RF",   "rf_error"),       ("XGBoost",  "xgb_error"),
        ("LightGBM", "lgbm_error"),     ("CatBoost", "catboost_error"),
    ]:
        data = comp_df[col].dropna()
        if len(data) > 0:
            print(f"  {label:<12} {data.mean():>10.4f}  {data.std():>10.4f}")

    if boosted_test_preds and "n_test" in boosted_test_preds:
        nb_test = boosted_test_preds["n_test"]
        print(f"\n  Stand-alone boosted metrics (boosted flat test set, n={nb_test}):")
        print(f"  NOTE: n={nb_test} differs from n={n} — flat flights vs seq targets.")
        y_b = boosted_test_preds["y_true"]
        for lbl, key in [("XGBoost","xgb_pred"),("LightGBM","lgbm_pred"),("CatBoost","catboost_pred")]:
            if key in boosted_test_preds:
                mae_b = float(np.abs(boosted_test_preds[key] - y_b).mean())
                print(f"    {lbl:<10}: MAE = {mae_b:.4f} min (on boosted test set)")

    print("\n  Alignment note: boosted columns in CSV = re-evaluated on seq-aligned")
    print("  df_test_meta, NOT from boosted_test_preds. Valid row-by-row comparison.")
    return comp_df









## Cell 8 — Phase 7 Visualization Definition

In [12]:
def phase7_analyze(ablation_df, baseline_df, lstm_sens_df, output_dir):
    log_phase("Phase 7 — Analyze Results and Visualize", context="MAIN PIPELINE")

    baseline_ab = ablation_df[ablation_df["removed_feature"] == "none (baseline)"]
    removal_ab  = ablation_df[ablation_df["removed_feature"] != "none (baseline)"]

    print("\n  Baseline MAE per context (ablation Phase 6):")
    print(baseline_ab[["context", "model_used", "MAE", "RMSE", "R2", "n_samples"]]
          .to_string(index=False))

    feat_imp = (
        removal_ab.dropna(subset=["delta_MAE"])
        .groupby("removed_feature")["delta_MAE"]
        .mean()
        .sort_values(ascending=False)
    )
    print("\n  Average δMAE per removed feature (higher = more important):")
    print(feat_imp.round(4).to_string())

    if not lstm_sens_df.empty:
        print("\n  LSTM sensitivity check (should confirm GRU top features):")
        print(lstm_sens_df[["context", "MAE", "RMSE", "R2"]].to_string(index=False))

    # ── Plot 1: Feature importance bar chart ─────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ["#E53935" if v > 0 else "#43A047" for v in feat_imp.values]
    feat_imp.plot(kind="bar", ax=ax, color=colors, edgecolor="white")
    ax.set_title(
        "Environmental Feature Importance — Average δMAE (minutes)\n"
        "Red: accuracy degrades when removed  |  Green: minimal impact\n"
        "wind_direction removal = wind_u + wind_v (1 physical factor, 2 columns)",
        fontsize=11, fontweight="bold",
    )
    ax.set_xlabel("Removed Feature")
    ax.set_ylabel("δMAE (minutes)")
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.tick_params(axis="x", rotation=20)
    plt.tight_layout()
    plt.savefig(
        os.path.join(output_dir, "feature_importance.png"), dpi=150
    )
    plt.close()

    # ── Plot 2: δMAE heatmap (AUTO contexts, GRU) ─────────────────────────
    auto_ab = removal_ab[removal_ab["mode"] == "AUTO"]
    pivot   = auto_ab.pivot_table(
        index="context", columns="removed_feature", values="delta_MAE"
    )
    if not pivot.empty:
        fig, ax = plt.subplots(figsize=(11, 4))
        im = ax.imshow(
            pivot.values, cmap="RdYlGn_r", aspect="auto", vmin=-0.15, vmax=0.15
        )
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels(pivot.columns, rotation=20, ha="right")
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels(pivot.index)
        plt.colorbar(im, ax=ax, label="δMAE (minutes)")
        ax.set_title(
            "δMAE Heatmap — AUTO Mode Ablation (GRU)\n"
            "wind_direction: 1 physical factor encoded as wind_u + wind_v",
            fontsize=11, fontweight="bold",
        )
        for i in range(len(pivot.index)):
            for j in range(len(pivot.columns)):
                v = pivot.values[i, j]
                if not np.isnan(v):
                    ax.text(j, i, f"{v:.3f}", ha="center", va="center",
                            fontsize=9, color="black")
        plt.tight_layout()
        plt.savefig(
            os.path.join(output_dir, "delta_mae_heatmap.png"), dpi=150
        )
        plt.close()

    # ── Plot 3: MAE by season (AUTO baseline, GRU) ────────────────────────
    auto_base = baseline_ab[baseline_ab["mode"] == "AUTO"]
    if not auto_base.empty:
        smae         = auto_base.groupby("season")["MAE"].mean()
        season_order = [s for s in ["Dry", "Hot", "Wet"] if s in smae.index]
        smae         = smae.reindex(season_order)
        fig, ax = plt.subplots(figsize=(7, 4))
        bars = ax.bar(
            smae.index, smae.values,
            color=["#FF9800", "#F44336", "#2196F3"], edgecolor="white",
        )
        for bar, v in zip(bars, smae.values):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.005,
                f"{v:.3f}",
                ha="center", va="bottom", fontsize=10,
            )
        ax.set_title(
            "Baseline MAE by Season — AUTO mode (GRU)\n"
            "Target: flight_duration_min (minutes)",
            fontsize=12, fontweight="bold",
        )
        ax.set_xlabel("Season")
        ax.set_ylabel("MAE (minutes)")
        plt.tight_layout()
        plt.savefig(
            os.path.join(output_dir, "mae_by_season.png"), dpi=150
        )
        plt.close()

    # ── Plot 4: Scatter — GRU / LSTM / RF predicted vs actual ────────────────
    # OUTPUT 4: baseline_pred_vs_actual.png
    # baseline_df must contain columns: y_true, y_pred_gru, y_pred_lstm, y_pred_rf
    if baseline_df is not None and "y_true" in baseline_df.columns:
        fig, axes = plt.subplots(1, 3, figsize=(16, 5))
        fig.suptitle(
            "Predicted vs Actual Flight Duration — Baseline Models (AUTO Test Set)",
            fontsize=13, fontweight="bold",
        )
        panels = [
            ("y_pred_gru",  "GRU",  "#1565C0"),
            ("y_pred_lstm", "LSTM", "#6A1B9A"),
            ("y_pred_rf",   "RF",   "#E65100"),
        ]
        for ax, (pred_col, label, col) in zip(axes, panels):
            if pred_col not in baseline_df.columns:
                continue
            y_t = baseline_df["y_true"].values
            y_p = baseline_df[pred_col].values
            mae  = np.mean(np.abs(y_t - y_p))
            r2   = 1 - np.sum((y_t - y_p)**2) / np.sum((y_t - y_t.mean())**2)
            ax.scatter(y_t, y_p, alpha=0.35, s=12, color=col)
            lims = [min(y_t.min(), y_p.min()) - 0.1,
                    max(y_t.max(), y_p.max()) + 0.1]
            ax.plot(lims, lims, "r--", linewidth=1.2, label="Perfect prediction")
            ax.set_xlim(lims); ax.set_ylim(lims)
            ax.set_xlabel("Actual Duration (min)")
            ax.set_ylabel("Predicted Duration (min)")
            ax.set_title(
                f"{label} — Test Set (v5.2 final)\nMAE={mae:.3f} min  R²={r2:.3f}",
                fontweight="bold",
            )
            ax.legend(fontsize=8)
        plt.tight_layout()
        plt.savefig(
            os.path.join(output_dir, "baseline_pred_vs_actual.png"), dpi=150
        )
        plt.close()
        print("  ✓ baseline_pred_vs_actual.png saved")

    # Save all justification strings for adviser/panel review
    with open(os.path.join(output_dir, "justifications_v5.txt"), "w") as fh:
        fh.write("JUSTIFICATION STRINGS — PIPELINE v5\n")
        fh.write("=" * 60 + "\n\n")
        fh.write("1. LEAKAGE VALIDATION\n" + LEAKAGE_JUSTIFICATION + "\n\n")
        fh.write("2. MANUAL MODEL CHOICE\n" + MANUAL_JUSTIFICATION + "\n\n")
        fh.write("3. ABLATION HYPERPARAMETERS\n" + ABLATION_HP_RATIONALE + "\n\n")
        fh.write("4. LSTM ABLATION DECISION\n" + ABLATION_LSTM_NOTE + "\n\n")
        fh.write("5. MIN DURATION THRESHOLD\n" + MIN_DURATION_THRESHOLD_NOTE + "\n\n")
        fh.write("6. IQR FILTER RESULT\n" + IQR_FILTER_NOTE + "\n\n")
        fh.write("7. FLAT RF LEAKAGE FIX (v5.3 / v6.1)\n" + FLAT_RF_LEAKAGE_NOTE + "\n\n")
        fh.write("8. DATASET SEPARATION (v6.1)\n" + DATASET_SEPARATION_NOTE + "\n\n")
        fh.write("9. LEAKAGE VALIDATION (v6.1)\n" + LEAKAGE_JUSTIFICATION + "\n\n")

    ablation_df.round(4).to_csv(
        os.path.join(output_dir, "full_summary_table.csv"), index=False
    )
    print(f"\n  ✓ All outputs saved to {output_dir}/")


# =============================================================================
# MAIN
# =============================================================================



# =============================================================================


# =============================================================================
# PHASE 7B — SEQUENTIAL MODEL PERMUTATION IMPORTANCE (v8.3)
# =============================================================================

# =============================================================================
# PHASE 7B — SEQUENTIAL MODEL PERMUTATION IMPORTANCE (v8.3, fixed)
# =============================================================================
# FIX: sklearn.inspection.permutation_importance requires .fit() method.
# Solution: implement permutation importance manually — shuffle each feature
# across all timesteps n_repeats times, compute MAE drop, average.
# This is identical in result to sklearn's implementation but works with
# any callable that has .predict().
# =============================================================================

def _manual_permutation_importance(predict_fn, X_flat, y_true,
                                    n_features_total, n_repeats, rng):
    """
    Manual permutation importance for any predict function.
    X_flat : (n_samples, n_cols) where n_cols = seq_len * n_feat
    y_true : (n_samples,) inverse-scaled targets in minutes
    Returns importances_mean : (n_cols,) — one value per column of X_flat
    """
    baseline_mae = np.mean(np.abs(predict_fn(X_flat) - y_true))
    importances  = np.zeros((n_repeats, n_features_total))

    for r in range(n_repeats):
        for col in range(n_features_total):
            X_perm        = X_flat.copy()
            perm_idx      = rng.permutation(len(X_flat))
            X_perm[:, col] = X_flat[perm_idx, col]
            perm_mae      = np.mean(np.abs(predict_fn(X_perm) - y_true))
            importances[r, col] = perm_mae - baseline_mae  # δMAE

    return importances.mean(axis=0), importances.std(axis=0)


class GRUWrapper:
    """Wraps a trained GRU model with a predict() interface."""
    def __init__(self, model, y_scaler, device, seq_len, n_feat):
        self.model    = model
        self.y_scaler = y_scaler
        self.device   = device
        self.seq_len  = seq_len
        self.n_feat   = n_feat

    def predict(self, X_flat):
        X_3d = X_flat.reshape(len(X_flat), self.seq_len, self.n_feat)
        self.model.eval()
        with torch.no_grad():
            X_t = torch.tensor(X_3d, dtype=torch.float32).to(self.device)
            pn  = self.model(X_t).cpu().numpy()
        return self.y_scaler.inverse_transform(pn.reshape(-1, 1)).ravel()


class LSTMWrapper:
    """Wraps a trained LSTM model with a predict() interface."""
    def __init__(self, model, y_scaler, device, seq_len, n_feat):
        self.model    = model
        self.y_scaler = y_scaler
        self.device   = device
        self.seq_len  = seq_len
        self.n_feat   = n_feat

    def predict(self, X_flat):
        X_3d = X_flat.reshape(len(X_flat), self.seq_len, self.n_feat)
        self.model.eval()
        with torch.no_grad():
            X_t = torch.tensor(X_3d, dtype=torch.float32).to(self.device)
            pn  = self.model(X_t).cpu().numpy()
        return self.y_scaler.inverse_transform(pn.reshape(-1, 1)).ravel()


class SeqRFWrapper:
    """Wraps trained Seq-RF with a predict() that handles flat input."""
    def __init__(self, model, y_scaler):
        self.model    = model
        self.y_scaler = y_scaler

    def predict(self, X_flat):
        pn = self.model.predict(X_flat).reshape(-1, 1)
        return self.y_scaler.inverse_transform(pn).ravel()


def phase7b_sequential_permutation_importance(
    gru_model, lstm_model, rf_model,
    X_te, y_te, y_sc,
    output_dir,
    n_repeats=5,   # v9 OPT: was 10; 5 repeats gives stable δMAE for thesis
    random_state=42,
):
    """
    v9 SEQUENTIAL MODEL PERMUTATION IMPORTANCE (manual implementation)

    Computes permutation importance for GRU, LSTM, and Seq-RF on the same
    AUTO sequence test set used in Phase 5 (RQ1). Each of the 12 features is
    shuffled across all 5 timesteps simultaneously for n_repeats=5 rounds (v9 OPT: was 10).
    Mean δMAE (MAE increase after shuffling) is recorded per feature per model.

    Manual implementation avoids sklearn's .fit() requirement on estimators.

    Cross-model comparison:
    ─────────────────────────────────────────────────────────────────────
    If GRU, LSTM, Seq-RF, and boosted models (Phase 5B permutation_importance
    .csv) agree on feature importance rankings, the finding is model-agnostic:
    operational features dominate regardless of model family.

    Methodology note (for manuscript / defence):
    ─────────────────────────────────────────────────────────────────────
    Shuffling a feature across all 5 timesteps simultaneously disrupts both
    the magnitude and the temporal pattern of that feature. This may slightly
    overestimate importance vs single-timestep shuffling, but provides a
    consistent cross-model comparison when applied uniformly to all models.

    Inputs (from Phase 5 exec cell E3):
      gru_model, lstm_model, rf_model : trained models
      X_te  : test sequences, shape (n_te, seq_len, n_feat)
      y_te  : scaled test targets, shape (n_te,)
      y_sc  : fitted y StandardScaler

    Output:
      sequential_permutation_importance.csv
    """
    log_phase("Phase 7B — Sequential Permutation Importance (v8.3)",
              context="MAIN PIPELINE")

    device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    feat_cols = get_feature_cols()
    n_feat    = len(feat_cols)          # 12 (v9 feature set)
    seq_len   = CFG["sequence_length"]  # 5
    n_cols    = seq_len * n_feat        # 85 — columns in flattened input
    rng       = np.random.RandomState(random_state)

    # Inverse-scale test targets (minutes)
    y_te_inv = y_sc.inverse_transform(y_te.reshape(-1, 1)).ravel()

    # Flatten X_te to 2D: (n_te, seq_len * n_feat)
    X_flat = X_te.reshape(len(X_te), n_cols).astype(np.float32)
    print(f"  Test set: {len(y_te_inv):,} sequences  "
          f"| Features: {n_feat}  | Timesteps: {seq_len}")
    print(f"  Flattened input shape: {X_flat.shape}  "
          f"| n_repeats: {n_repeats}")
    print(f"  Implementation: manual (no sklearn .fit() required)\n")

    # ── GRU ───────────────────────────────────────────────────────────────
    print("  [1/3] GRU permutation importance...")
    gru_wrap  = GRUWrapper(gru_model, y_sc, device, seq_len, n_feat)
    gru_col_mean, _ = _manual_permutation_importance(
        gru_wrap.predict, X_flat, y_te_inv, n_cols, n_repeats, rng
    )
    # Aggregate: reshape (n_cols,) → (seq_len, n_feat) → mean across timesteps
    gru_imp = gru_col_mean.reshape(seq_len, n_feat).mean(axis=0)
    gru_imp = np.clip(gru_imp, 0, None)
    print(f"  GRU done. Top: {feat_cols[np.argmax(gru_imp)]} "
          f"(δMAE={gru_imp.max():.4f})")

    # ── LSTM ──────────────────────────────────────────────────────────────
    print("\n  [2/3] LSTM permutation importance...")
    lstm_wrap = LSTMWrapper(lstm_model, y_sc, device, seq_len, n_feat)
    lstm_col_mean, _ = _manual_permutation_importance(
        lstm_wrap.predict, X_flat, y_te_inv, n_cols, n_repeats, rng
    )
    lstm_imp = lstm_col_mean.reshape(seq_len, n_feat).mean(axis=0)
    lstm_imp = np.clip(lstm_imp, 0, None)
    print(f"  LSTM done. Top: {feat_cols[np.argmax(lstm_imp)]} "
          f"(δMAE={lstm_imp.max():.4f})")

    # ── Seq-RF ────────────────────────────────────────────────────────────
    print("\n  [3/3] Seq-RF permutation importance...")
    rf_wrap  = SeqRFWrapper(rf_model, y_sc)
    rf_col_mean, _ = _manual_permutation_importance(
        rf_wrap.predict, X_flat, y_te_inv, n_cols, n_repeats, rng
    )
    rf_imp = rf_col_mean.reshape(seq_len, n_feat).mean(axis=0)
    rf_imp = np.clip(rf_imp, 0, None)
    print(f"  Seq-RF done. Top: {feat_cols[np.argmax(rf_imp)]} "
          f"(δMAE={rf_imp.max():.4f})")

    # ── Load boosted permutation importance for cross-model table ─────────
    boosted_imp = np.zeros(n_feat)
    boosted_path = os.path.join(output_dir, "permutation_importance.csv")
    if os.path.exists(boosted_path):
        bdf = pd.read_csv(boosted_path)
        for i, feat in enumerate(feat_cols):
            row = bdf[bdf["feature"] == feat]
            if not row.empty:
                col_name = "mean_importance" if "mean_importance" in bdf.columns else bdf.columns[1]
                boosted_imp[i] = row[col_name].values[0]
        print(f"\n  Boosted importance loaded from permutation_importance.csv")
    else:
        print(f"\n  WARNING: permutation_importance.csv not found — "
              "boosted column will be zeros. Run Phase 5B first.")

    # ── Build results DataFrame ───────────────────────────────────────────
    seq_mean = (gru_imp + lstm_imp + rf_imp) / 3.0
    results_df = pd.DataFrame({
        "feature":             feat_cols,
        "gru_delta_mae":       gru_imp.round(4),
        "lstm_delta_mae":      lstm_imp.round(4),
        "seqrf_delta_mae":     rf_imp.round(4),
        "boosted_delta_mae":   boosted_imp.round(4),
        "seq_mean_delta_mae":  seq_mean.round(4),
    })
    results_df["rank"] = results_df["seq_mean_delta_mae"].rank(
        ascending=False).astype(int)
    results_df = results_df.sort_values("seq_mean_delta_mae", ascending=False)

    out_path = os.path.join(output_dir, "sequential_permutation_importance.csv")
    results_df.to_csv(out_path, index=False)
    print(f"\n  ✓ sequential_permutation_importance.csv → {out_path}")

    # ── Print summary ─────────────────────────────────────────────────────
    print(f"\n  {'Feature':<28} {'GRU':>8} {'LSTM':>8} {'Seq-RF':>8} "
          f"{'Boosted':>8} {'Mean':>8}")
    print(f"  {'-'*68}")
    for _, row in results_df.iterrows():
        print(f"  {row['feature']:<28} "
              f"{row['gru_delta_mae']:>8.4f} "
              f"{row['lstm_delta_mae']:>8.4f} "
              f"{row['seqrf_delta_mae']:>8.4f} "
              f"{row['boosted_delta_mae']:>8.4f} "
              f"{row['seq_mean_delta_mae']:>8.4f}")

    print(f"\n  INTERPRETATION:")
    print(f"  δMAE > 0  → feature IS important (shuffling hurts accuracy)")
    print(f"  δMAE ≈ 0  → feature contributes little")
    print(f"  Clipped at 0 (negative = noise)")
    print(f"  Aggregated across {seq_len} timesteps (mean per feature).")
    return results_df



## Cell 9 — V6 Phase Definitions (threshold comparison, 5M, 5F, 5R)

In [13]:
# V6 NEW PHASES (moved before main() so they are defined when main() runs)
# =============================================================================

def phase_threshold_comparison(output_dir):
    """
    Adviser req item 2: Show the difference between 1-min and 2-min threshold.
    Runs phase1 at both thresholds, reports dataset stats side by side,
    then trains a flat RF at each threshold to show impact on model performance.
    No RNN training here — this is a dataset characterisation comparison only.
    """
    log_phase("Threshold Comparison: 1 min vs 2 min", context="[EXPERIMENT]")

    rows = []
    models_per_threshold = {}

    for threshold in [1.0, 2.0]:
        print(f"\n  ── Threshold = {threshold} min ──")
        _ctx = f"THRESHOLD={threshold}min"
        df_t = phase1_load_and_clean(min_duration_min=threshold, context=_ctx, skip_audit=True)
        df_t = phase2_context_labels(df_t, ctx=_ctx)
        df_t = phase3_feature_engineering(df_t, ctx=_ctx)

        auto_t   = df_t[df_t["operation_mode"] == "AUTO"]
        manual_t = df_t[df_t["operation_mode"] == "MANUAL"]

        def get_season_counts(sub):
            return sub["season"].value_counts().to_dict() if "season" in sub.columns else {}

        row = {
            "threshold_min":       threshold,
            "total_flights":       len(df_t),
            "auto_flights":        len(auto_t),
            "manual_flights":      len(manual_t),
            "auto_dry":            get_season_counts(auto_t).get("Dry", 0),
            "auto_hot":            get_season_counts(auto_t).get("Hot", 0),
            "auto_wet":            get_season_counts(auto_t).get("Wet", 0),
            "manual_dry":          get_season_counts(manual_t).get("Dry", 0),
            "manual_hot":          get_season_counts(manual_t).get("Hot", 0),
            "manual_wet":          get_season_counts(manual_t).get("Wet", 0),
            "target_mean_min":     round(df_t["flight_duration_min"].mean(), 3),
            "target_std_min":      round(df_t["flight_duration_min"].std(), 3),
            "target_median_min":   round(df_t["flight_duration_min"].median(), 3),
            "target_min":          round(df_t["flight_duration_min"].min(), 3),
            "target_max":          round(df_t["flight_duration_min"].max(), 3),
        }

        # Train a quick flat RF (clean, no ending_battery) on AUTO subset
        # to show the impact of threshold on model performance
        feat_cols  = get_flat_rf_feature_cols()
        target_col = "flight_duration_min"
        df_auto_t  = auto_t[feat_cols + [target_col]].dropna().reset_index(drop=True)
        n          = len(df_auto_t)
        n_tr       = int(n * CFG["train_frac"])
        n_va       = int(n * CFG["val_frac"])
        tr_idx     = np.arange(n_tr)

        if n_tr < CFG["min_rows_rf"] or (n - n_tr - n_va) < 3:
            print(f"    Insufficient data for RF at threshold={threshold} — skip")
            row["flat_rf_MAE"] = np.nan
            row["flat_rf_RMSE"] = np.nan
            row["flat_rf_R2"] = np.nan
            rows.append(row)
            continue

        X, y, _, y_sc = normalize_rows(df_auto_t, feat_cols, target_col, tr_idx)
        Xtr, ytr = X[:n_tr], y[:n_tr]
        Xva, yva = X[n_tr:n_tr+n_va], y[n_tr:n_tr+n_va]
        Xte, yte = X[n_tr+n_va:], y[n_tr+n_va:]

        m = flat_rf_grid_search(Xtr, ytr, Xva, yva, Xte, yte, y_sc,
                                label=f"flat RF AUTO threshold={threshold}min")
        row["flat_rf_MAE"]  = round(m["MAE"],  4)
        row["flat_rf_RMSE"] = round(m["RMSE"], 4)
        row["flat_rf_R2"]   = round(m["R2"],   4)
        row["flat_rf_MAPE"] = round(m["MAPE"], 4)
        row["flat_rf_n_test"] = n - n_tr - n_va
        rows.append(row)

        print(f"    Total={len(df_t):,}  AUTO={len(auto_t):,}  MANUAL={len(manual_t):,}")
        print(f"    flat RF AUTO — MAE={m['MAE']:.4f} min  R²={m['R2']:.4f}")

    comp_df = pd.DataFrame(rows)
    comp_df.to_csv(os.path.join(output_dir, "threshold_comparison.csv"), index=False)
    print("\n  ✓ threshold_comparison.csv saved")

    # Print side-by-side table
    print("\n  THRESHOLD COMPARISON SUMMARY:")
    print(f"  {'Metric':<35} {'1-min threshold':>18} {'2-min threshold':>18} {'Difference':>12}")
    print("  " + "-" * 85)
    if len(comp_df) == 2:
        r1 = comp_df[comp_df["threshold_min"] == 1.0].iloc[0]
        r2 = comp_df[comp_df["threshold_min"] == 2.0].iloc[0]
        metrics = [
            ("Total flights retained", "total_flights"),
            ("AUTO flights", "auto_flights"),
            ("MANUAL flights", "manual_flights"),
            ("AUTO-Dry flights", "auto_dry"),
            ("AUTO-Hot flights", "auto_hot"),
            ("AUTO-Wet flights", "auto_wet"),
            ("MANUAL-Dry flights", "manual_dry"),
            ("MANUAL-Hot flights", "manual_hot"),
            ("MANUAL-Wet flights", "manual_wet"),
            ("Target mean (min)", "target_mean_min"),
            ("Target median (min)", "target_median_min"),
            ("Target std (min)", "target_std_min"),
            ("Flat RF AUTO MAE (min)", "flat_rf_MAE"),
            ("Flat RF AUTO R²", "flat_rf_R2"),
        ]
        for label, key in metrics:
            v1 = r1.get(key, "N/A")
            v2 = r2.get(key, "N/A")
            try:
                diff = f"{float(v1) - float(v2):+.4g}"
            except Exception:
                diff = "—"
            print(f"  {label:<35} {str(v1):>18} {str(v2):>18} {diff:>12}")


def phase5_manual(df, output_dir):
    """
    Adviser req item 3: MANUAL mode trained SEPARATELY.
    Trains flat RF on MANUAL-only data (no AUTO data involved).
    Produces manual_baseline_results.csv.
    MANUAL uses flat RF because median session length = ~2 flights,
    which is insufficient for N=5 sequence windows.
    """
    log_phase("Phase 5M — MANUAL Baseline", context="MAIN PIPELINE")

    feat_cols  = get_flat_rf_feature_cols()
    target_col = "flight_duration_min"

    df_manual = (
        df[df["operation_mode"] == "MANUAL"]
        .copy()
        .reset_index(drop=True)
    )
    df_manual = df_manual[feat_cols + [target_col, "season"]].dropna().reset_index(drop=True)
    n     = len(df_manual)
    n_tr  = int(n * CFG["train_frac"])
    n_va  = int(n * CFG["val_frac"])
    tr_idx = np.arange(n_tr)

    print(f"  MANUAL flights: {n}  (train={n_tr}, val={n_va}, test={n-n_tr-n_va})")
    print(f"  Model: flat RF (no sequence windows — MANUAL sessions too short for N=5)")
    print(f"  Note: {MANUAL_JUSTIFICATION[:120]}...")

    if n_tr < CFG["min_rows_rf"] or (n - n_tr - n_va) < 3:
        print("  Insufficient data — skip")
        return

    X, y, _, y_sc = normalize_rows(df_manual, feat_cols, target_col, tr_idx)
    Xtr, ytr = X[:n_tr], y[:n_tr]
    Xva, yva = X[n_tr:n_tr+n_va], y[n_tr:n_tr+n_va]
    Xte, yte = X[n_tr+n_va:], y[n_tr+n_va:]

    m_overall = flat_rf_grid_search(Xtr, ytr, Xva, yva, Xte, yte, y_sc,
                                     label="MANUAL flat RF (overall)")

    # Seasonal breakdown on test set
    test_df = df_manual.iloc[n_tr+n_va:].reset_index(drop=True)
    season_rows = []
    for season in ["Dry", "Hot", "Wet"]:
        idx_s = test_df[test_df["season"] == season].index.tolist()
        if len(idx_s) < 3:
            continue
        y_true_s = y_sc.inverse_transform(yte[idx_s].reshape(-1, 1)).ravel()
        # Predict using the overall trained RF
        from sklearn.ensemble import RandomForestRegressor
        # Re-fit RF to get the model object (flat_rf_grid_search returns metrics only)
        grid = CFG["rf_grid"]
        best_rf, best_vl = None, float("inf")
        for n_est in grid["n_estimators"]:
            for max_d in grid["max_depth"]:
                rf_tmp = RandomForestRegressor(n_estimators=n_est, max_depth=max_d,
                                               random_state=SEED, n_jobs=-1)
                rf_tmp.fit(Xtr, ytr)
                vl = mean_squared_error(yva, rf_tmp.predict(Xva))
                if vl < best_vl:
                    best_vl, best_rf = vl, rf_tmp
        y_pred_s = y_sc.inverse_transform(
            best_rf.predict(X[np.array(idx_s) + n_tr + n_va]).reshape(-1, 1)
        ).ravel()
        m_s = _metrics(y_true_s, y_pred_s, f"MANUAL-{season}")
        season_rows.append({
            "mode": "MANUAL", "season": season, "model": "flat RF",
            "n_test": len(idx_s),
            "MAE": round(m_s["MAE"], 4), "RMSE": round(m_s["RMSE"], 4),
            "R2": round(m_s["R2"], 4),   "MAPE": round(m_s["MAPE"], 4),
        })
        print(f"  MANUAL-{season}: N={len(idx_s)}  MAE={m_s['MAE']:.4f}  R²={m_s['R2']:.4f}")

    overall_row = {
        "mode": "MANUAL", "season": "ALL", "model": "flat RF",
        "n_test": n - n_tr - n_va,
        "MAE": round(m_overall["MAE"], 4), "RMSE": round(m_overall["RMSE"], 4),
        "R2": round(m_overall["R2"], 4),   "MAPE": round(m_overall["MAPE"], 4),
    }
    result_df = pd.DataFrame([overall_row] + season_rows)
    result_df.to_csv(os.path.join(output_dir, "manual_baseline_results.csv"), index=False)
    print(f"\n  MANUAL overall: MAE={m_overall['MAE']:.4f} min  RMSE={m_overall['RMSE']:.4f}  R²={m_overall['R2']:.4f}")
    print("  ✓ manual_baseline_results.csv saved")


def phase5_flat_vs_seq_rf(df, output_dir):
    """
    Adviser req item 6: Explicit comparison of flat RF vs sequential RF.
    Both run on AUTO data. Key difference:
      - Flat RF: one flight at a time, 12 features (v9), NO temporal history
      - Seq RF:  5-flight window flattened to 5×18=90 features, HAS temporal history
    ending_battery excluded from flat RF (post-flight leakage).
    ending_battery INCLUDED in seq RF window (past flights only — not leakage).
    """
    log_phase("Phase 5F — Flat RF vs Seq-RF", context="MAIN PIPELINE")

    target_col = "flight_duration_min"
    rows = []

    # ── 1. FLAT RF (no sequence, no temporal context) ─────────────────────
    print("\n  [1/2] Flat RF — single flight rows, no sequence window")
    flat_feat = get_flat_rf_feature_cols()  # ending_battery excluded
    df_auto   = df[df["operation_mode"] == "AUTO"].copy().reset_index(drop=True)
    df_flat   = df_auto[flat_feat + [target_col]].dropna().reset_index(drop=True)
    n_f       = len(df_flat)
    n_tr_f    = int(n_f * CFG["train_frac"])
    n_va_f    = int(n_f * CFG["val_frac"])
    tr_f      = np.arange(n_tr_f)

    Xf, yf, _, yscf = normalize_rows(df_flat, flat_feat, target_col, tr_f)
    m_flat = flat_rf_grid_search(
        Xf[:n_tr_f], yf[:n_tr_f],
        Xf[n_tr_f:n_tr_f+n_va_f], yf[n_tr_f:n_tr_f+n_va_f],
        Xf[n_tr_f+n_va_f:],       yf[n_tr_f+n_va_f:],
        yscf, label="Flat RF (no temporal context)"
    )
    rows.append({
        "model_type":        "Flat RF",
        "description":       "No sequence window. One flight = one prediction. 17 features (ending_battery excluded).",
        "n_features":        len(flat_feat),
        "n_flights_input":   1,
        "ending_battery_in_input": "NO (excluded — post-flight observable)",
        "temporal_context":  "None — each flight predicted independently",
        "n_test_rows":       n_f - n_tr_f - n_va_f,
        "MAE":               round(m_flat["MAE"],  4),
        "RMSE":              round(m_flat["RMSE"], 4),
        "R2":                round(m_flat["R2"],   4),
        "MAPE":              round(m_flat["MAPE"], 4),
    })
    print(f"    n_features={len(flat_feat)}  n_test={n_f-n_tr_f-n_va_f}")
    print(f"    MAE={m_flat['MAE']:.4f} min  R²={m_flat['R2']:.4f}")

    # ── 2. SEQUENTIAL RF (sequence window flattened) ──────────────────────
    print("\n  [2/2] Sequential RF — 5-flight window flattened, temporal context present")
    seq_feat = get_feature_cols()  # includes ending_battery (from PAST flights)
    df_seq   = (
        df_auto[seq_feat + [target_col, "session_id"]]
        .dropna()
        .reset_index(drop=True)
    )
    n_s    = len(df_seq)
    n_tr_s = int(n_s * CFG["train_frac"])
    tr_s   = np.arange(n_tr_s)

    Xs, ys, _, yscs = normalize_rows(df_seq, seq_feat, target_col, tr_s)
    X_seq, y_seq, _tgt = build_sequences_session_aware(
        Xs, ys, df_seq["session_id"].values, CFG["sequence_length"]
    )  # v7.3: _tgt unused in flat-vs-seq comparison
    (Xtr_s, ytr_s), (Xva_s, yva_s), (Xte_s, yte_s) = temporal_split_sequences(
        X_seq, y_seq, CFG["train_frac"], CFG["val_frac"]
    )
    _, rf_seq_m = grid_search_rf(Xtr_s, ytr_s, Xva_s, yva_s, Xte_s, yte_s, yscs)

    rows.append({
        "model_type":        "Sequential RF",
        "description":       f"N={CFG['sequence_length']}-flight window flattened. {CFG['sequence_length']}×{len(seq_feat)}={CFG['sequence_length']*len(seq_feat)} features. ending_battery from PAST flights (not leakage).",
        "n_features":        CFG["sequence_length"] * len(seq_feat),
        "n_flights_input":   CFG["sequence_length"],
        "ending_battery_in_input": f"YES — from {CFG['sequence_length']} past completed flights (observable before next flight)",
        "temporal_context":  f"Last {CFG['sequence_length']} flights — captures operational patterns, battery trends",
        "n_test_rows":       len(yte_s),
        "MAE":               round(rf_seq_m["MAE"],  4),
        "RMSE":              round(rf_seq_m["RMSE"], 4),
        "R2":                round(rf_seq_m["R2"],   4),
        "MAPE":              round(rf_seq_m["MAPE"], 4),
    })
    print(f"    n_features={CFG['sequence_length']*len(seq_feat)}  n_test={len(yte_s)}")
    print(f"    MAE={rf_seq_m['MAE']:.4f} min  R²={rf_seq_m['R2']:.4f}")

    # ── Summary ───────────────────────────────────────────────────────────
    comp_df = pd.DataFrame(rows)
    comp_df.to_csv(os.path.join(output_dir, "flat_vs_seq_rf_comparison.csv"), index=False)

    print("\n  FLAT RF vs SEQUENTIAL RF COMPARISON:")
    print(f"  {'Metric':<10} {'Flat RF':>12} {'Seq RF':>12} {'Delta':>12}")
    print("  " + "-" * 48)
    for metric in ["MAE", "RMSE", "R2", "MAPE"]:
        v_flat = rows[0][metric]
        v_seq  = rows[1][metric]
        delta  = v_seq - v_flat
        print(f"  {metric:<10} {v_flat:>12.4f} {v_seq:>12.4f} {delta:>+12.4f}")
    print(f"\n  Key: Seq RF has {rows[1]['n_features'] - rows[0]['n_features']} more features")
    print(f"  (temporal history of past {CFG['sequence_length']} flights vs. no history)")
    print("  ✓ flat_vs_seq_rf_comparison.csv saved")


def phase5_repetitions(df, output_dir, n_reps=5):
    """
    Statistician req: Run GRU, LSTM, seq-RF each n_reps times with different
    random seeds to establish which model is consistently better.

    Exports:
      - repetition_results.csv: MAE/RMSE/R²/MAPE for every rep × model
      - ground_truth_and_predictions.csv: y_true + all model predictions
        (last rep only for RNN; all reps columns labelled _rep1..rep5)

    This directly answers: "train models in at least 5 repetitions to establish
    which model is better than the other two."
    """
    log_phase(f"Phase 5R — Repetition Stability Analysis ({n_reps} reps)", context="MAIN PIPELINE")
    print(f"  Purpose: Establish statistical consistency across random seeds.")
    print(f"  Models: GRU, LSTM, Sequential RF")
    print(f"  Reps: {n_reps} per model (different random seeds)")
    print(f"  Ground truth: exported alongside all predictions for statistician.")

    device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    feat_cols  = get_feature_cols()
    target_col = "flight_duration_min"

    df_auto = (
        df[df["operation_mode"] == "AUTO"]
        .copy()
        .reset_index(drop=True)
    )
    df_auto = df_auto[feat_cols + [target_col, "session_id"]].dropna().reset_index(drop=True)
    n_a   = len(df_auto)
    n_tr  = int(n_a * CFG["train_frac"])
    n_va  = int(n_a * CFG["val_frac"])
    tr_idx = np.arange(n_tr)

    X_norm, y_norm, x_sc, y_sc = normalize_rows(df_auto, feat_cols, target_col, tr_idx)
    X_seq, y_seq, _tgt = build_sequences_session_aware(
        X_norm, y_norm, df_auto["session_id"].values, CFG["sequence_length"]
    )  # v7.3: _tgt unused in repetition analysis
    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = temporal_split_sequences(
        X_seq, y_seq, CFG["train_frac"], CFG["val_frac"]
    )
    input_size = len(feat_cols)

    # Ground truth (same for all reps — test set doesn't change)
    y_true_orig = y_sc.inverse_transform(y_te.reshape(-1, 1)).ravel()

    rep_rows    = []
    pred_dict   = {"y_true": y_true_orig}
    SEEDS       = [42, 7, 13, 99, 2024][:n_reps]

    print(f"\n  Seeds used: {SEEDS}")
    print(f"  Test set size: {len(y_te)} sequences\n")

    for rep, seed in enumerate(SEEDS, start=1):
        print(f"  ── Rep {rep}/{n_reps}  seed={seed} ──────────────────────────")
        torch.manual_seed(seed)
        np.random.seed(seed)

        # ── GRU ───────────────────────────────────────────────────────────
        # verbose suppressed in repetitions to avoid flooding output
        # (5 reps × 12 configs × up to 100 epochs each)
        gru_rep, gru_m, _ = random_search_rnn(
            GRUModel, f"GRU rep{rep}",
            X_tr, y_tr, X_va, y_va, X_te, y_te,
            y_sc, input_size, device, CFG["hp_search_configs"],
            verbose_train=False,
        )
        gru_rep.eval()
        with torch.no_grad():
            gru_pred = y_sc.inverse_transform(
                gru_rep(torch.tensor(X_te, dtype=torch.float32).to(device))
                .cpu().numpy().reshape(-1, 1)
            ).ravel()
        pred_dict[f"GRU_rep{rep}"] = gru_pred
        rep_rows.append({
            "rep": rep, "seed": seed, "model": "GRU",
            "MAE": round(gru_m["MAE"],4), "RMSE": round(gru_m["RMSE"],4),
            "R2":  round(gru_m["R2"],4),  "MAPE": round(gru_m["MAPE"],4),
        })
        print(f"    GRU  MAE={gru_m['MAE']:.4f}  R²={gru_m['R2']:.4f}")

        # ── LSTM ──────────────────────────────────────────────────────────
        lstm_rep, lstm_m, _ = random_search_rnn(
            LSTMModel, f"LSTM rep{rep}",
            X_tr, y_tr, X_va, y_va, X_te, y_te,
            y_sc, input_size, device, CFG["hp_search_configs"],
            verbose_train=False,
        )
        lstm_rep.eval()
        with torch.no_grad():
            lstm_pred = y_sc.inverse_transform(
                lstm_rep(torch.tensor(X_te, dtype=torch.float32).to(device))
                .cpu().numpy().reshape(-1, 1)
            ).ravel()
        pred_dict[f"LSTM_rep{rep}"] = lstm_pred
        rep_rows.append({
            "rep": rep, "seed": seed, "model": "LSTM",
            "MAE": round(lstm_m["MAE"],4), "RMSE": round(lstm_m["RMSE"],4),
            "R2":  round(lstm_m["R2"],4),  "MAPE": round(lstm_m["MAPE"],4),
        })
        print(f"    LSTM MAE={lstm_m['MAE']:.4f}  R²={lstm_m['R2']:.4f}")

        # ── Seq RF ────────────────────────────────────────────────────────
        # RF is deterministic for same random_state — use seed as random_state
        n_, s_, f_ = X_te.shape
        Xtr_f = X_tr.reshape(len(X_tr), s_*f_)
        Xva_f = X_va.reshape(len(X_va), s_*f_)
        Xte_f = X_te.reshape(n_, s_*f_)

        from sklearn.ensemble import RandomForestRegressor
        best_rf_rep, best_vl = None, float("inf")
        for n_est in CFG["rf_grid"]["n_estimators"]:
            for max_d in CFG["rf_grid"]["max_depth"]:
                rf_tmp = RandomForestRegressor(n_estimators=n_est, max_depth=max_d,
                                               random_state=seed, n_jobs=-1)
                rf_tmp.fit(Xtr_f, y_tr)
                vl = mean_squared_error(y_va, rf_tmp.predict(Xva_f))
                if vl < best_vl:
                    best_vl, best_rf_rep = vl, rf_tmp

        rf_pred = y_sc.inverse_transform(
            best_rf_rep.predict(Xte_f).reshape(-1, 1)
        ).ravel()
        rf_true = y_sc.inverse_transform(y_te.reshape(-1, 1)).ravel()
        rf_m    = _metrics(rf_true, rf_pred, f"RF rep{rep}")
        pred_dict[f"RF_rep{rep}"] = rf_pred
        rep_rows.append({
            "rep": rep, "seed": seed, "model": "RF_seq",
            "MAE": round(rf_m["MAE"],4), "RMSE": round(rf_m["RMSE"],4),
            "R2":  round(rf_m["R2"],4),  "MAPE": round(rf_m["MAPE"],4),
        })
        print(f"    RF   MAE={rf_m['MAE']:.4f}  R²={rf_m['R2']:.4f}")

    # ── Save repetition results ───────────────────────────────────────────
    rep_df = pd.DataFrame(rep_rows)
    rep_df.to_csv(os.path.join(output_dir, "repetition_results.csv"), index=False)

    # ── Save ground truth + all predictions for statistician ─────────────
    gt_pred_df = pd.DataFrame(pred_dict)
    gt_pred_df.to_csv(os.path.join(output_dir, "ground_truth_and_predictions.csv"), index=False)

    # ── Summary statistics across reps ───────────────────────────────────
    print("\n  REPETITION SUMMARY (mean ± std across 5 runs):")
    print(f"  {'Model':<10} {'MAE mean':>10} {'MAE std':>10} {'R² mean':>10} {'R² std':>10}")
    print("  " + "-" * 52)
    for model in ["GRU", "LSTM", "RF_seq"]:
        sub = rep_df[rep_df["model"] == model]
        print(f"  {model:<10} {sub['MAE'].mean():>10.4f} {sub['MAE'].std():>10.4f} "
              f"{sub['R2'].mean():>10.4f} {sub['R2'].std():>10.4f}")

    print(f"\n  ✓ repetition_results.csv saved ({len(rep_df)} rows)")
    print(f"  ✓ ground_truth_and_predictions.csv saved")
    print(f"    Columns: y_true + GRU/LSTM/RF predictions for each of {n_reps} reps")
    print(f"    Row = one test flight. Ground truth = DJI flight controller log.")





## Cell 10 — Setup: Output Directory

In [14]:
# =============================================================================
# SETUP — Run this cell FIRST after running Cells 1–9
# =============================================================================
import os
output_dir = r"C:\Users\User\Documents\THESIS_HUHU\FINAL PIPELINE\outputs_v9"
os.makedirs(output_dir, exist_ok=True)
os.environ["OUTPUT_DIR"] = output_dir
print(f"Output directory ready: {output_dir}")
print("Now run Phase cells below in order.")

def main():
    output_dir = CFG["output_dir"]
    os.makedirs(output_dir, exist_ok=True)

    # Make output_dir accessible to phase3 dataset save
    os.environ["OUTPUT_DIR"] = output_dir

    print("█" * 60)
    print("  DRONE PIPELINE v8.3")
    print("  v6 1: Dual threshold comparison (1 min vs 2 min)")
    print("  v6 2: AUTO and MANUAL trained SEPARATELY")
    print("  v6 3: Flat RF vs Seq RF explicit comparison")
    print("  v6 4: 5-repetition stability analysis (statistician req)")
    print("  v6 5: Ground truth + predictions CSV for statistician")
    print("  v7.1: Full 17-feature GRU baseline control (Phase 6A)")
    print("  v7.2: Prediction comparison table — all 6 models (Phase 6C)")
    print("  v7.3: Session sort fix, end_datetime, traceable pred output")
    print("  v8.1: Split flat vs seq-aligned eval; flat_evaluation_results.csv")
    print("  v8.1.2: Phase 6D operational ablation (18 runs, supplementary RQ2)")
    print("  v8.3: Phase 7B sequential permutation importance (GRU, LSTM, Seq-RF)")
    print("█" * 60)

    # ── PHASE 1A: Dual threshold comparison (adviser req + statistician req) ─
    # Run the FULL pipeline at both thresholds and report results side by side.
    # This directly addresses: "show the difference between 1-min and 2-min"
    phase_threshold_comparison(output_dir)

    # ── PHASES 1–3: Data preparation at the v6 default threshold (1 min) ─────
    df = phase1_load_and_clean()
    df = phase2_context_labels(df)
    df = phase3_feature_engineering(df)       # saves final_dataset.csv

    # Phase 3b: EDA visualization
    phase3b_eda(df, output_dir)               # saves flight_duration_distribution.png

    # ── PHASE 5: Baseline — AUTO only (GRU, LSTM, seq-RF) ────────────────────
    # v6: AUTO and MANUAL are trained SEPARATELY (adviser req item 3).
    # phase5_baseline now trains only on AUTO; MANUAL has its own phase.
    # v7.3: return order = gru, lstm, rf, y_sc, x_sc, gru_m, lstm_m, X_seq, y_seq, df_auto, pred_df
    gru_model, lstm_model, rf_model, y_sc, x_sc, gru_m, lstm_m, X_seq, y_seq, df_auto, pred_df = phase5_baseline(df, output_dir)

    # ── PHASE 5M: MANUAL-only flat RF ────────────────────────────────────────
    phase5_manual(df, output_dir)

    # ── PHASE 5F: Flat RF vs Sequential RF comparison ────────────────────────
    # v6: Explicit side-by-side comparison (adviser req item 6).
    phase5_flat_vs_seq_rf(df, output_dir)

    # ── PHASE 5R: 5-repetition stability analysis (statistician req) ─────────
    # Runs GRU, LSTM, seq-RF each 5 times with different random seeds.
    # Exports ground truth + all 5 predictions per model for statistician.
    phase5_repetitions(df, output_dir, n_reps=5)

    # ── PHASE 6: Ablation experiments (RQ2) ──────────────────────────────────
    ablation_df = phase6_ablation(df, output_dir)

    # Phase 6b: LSTM sensitivity check
    lstm_sens_df = phase6b_lstm_sensitivity(df, output_dir)

    # Phase 7: analysis and visualization
    phase7_analyze(ablation_df, pred_df, lstm_sens_df, output_dir)

    print("\n" + "=" * 60)
    print("  PIPELINE v6.1.5 COMPLETE")
    print(f"  All outputs in ./{output_dir}/")
    print("  Expected outputs (17):")
    expected = [
        "final_dataset.csv",
        "flight_duration_distribution.png",
        "threshold_comparison.csv",           # NEW v6: 1-min vs 2-min results
        "manual_baseline_results.csv",        # NEW v6: MANUAL separate training
        "flat_vs_seq_rf_comparison.csv",      # NEW v6: flat RF vs seq RF
        "repetition_results.csv",             # NEW v6: 5-rep stability (statistician)
        "ground_truth_and_predictions.csv",   # NEW v6: for statistician
        "baseline_results.csv",
        "computational_efficiency.csv",
        "computational_efficiency.png",
        "baseline_pred_vs_actual.png",
        "mode_comparison_rf.csv",
        "hp_search_GRU.csv",
        "hp_search_LSTM.csv",
        "ablation_results.csv",
        "lstm_sensitivity_check.csv",
        "feature_importance.png",
        "delta_mae_heatmap.png",
        "mae_by_season.png",
        "full_summary_table.csv",
        "justifications_v5.txt",
    ]
    for f in expected:
        print(f"    {f}")
    print("=" * 60)




if __name__ == "__main__":
    main()













Output directory ready: C:\Users\User\Documents\THESIS_HUHU\FINAL PIPELINE\outputs_v9
Now run Phase cells below in order.
████████████████████████████████████████████████████████████
  DRONE PIPELINE v8.3
  v6 1: Dual threshold comparison (1 min vs 2 min)
  v6 2: AUTO and MANUAL trained SEPARATELY
  v6 3: Flat RF vs Seq RF explicit comparison
  v6 4: 5-repetition stability analysis (statistician req)
  v6 5: Ground truth + predictions CSV for statistician
  v7.1: Full 17-feature GRU baseline control (Phase 6A)
  v7.2: Prediction comparison table — all 6 models (Phase 6C)
  v7.3: Session sort fix, end_datetime, traceable pred output
  v8.1: Split flat vs seq-aligned eval; flat_evaluation_results.csv
  v8.1.2: Phase 6D operational ablation (18 runs, supplementary RQ2)
  v8.3: Phase 7B sequential permutation importance (GRU, LSTM, Seq-RF)
████████████████████████████████████████████████████████████

[[EXPERIMENT]] Threshold Comparison: 1 min vs 2 min

  ── Threshold = 1.0 min ──

[THRESHO

---
# Execution Cells

Run each cell below in order. All definition cells (0–10) must be run first.

**Context labels:**
- `[EXPERIMENT]` — comparison/diagnostic runs, not used in primary results
- `[MAIN PIPELINE]` — primary pipeline producing all reported results
- `[AUDIT]` — validation and diagnostic outputs


## [EXPERIMENT] Threshold Comparison — 1 min vs 2 min

In [15]:
# [EXPERIMENT] Threshold Comparison — 1 min vs 2 min
# Produces: threshold_comparison.csv
# Note: skip_audit=True inside — no duplicate filtering_audit.csv
phase_threshold_comparison(output_dir)



[[EXPERIMENT]] Threshold Comparison: 1 min vs 2 min

  ── Threshold = 1.0 min ──

[THRESHOLD=1.0min] Phase 1 — Load, Clean, Assign Sessions
  Raw combined : 8,675  (AUTO=7,051  MANUAL=1,624)
  [v9 unit fix] sprayed_area converted: mu → ha (÷15). Median 4.76 mu → 0.3173 ha
  Dropped 10 rows with null required fields
  Dropped 1,128 flights < 1.0 min  [threshold=1.0 min]
  IQR filter: bounds=[-3.54, 10.26] min  dropped 1 flights  [IQR FILTER RESULT: At the 1-min threshold, IQR bounds on fli...]
  ✓ Session audit: 0 mixed-mode sessions
  Sessions: 962 total  (AUTO=533  MANUAL=429)  gap=60 min
  ERA5 merge: 0/7536 unmatched (0.0%)

  ✓ Final dataset: 7,536 flights
  Modes  : {'AUTO': 6446, 'MANUAL': 1090}
  Range  : 2024-01-02 → 2024-11-25
  Target : flight_duration_min — mean=3.49  std=2.40  min=1.00  max=10.20

[THRESHOLD=1.0min] Phase 2 — Context Labels
  AUTO-Dry  : 2,921
  MANUAL-Dry  : 399
  AUTO-Hot  : 2,312
  MANUAL-Hot  : 355
  AUTO-Wet  : 1,213
  MANUAL-Wet  : 336

  MANUAL MODE

## [MAIN PIPELINE] Phase 1–3 — Load, Clean, Feature Engineering + Audits

In [16]:
# [MAIN PIPELINE] Load, clean, feature-engineer, EDA
# Audit outputs produced here:
#   filtering_audit.csv        (Issue 3: filter step accounting)
#   weather_alignment_audit.csv (Issue 5: ERA5 merge quality)
#   baseline_mean_model.csv     (Issue 8: naive baseline)
#   proxy_feature_audit.csv     (Issue 2: operational feature dominance)
df = phase1_load_and_clean(context="MAIN PIPELINE")
df = phase2_context_labels(df, ctx="MAIN PIPELINE")
df = phase3_feature_engineering(df, ctx="MAIN PIPELINE")
phase3b_eda(df, output_dir)
print(f"\nDataset ready: {len(df):,} flights")
print(df['operation_mode'].value_counts().to_dict())
print(df['season'].value_counts().to_dict())

# Issue 8: Mean predictor baseline — must be beaten by all learned models
mean_baseline = compute_mean_baseline(df, output_dir, mode="AUTO")

# Issue 2: Proxy feature dominance audit
proxy_audit = audit_proxy_features(df, output_dir)



[MAIN PIPELINE] Phase 1 — Load, Clean, Assign Sessions
  Raw combined : 8,675  (AUTO=7,051  MANUAL=1,624)
  [v9 unit fix] sprayed_area converted: mu → ha (÷15). Median 4.76 mu → 0.3173 ha
  Dropped 10 rows with null required fields
  Dropped 1,128 flights < 1.0 min  [threshold=1.0 min]
  IQR filter: bounds=[-3.54, 10.26] min  dropped 1 flights  [IQR FILTER RESULT: At the 1-min threshold, IQR bounds on fli...]
  ✓ Session audit: 0 mixed-mode sessions
  Sessions: 962 total  (AUTO=533  MANUAL=429)  gap=60 min
  ERA5 merge: 0/7536 unmatched (0.0%)

  [FILTERING AUDIT]
  Step                                   Removed        %    Remaining
  --------------------------------------------------------------------
  null_required_fields                        10     0.12%        8,665
  duration_threshold (<1.0 min)            1,128    13.02%        7,537
  iqr_outlier_filter                           1     0.01%        7,536
  TOTAL                                    1,139    13.13%        7,53

## [MAIN PIPELINE] Phase 5 — AUTO Baseline: GRU, LSTM, Seq-RF (RQ1)

Returns: `gru_model`, `lstm_model`, `rf_model`, `y_sc`, `x_sc`, `gru_m`, `lstm_m`,
`X_seq`, `y_seq`, `df_auto`, `pred_df`

`x_sc`, `gru_m`, `lstm_m` are new in v6.1.4 — needed by the export cell.


In [17]:
# [MAIN PIPELINE] Phase 5 — AUTO Baseline: GRU, LSTM, Seq-RF (RQ1)
# Produces: baseline_results.csv, computational_efficiency.csv/.png,
#           baseline_pred_vs_actual.png, mode_comparison_rf.csv,
#           hp_search_GRU.csv, hp_search_LSTM.csv
# Issue 1: session leakage assertion runs inside phase5_baseline
# Issue 7: train/val/test stability check runs inside phase5_baseline (Seq-RF)
(
    gru_model, lstm_model, rf_model,
    y_sc, x_sc, gru_m, lstm_m,
    X_seq, y_seq, df_auto, pred_df,
) = phase5_baseline(df, output_dir)




[MAIN PIPELINE] Phase 5 — Baseline Experiment (RQ1: architecture comparison)
  Device: cuda  |  Features: 12
  ✓ Leakage check passed (v6.1):
    - target 'flight_duration_min' NOT in feature_cols
    - ending_battery NOT in feature_cols
    - all 12 features present in DataFrame
    - rolling_duration_3 uses shift(1): past-flight values only
    - LEAKAGE VALIDATION (v6.1): All model input features are sourced exclusively from information observa...
  AUTO sequences total: 3,848
  Train=2,693  Val=577  Test=578

  [LEAKAGE AUDIT] AUTO sequences (phase5_baseline)
    Total sequences  : 3,848
    Train sequences  : 2,693  (idx 0 … 2692)
    Val sequences    : 577  (idx 2693 … 3269)
    Test sequences   : 578  (idx 3270 … 3847)
    ✓ Train / Val / Test sequence index sets: DISJOINT
    ✓ No sequence target appears as input in a different split
    ✓ Session-aware builder guarantees no cross-session windows
    Note: Long sessions may contribute sequences to multiple splits —
          t

## [MAIN PIPELINE] Phase 5B — Boosted Models: XGBoost, LightGBM, CatBoost

Returns: `boosted_results`, `xgb_model`, `lgbm_model`, `cb_best_model`,
`boosted_test_preds`

`boosted_test_preds` (v7.2): dict with `y_true`, `xgb_pred`, `lgbm_pred`,
`catboost_pred`, `n_test` on the boosted-model flat test set.

**Alignment note:** `boosted_test_preds` uses flat individual-flight rows
(last 15% of AUTO flights). `pred_df` from Phase 5 uses sequence-based
test targets. Different n — not comparable row-by-row. See Phase 6C.


In [18]:
# [MAIN PIPELINE] Phase 5B — Boosted Models (XGBoost, LightGBM, CatBoost)
# Produces:
#   boosted_models_results.csv      — MAE/RMSE/R2/MAPE (same format as baseline_results.csv)
#   boosted_feature_importance.png  — normalised avg importance across 3 models
#   permutation_importance.csv      — model-agnostic feature importance (Issue 6)
# CAUTION: internal importance (plot) and permutation importance (CSV) use
#          different scales — NOT directly comparable.
boosted_results, xgb_model, lgbm_model, cb_best_model, boosted_test_preds = \
    phase5_boosted_models(df, output_dir)
# boosted_test_preds: predictions on boosted-model flat test set (flat 15% flights).
# Used for stand-alone boosted evaluation.
# For aligned comparison with seq models, phase6c re-evaluates on df_test_meta.





[MAIN PIPELINE] Phase 5B — Boosted Models (XGBoost, LightGBM, CatBoost)
  Features: 12  (flat — no sequencing, same as flat RF)
  ✓ Leakage check passed (v6.1):
    - target 'flight_duration_min' NOT in feature_cols
    - ending_battery NOT in feature_cols
    - all 12 features present in DataFrame
    - rolling_duration_3 uses shift(1): past-flight values only
    - LEAKAGE VALIDATION (v6.1): All model input features are sourced exclusively from information observa...
  AUTO rows after dropna: 6,446
  Train=4,512  Val=966  Test=968

  [XGBoost] Grid search — 6 configs
    n_estimators=200  max_depth=4  learning_rate=0.05  subsample=0.8  → val_MSE=0.65465 ← best
    n_estimators=300  max_depth=6  learning_rate=0.05  subsample=0.8  → val_MSE=0.63919 ← best
    n_estimators=500  max_depth=4  learning_rate=0.01  subsample=0.9  → val_MSE=0.67434
    n_estimators=300  max_depth=8  learning_rate=0.1  subsample=0.7  → val_MSE=0.69865
    n_estimators=200  max_depth=6  learning_rate=0.1  subs

## [MAIN PIPELINE] Phase 5B-Flat — Flat Evaluation (v8.1)

Evaluates RF, XGBoost, LightGBM, CatBoost on the full flat individual-flight AUTO test set.
**This is NOT a cross-model comparison** — use Phase 6C for that.
Output: `flat_evaluation_results.csv`


In [19]:
# [MAIN PIPELINE] Phase 5B-Flat — Flat Individual-Flight Evaluation (v8.1)
# Evaluates boosted models on the FULL FLAT test set (last 15% of AUTO flights).
# This is the valid benchmark for flat models in isolation.
# DO NOT compare these numbers directly with GRU/LSTM/Seq-RF (different test set).
# For cross-model comparison, use Phase 6C (sequence-aligned, same rows).
#
# Requires: xgb_model, lgbm_model, cb_best_model from Phase 5B (cell E4)
# Produces: flat_evaluation_results.csv
flat_eval_df = phase5_boosted_flat_eval(
    df            = df,
    output_dir    = output_dir,
    xgb_model     = xgb_model,
    lgbm_model    = lgbm_model,
    cb_best_model = cb_best_model,
)



[MAIN PIPELINE] Phase 5B-Flat — Flat Evaluation (RF, XGB, LGBM, CatBoost)
  Flat individual-flight test set (last 15% of AUTO flights).
  This is NOT comparable row-by-row with seq model results.
  Flat test set: 968 individual flights
    Flat XGBoost                                  MAE=0.5797  RMSE=0.7709  R²=0.8992  MAPE=19.51%
    Flat LightGBM                                 MAE=0.5903  RMSE=0.7864  R²=0.8951  MAPE=19.57%
    Flat CatBoost                                 MAE=0.5508  RMSE=0.7379  R²=0.9077  MAPE=18.80%
  ✓ flat_evaluation_results.csv → C:\Users\User\Documents\THESIS_HUHU\FINAL PIPELINE\outputs_v9\flat_evaluation_results.csv
    XGBoost      MAE=0.5797  RMSE=0.7709  R²=0.8992
    LightGBM     MAE=0.5903  RMSE=0.7864  R²=0.8951
    CatBoost     MAE=0.5508  RMSE=0.7379  R²=0.9077


## [MAIN PIPELINE] Phase 5E — Export All Trained Models

Run **once** after Phase 5B. Does **not** retrain anything.
Self-contained — no external script import required.

**Outputs** (`outputs_v9/saved_models/`) — v9: 12-feature models:
- `gru_model.pth`, `lstm_model.pth`
- `rf_model.pkl`, `xgb_model.pkl`, `lgbm_model.pkl`, `catboost_model.pkl`
- `x_scaler.pkl`, `y_scaler.pkl`, `feature_cols.json`


In [20]:
# [MAIN PIPELINE] Phase 5E — Export Trained Models
# Run ONCE after Phase 5B. No retraining occurs.
# Self-contained: no external script import needed.
#
# Depends on variables set by preceding cells:
#   gru_model, lstm_model, rf_model  ← phase5_baseline()        (cell E3)
#   y_sc, x_sc, gru_m, lstm_m       ← phase5_baseline()        (cell E3)
#   xgb_model, lgbm_model            ← phase5_boosted_models()  (cell E4)
#   cb_best_model                    ← phase5_boosted_models()  (cell E4)

import os, json as _json, joblib

_SAVE_DIR = os.path.join(output_dir, "saved_models")
os.makedirs(_SAVE_DIR, exist_ok=True)

# ── Step 1: Validate before writing anything ───────────────────────────────────
_feature_cols_for_export = get_feature_cols()   # 12-feature list (v9), exact training order

validate_pipeline_artifacts(
    df            = df,
    feature_cols  = _feature_cols_for_export,
    x_sc          = x_sc,
    y_sc          = y_sc,
    gru_model     = gru_model,
    lstm_model    = lstm_model,
    rf_model      = rf_model,
    xgb_model     = xgb_model,
    lgbm_model    = lgbm_model,
    cb_best_model = cb_best_model,
    gru_m         = gru_m,
    lstm_m        = lstm_m,
)

# ── Step 2: Export all artifacts ───────────────────────────────────────────────
print(f"\nExporting to {_SAVE_DIR}/...")

# GRU — state_dict + arch config + best_hp
_gru_hp = gru_m["best_hp"]
torch.save(
    {
        "state_dict": gru_model.state_dict(),
        "config": {
            "input_size":  len(_feature_cols_for_export),
            "hidden_size": _gru_hp["hidden_size"],
            "num_layers":  _gru_hp["num_layers"],
            "dropout":     _gru_hp["dropout"],
        },
        "best_hp": _gru_hp,
    },
    os.path.join(_SAVE_DIR, "gru_model.pth"),
)
print(f"  ✓ gru_model.pth  "
      f"(h={_gru_hp['hidden_size']} l={_gru_hp['num_layers']} d={_gru_hp['dropout']})")

# LSTM — state_dict + arch config + best_hp
_lstm_hp = lstm_m["best_hp"]
torch.save(
    {
        "state_dict": lstm_model.state_dict(),
        "config": {
            "input_size":  len(_feature_cols_for_export),
            "hidden_size": _lstm_hp["hidden_size"],
            "num_layers":  _lstm_hp["num_layers"],
            "dropout":     _lstm_hp["dropout"],
        },
        "best_hp": _lstm_hp,
    },
    os.path.join(_SAVE_DIR, "lstm_model.pth"),
)
print(f"  ✓ lstm_model.pth "
      f"(h={_lstm_hp['hidden_size']} l={_lstm_hp['num_layers']} d={_lstm_hp['dropout']})")

# Seq-RF — sklearn pickle
joblib.dump(rf_model, os.path.join(_SAVE_DIR, "rf_model.pkl"))
print(f"  ✓ rf_model.pkl   (n_estimators={rf_model.n_estimators}, "
      f"n_features_in_={rf_model.n_features_in_} = 5×17)")

# Boosted models — sklearn / native pickles
# NOTE: boosted models trained on RAW X and RAW y — no scaler applied
joblib.dump(xgb_model,    os.path.join(_SAVE_DIR, "xgb_model.pkl"))
print(f"  ✓ xgb_model.pkl")
joblib.dump(lgbm_model,   os.path.join(_SAVE_DIR, "lgbm_model.pkl"))
print(f"  ✓ lgbm_model.pkl")
joblib.dump(cb_best_model, os.path.join(_SAVE_DIR, "catboost_model.pkl"))
print(f"  ✓ catboost_model.pkl")

# Scalers — fitted on AUTO training rows only
# x_sc: StandardScaler on 17 features (seq models only — NOT for boosted)
# y_sc: StandardScaler on flight_duration_min (seq models only)
joblib.dump(x_sc, os.path.join(_SAVE_DIR, "x_scaler.pkl"))
print(f"  ✓ x_scaler.pkl   (fitted on {x_sc.n_features_in_} features, train rows only)")
joblib.dump(y_sc, os.path.join(_SAVE_DIR, "y_scaler.pkl"))
print(f"  ✓ y_scaler.pkl   (fitted on flight_duration_min, train rows only)")

# Feature column list — exact order used in training
with open(os.path.join(_SAVE_DIR, "feature_cols.json"), "w") as _f:
    _json.dump(_feature_cols_for_export, _f, indent=2)
print(f"  ✓ feature_cols.json ({len(_feature_cols_for_export)} features, exact training order)")

print(f"\n  ✓ Export complete → {_SAVE_DIR}/")
print(f"  Reload artifacts using: joblib.load() for .pkl, torch.load() for .pth")
print(f"  See export_reload_predict2.py for load_artifacts() and predict_duration()")



[EXPORT] Pipeline Artifact Validation
  ✓ Feature count: 12 == 12
  ✓ No leakage features in feature_cols (['ending_battery'] excluded)
  ✓ Feature order matches get_feature_cols() exactly
  ✓ AUTO data: 6,446 rows, 0 NaNs in feature matrix
  ✓ x_sc: StandardScaler, n_features_in_=12
  ✓ y_sc: StandardScaler, n_features_in_=1 (flight_duration_min)
  ✓ GRU  best_hp: {'hidden_size': 64, 'num_layers': 1, 'dropout': 0.3, 'learning_rate': 0.001, 'batch_size': 64}
  ✓ LSTM best_hp: {'hidden_size': 256, 'num_layers': 2, 'dropout': 0.3, 'learning_rate': 0.001, 'batch_size': 64}
  ✓ Seq model types: GRUModel, LSTMModel, RandomForestRegressor
  ✓ Boosted model types: XGBRegressor, LGBMRegressor, CatBoostRegressor
  ✓ Seq-RF n_features_in_=60 == 5×12

  ✓ ALL 10 VALIDATION CHECKS PASSED — safe to call export_all_models()

Exporting to C:\Users\User\Documents\THESIS_HUHU\FINAL PIPELINE\outputs_v9\saved_models/...
  ✓ gru_model.pth  (h=64 l=1 d=0.3)
  ✓ lstm_model.pth (h=256 l=2 d=0.3)
  ✓ rf_mode

## [MAIN PIPELINE] Phase 5M — MANUAL Baseline (flat RF, separate training)

In [21]:
# Adviser requirement: AUTO and MANUAL trained SEPARATELY.
# MANUAL uses flat RF — median session = 2 flights, too short for N=5 windows.
# Produces: manual_baseline_results.csv
phase5_manual(df, output_dir)


[MAIN PIPELINE] Phase 5M — MANUAL Baseline
  MANUAL flights: 1090  (train=763, val=163, test=164)
  Model: flat RF (no sequence windows — MANUAL sessions too short for N=5)
  Note: MANUAL MODEL CHOICE: Sequential models (GRU, LSTM) require temporal continuity across a window of N consecutive flights ...
    MANUAL flat RF (overall)                      MAE=0.3643  RMSE=0.5728  R²=0.6880  MAPE=17.46%
    MANUAL-Wet                                    MAE=0.3643  RMSE=0.5728  R²=0.6880  MAPE=17.46%
  MANUAL-Wet: N=164  MAE=0.3643  R²=0.6880

  MANUAL overall: MAE=0.3643 min  RMSE=0.5728  R²=0.6880
  ✓ manual_baseline_results.csv saved


## [MAIN PIPELINE] Phase 5F — Flat RF vs Sequential RF

In [22]:
# Adviser requirement: explicitly show what flat RF and sequential RF are.
# Flat RF:   1 flight = 1 prediction, 17 features, no temporal history
# Seq RF:    5-flight window, 90 features (5x18), has temporal history
# Produces: flat_vs_seq_rf_comparison.csv
phase5_flat_vs_seq_rf(df, output_dir)


[MAIN PIPELINE] Phase 5F — Flat RF vs Seq-RF

  [1/2] Flat RF — single flight rows, no sequence window
    Flat RF (no temporal context)                 MAE=0.5404  RMSE=0.7381  R²=0.9076  MAPE=17.92%
    n_features=12  n_test=968
    MAE=0.5404 min  R²=0.9076

  [2/2] Sequential RF — 5-flight window flattened, temporal context present

  [RF] Grid search — 9 configs
    n_est=100 max_d=10    → val_MSE=0.55617 ← best
    n_est=100 max_d=20    → val_MSE=0.55220 ← best
    n_est=100 max_d=None  → val_MSE=0.55162 ← best
    n_est=200 max_d=10    → val_MSE=0.54246 ← best
    n_est=200 max_d=20    → val_MSE=0.54169 ← best
    n_est=200 max_d=None  → val_MSE=0.54060 ← best
    n_est=500 max_d=10    → val_MSE=0.53659 ← best
    n_est=500 max_d=20    → val_MSE=0.53987
    n_est=500 max_d=None  → val_MSE=0.53716
    RF (seq-flattened)                            MAE=0.8082  RMSE=1.1470  R²=0.6922  MAPE=31.43%
    n_features=60  n_test=578
    MAE=0.8082 min  R²=0.6922

  FLAT RF vs SEQUENTIAL R

## [MAIN PIPELINE] Phase 5R — 5-Repetition Stability Analysis (statistician)

In [23]:
# Statistician requirement: train GRU, LSTM, RF each 5 times
# with seeds [42, 7, 13, 99, 2024] to establish consistency.
# Produces:
#   repetition_results.csv — MAE/RMSE/R2/MAPE per rep per model
#   ground_truth_and_predictions.csv — y_true + all 5 predictions per model
# NOTE: This is the slowest phase (5x3 = 15 model trainings).
phase5_repetitions(df, output_dir, n_reps=5)


[MAIN PIPELINE] Phase 5R — Repetition Stability Analysis (5 reps)
  Purpose: Establish statistical consistency across random seeds.
  Models: GRU, LSTM, Sequential RF
  Reps: 5 per model (different random seeds)
  Ground truth: exported alongside all predictions for statistician.

  Seeds used: [42, 7, 13, 99, 2024]
  Test set size: 578 sequences

  ── Rep 1/5  seed=42 ──────────────────────────

  [GRU rep1] Random search — 12 configs
    Config 01  h=128 l=1 d=0.1 lr=0.001 bs=32 → val_loss=0.52374 ← best
    Config 02  h=256 l=1 d=0.1 lr=0.001 bs=64 → val_loss=0.54001
    Config 03  h=128 l=2 d=0.2 lr=0.001 bs=32 → val_loss=0.52844
    Config 04  h=128 l=2 d=0.3 lr=0.0005 bs=32 → val_loss=0.52939
    Config 05  h=256 l=1 d=0.2 lr=0.0001 bs=32 → val_loss=0.54891
    Config 06  h=128 l=1 d=0.3 lr=0.001 bs=32 → val_loss=0.54006
    Config 07  h=128 l=1 d=0.1 lr=0.001 bs=32 → val_loss=0.57032
    Config 08  h=128 l=2 d=0.2 lr=0.001 bs=64 → val_loss=0.52994
    Config 09  h=256 l=2 d=0.3

## [MAIN PIPELINE] Phase 6 — Ablation Experiments (6×6 = 36 runs, RQ2)

In [24]:
# 6 contexts (AUTO/MANUAL × Dry/Hot/Wet) × 6 feature sets = 36 runs
# Produces: ablation_results.csv, full_summary_table.csv
ablation_df = phase6_ablation(df, output_dir)


[MAIN PIPELINE] Phase 6 — Ablation Experiments (36 runs, RQ2)
  ABLATION HYPERPARAMETERS: Fixed hyperparameters are used across all ablation runs (hidden=128, layers=2, dropout=0.2, lr...
  Fixed HP: {'hidden_size': 128, 'num_layers': 2, 'dropout': 0.2, 'learning_rate': 0.001, 'batch_size': 64}
  LSTM ABLATION SENSITIVITY CHECK (v6.1.5 extended): GRU is the primary model for the full 36-run ablation (6 contexts x 6...

  [AUTO-Dry]  2,921 flights  model=GRU
      Epoch   1/50 | Train Loss: 0.4797 | Val Loss: 0.2796
      Epoch   2/50 | Train Loss: 0.4335 | Val Loss: 0.2564
      Epoch   3/50 | Train Loss: 0.3440 | Val Loss: 0.2536
      Epoch   4/50 | Train Loss: 0.3183 | Val Loss: 0.2388
      Epoch   5/50 | Train Loss: 0.2966 | Val Loss: 0.2261
      Epoch   6/50 | Train Loss: 0.2785 | Val Loss: 0.2134
      Epoch   7/50 | Train Loss: 0.2697 | Val Loss: 0.2130
      Epoch   8/50 | Train Loss: 0.2637 | Val Loss: 0.2107
      Epoch   9/50 | Train Loss: 0.2620 | Val Loss: 0.2082
      

## [MAIN PIPELINE] Phase 6D — Operational & Sequential Feature Ablation (v8.1.2)

**Supplementary to Phase 6 (environmental ablation).** Provides context for RQ2 by showing how much
operational and sequential features contribute when removed — enabling direct comparison with
the environmental δMAE values from Phase 6.

- **Model**: GRU with same `ABLATION_HP` and chronological split as Phase 6
- **Contexts**: AUTO-Dry, AUTO-Hot, AUTO-Wet
- **Features removed**: `sprayed_area`, `total_amount_l`, `starting_battery`, `rolling_duration_3`, `time_since_last_flight`
- **Runs**: 3 contexts × 6 feature sets = **18 runs**
- **Output**: `operational_ablation_results.csv`


In [25]:
# [MAIN PIPELINE] Phase 6D — Operational & Sequential Feature Ablation (v8.1.2)
# Supplementary to Phase 6. Same GRU + ABLATION_HP + split.
# 18 runs: 3 AUTO contexts × 6 feature sets.
# Removes: sprayed_area, total_amount_l, starting_battery,
#          rolling_duration_3, time_since_last_flight
# Produces: operational_ablation_results.csv
#
# Run AFTER Phase 6 (E9) — uses same session-aware GRU ablation pattern.
op_ablation_df = phase6d_operational_ablation(df, output_dir)



[MAIN PIPELINE] Phase 6D — Operational & Sequential Feature Ablation (v8.1.2)
  OPERATIONAL ABLATION (v8.1.2 SUPPLEMENTARY): Phase 6D removes operational and sequential features one at a time from the...
  Supplementary to Phase 6. Same GRU + ABLATION_HP + split.
  18 runs: 3 AUTO contexts × 6 feature sets.

  [AUTO-Dry]  2,921 flights  model=GRU
      Epoch   1/50 | Train Loss: 0.4820 | Val Loss: 0.2840
      Epoch   2/50 | Train Loss: 0.4019 | Val Loss: 0.2562
      Epoch   3/50 | Train Loss: 0.3597 | Val Loss: 0.2504
      Epoch   4/50 | Train Loss: 0.3200 | Val Loss: 0.2371
      Epoch   5/50 | Train Loss: 0.2838 | Val Loss: 0.2211
      Epoch   6/50 | Train Loss: 0.2777 | Val Loss: 0.2112
      Epoch   7/50 | Train Loss: 0.2633 | Val Loss: 0.2085
      Epoch   8/50 | Train Loss: 0.2624 | Val Loss: 0.2088
      Epoch   9/50 | Train Loss: 0.2649 | Val Loss: 0.2083
      Epoch  10/50 | Train Loss: 0.2499 | Val Loss: 0.2029
      Epoch  11/50 | Train Loss: 0.2506 | Val Loss: 0.2093
 

## [MAIN PIPELINE] Phase 6A — Ablation Baseline Control (v7.1)

Runs GRU with all 17 features on 3 AUTO contexts. Uses same `ABLATION_HP` and
chronological split as the main ablation grid. Provides explicit labeled reference.

Produces: `ablation_baseline_control.csv`

In [26]:
# [MAIN PIPELINE] Phase 6A — Ablation Baseline Control (v7.1)
# Provides explicit full 17-feature GRU reference for the ablation framework.
# Uses ABLATION_HP — same as Phase 6 ablation grid. No Phase 5 retraining.
# Produces: ablation_baseline_control.csv
baseline_ctrl_df = phase6_ablation_baseline(df, output_dir)



[MAIN PIPELINE] Phase 6A — Ablation Baseline Control (v9: full 12 features)
  FULL 17-FEATURE BASELINE CONTROL (v7.1): phase6_ablation_baseline() provides an explicit labeled reference run using all...
  ✓ Leakage check passed (v6.1):
    - target 'flight_duration_min' NOT in feature_cols
    - ending_battery NOT in feature_cols
    - all 12 features present in DataFrame
    - rolling_duration_3 uses shift(1): past-flight values only
    - LEAKAGE VALIDATION (v6.1): All model input features are sourced exclusively from information observa...
    AUTO-Dry|full_features_baseline               MAE=0.9650  RMSE=1.4656  R²=0.4990  MAPE=32.72%
    AUTO-Dry | full_features_baseline | MAE=0.9650  R²=0.4990
    AUTO-Hot|full_features_baseline               MAE=1.4166  RMSE=1.8285  R²=0.5150  MAPE=44.36%
    AUTO-Hot | full_features_baseline | MAE=1.4166  R²=0.5150
    AUTO-Wet|full_features_baseline               MAE=0.7598  RMSE=1.1402  R²=0.7069  MAPE=26.83%
    AUTO-Wet | full_features_base

## [MAIN PIPELINE] Phase 6b — LSTM Sensitivity Check (v6.1.5: 9 targeted runs)

3 AUTO contexts × 3 feature sets = 9 LSTM runs.
Feature sets: full / remove precipitation (top-1) / remove temperature (top-2).
GRU 36-run grid remains the primary ablation source (Phase 6).

Outputs:
- `lstm_sensitivity_check.csv` (backward-compatible, 3 rows, full feature set)
- `lstm_sensitivity_extended.csv` (NEW, 9 rows with δMAE column)


In [27]:
# [MAIN PIPELINE] Phase 6b — LSTM Sensitivity Check (v6.1.5: 9 targeted runs)
# GRU full 36-run ablation (Phase 6) is the primary source for RQ2.
# This phase cross-checks the top-2 GRU features using LSTM:
#   3 contexts (AUTO-Dry, AUTO-Hot, AUTO-Wet) × 3 feature sets = 9 runs
#   feature sets: full | remove precipitation | remove temperature
#
# Produces:
#   lstm_sensitivity_check.csv    (3 rows, backward-compatible)
#   lstm_sensitivity_extended.csv (9 rows, NEW v6.1.5 δMAE output)
#
# Limitation notes printed automatically:
#   ABLATION_SINGLE_RUN_NOTE   — single-run stochastic variance
#   ABLATION_CORR_FEATURE_NOTE — correlated-feature marginal ablation limit
lstm_sens_df = phase6b_lstm_sensitivity(df, output_dir)



[MAIN PIPELINE] Phase 6b — LSTM Sensitivity Check (v6.1.5: 9 targeted runs)
  LSTM ABLATION SENSITIVITY CHECK (v6.1.5 extended): GRU is the primary model for the full 36-run ablation (6 contexts x 6...
  Single-run note: SINGLE-RUN ABLATION LIMITATION: Each ablation configuration runs once using fixe...
  Correlation note: CORRELATED FEATURE LIMITATION: Marginal ablation (remove one feature at a time) ...
    AUTO-Dry|LSTM|full                            MAE=0.9729  RMSE=1.4700  R²=0.4960  MAPE=33.61%
    AUTO-Dry|LSTM|remove_precip                   MAE=0.9883  RMSE=1.4812  R²=0.4883  MAPE=33.60%
    AUTO-Dry|LSTM|remove_temp                     MAE=0.9604  RMSE=1.4888  R²=0.4830  MAPE=31.41%
    AUTO-Hot|LSTM|full                            MAE=1.5695  RMSE=2.0706  R²=0.3781  MAPE=52.61%
    AUTO-Hot|LSTM|remove_precip                   MAE=1.6073  RMSE=2.1477  R²=0.3309  MAPE=53.98%
    AUTO-Hot|LSTM|remove_temp                     MAE=1.5383  RMSE=2.0519  R²=0.3893  MAPE=55.39%
  

## [MAIN PIPELINE] Phase 6C — Sequence-Aligned Evaluation (v8.1)

**All 7 models on the SAME test rows** (GRU, LSTM, Seq-RF, RF, XGBoost, LightGBM, CatBoost).
Rows recovered via `target_row_idx` from `pred_df`.
This is the only valid cross-model comparison.

Output: `prediction_comparison_models.csv`


In [28]:
# [MAIN PIPELINE] Phase 6C — Prediction Comparison Table (v7.2)
# Produces: prediction_comparison_models.csv — all 6 models + metadata
#
# ALIGNMENT: boosted_test_preds (from Phase 5B) uses flat individual-flight
# test rows; pred_df (from Phase 5) uses sequence-based test targets.
# These are different n and different rows — NOT combinable row-by-row.
# phase6c re-evaluates boosted models on df_test_meta (seq-aligned rows)
# so all 6 models are compared on the SAME n test flights.
#
# df_test_meta: exact df_auto rows corresponding to test sequence targets,
# recovered via target_row_idx from pred_df (v7.3 fix for correct alignment).
# v7.3 FIX: df_auto.tail() was wrong — it took the last N rows of df_auto
# which does NOT correspond to the sequence test targets (sessions with
# ≤5 flights are skipped in sequence building, so row count differs).
# Correct approach: use test_target_idx from phase5_baseline to recover
# the exact df_auto rows that are the TARGET FLIGHTS of test sequences.
if "target_row_idx" in pred_df.columns:
    _tidx        = pred_df["target_row_idx"].values.astype(int)
    df_test_meta = df_auto.iloc[_tidx].reset_index(drop=True)
else:
    # Fallback for notebooks run before v7.3 (target_row_idx not in pred_df)
    print("WARNING: target_row_idx not in pred_df — using tail() fallback.")
    print("  Re-run Phase 5 to get correct row alignment.")
    df_test_meta = df_auto.tail(len(pred_df)).reset_index(drop=True)

pred_comp_df = phase6c_prediction_comparison(
    pred_df           = pred_df,
    boosted_test_preds= boosted_test_preds,
    df_test_meta      = df_test_meta,
    output_dir        = output_dir,
    xgb_model         = xgb_model,
    lgbm_model        = lgbm_model,
    cb_model          = cb_best_model,
)





[MAIN PIPELINE] Phase 6C — Sequence-Aligned Evaluation (v8.1): 7 models, same rows
  SEQUENCE-ALIGNED EVALUATION (v8.1): 7 models on identical rows.
  GRU, LSTM, Seq-RF, RF, XGBoost, LightGBM, CatBoost.
  See flat_evaluation_results.csv for flat-only benchmark.
  Boosted models: re-evaluated on df_test_meta (seq-aligned rows).
  Test samples (seq-aligned): 578
  Saved: prediction_comparison_models.csv
    Rows: 578  Columns: 24

  MAE summary (seq-aligned test set, n=578):
  Model         MAE (min)   Std (min)
  ------------------------------------
  LSTM             0.8084      0.7965
  GRU              0.9907      0.7476
  Seq-RF           0.8082      0.8146
  XGBoost          0.5462      0.4913
  LightGBM         0.5572      0.5108
  CatBoost         0.5382      0.4924

  Stand-alone boosted metrics (boosted flat test set, n=968):
  NOTE: n=968 differs from n=578 — flat flights vs seq targets.
    XGBoost   : MAE = 0.5797 min (on boosted test set)
    LightGBM  : MAE = 0.5901 min (

## [MAIN PIPELINE] Phase 7 — Visualizations + Summary Tables

In [29]:
# Produces: feature_importance.png, delta_mae_heatmap.png,
#           mae_by_season.png, full_summary_table.csv, justifications_v5.txt
phase7_analyze(ablation_df, pred_df, lstm_sens_df, output_dir)
print(f"\n{'='*60}")
print("PIPELINE v8.3 COMPLETE")
print(f"All outputs saved to ./{output_dir}/")



[MAIN PIPELINE] Phase 7 — Analyze Results and Visualize

  Baseline MAE per context (ablation Phase 6):
   context model_used      MAE     RMSE       R2  n_samples
  AUTO-Dry        GRU 0.936376 1.457296 0.504670       1848
  AUTO-Hot        GRU 1.481195 1.848070 0.504582       1325
  AUTO-Wet        GRU 0.746807 1.121604 0.716396        675
MANUAL-Dry    flat RF 0.404721 0.577494 0.722606        399
MANUAL-Hot    flat RF 0.339836 0.503226 0.700972        355
MANUAL-Wet    flat RF 0.347875 0.461678 0.622073        336

  Average δMAE per removed feature (higher = more important):
removed_feature
precipitation    -0.0000
wind_speed_10m   -0.0076
temperature_2m   -0.0099

  LSTM sensitivity check (should confirm GRU top features):
 context    MAE   RMSE     R2
AUTO-Dry 0.9729 1.4700 0.4960
AUTO-Hot 1.5695 2.0706 0.3781
AUTO-Wet 0.8525 1.1798 0.6862
  ✓ baseline_pred_vs_actual.png saved

  ✓ All outputs saved to C:\Users\User\Documents\THESIS_HUHU\FINAL PIPELINE\outputs_v9/

PIPELINE v8.

## [MAIN PIPELINE] Phase 7B — Sequential Permutation Importance (v8.3)

Computes permutation importance for **GRU, LSTM, and Seq-RF** on the same AUTO sequence
test set used in Phase 5. Cross-validates the boosted model importance from Phase 5B.

- **Requires**: `gru_model`, `lstm_model`, `rf_model`, `X_te`, `y_te`, `y_sc` from Phase 5 (E3)
- **n_repeats**: 5 per feature per model (v9 OPT: was 10; stable for thesis reporting)
- **Aggregation**: mean importance across all 5 timesteps per feature
- **Output**: `sequential_permutation_importance.csv`

> **Run AFTER E3 (Phase 5) and E4 (Phase 5B).**


In [30]:
# [MAIN PIPELINE] Phase 7B — Sequential Permutation Importance (v8.3)
# Requires gru_model, lstm_model, rf_model, X_te, y_te, y_sc from Phase 5 (E3)
# Requires permutation_importance.csv from Phase 5B (E4) for boosted comparison
#
# WHAT THIS DOES:
# Shuffles each of the 17 features across all 5 timesteps, 10 times each.
# Measures how much MAE increases when a feature is removed.
# Runs on GRU, LSTM, Seq-RF separately, then compares with boosted importance.
#
# If all 4 model families agree on the ranking, the finding is model-agnostic.
#
# Produces: sequential_permutation_importance.csv

# Recover X_te and y_te from X_seq, y_seq (already available from E3)
_n_seq    = len(y_seq)
_n_tr_seq = int(_n_seq * CFG["train_frac"])
_n_va_seq = int(_n_seq * CFG["val_frac"])
(_, _), (_, _), (X_te, y_te) = temporal_split_sequences(
    X_seq, y_seq, CFG["train_frac"], CFG["val_frac"]
)

seq_perm_df = phase7b_sequential_permutation_importance(
    gru_model  = gru_model,
    lstm_model = lstm_model,
    rf_model   = rf_model,
    X_te       = X_te,
    y_te       = y_te,
    y_sc       = y_sc,
    output_dir = output_dir,
    n_repeats  = 5,   # v9 OPT: was 10
    random_state = 42,
)
print("\n✓ Phase 7B complete. "
      "sequential_permutation_importance.csv saved to output_dir.")



[MAIN PIPELINE] Phase 7B — Sequential Permutation Importance (v8.3)
  Test set: 578 sequences  | Features: 12  | Timesteps: 5
  Flattened input shape: (578, 60)  | n_repeats: 5
  Implementation: manual (no sklearn .fit() required)

  [1/3] GRU permutation importance...
  GRU done. Top: rolling_duration_3 (δMAE=0.0538)

  [2/3] LSTM permutation importance...
  LSTM done. Top: sprayed_area (δMAE=0.0271)

  [3/3] Seq-RF permutation importance...
  Seq-RF done. Top: rolling_duration_3 (δMAE=0.1143)

  Boosted importance loaded from permutation_importance.csv

  ✓ sequential_permutation_importance.csv → C:\Users\User\Documents\THESIS_HUHU\FINAL PIPELINE\outputs_v9\sequential_permutation_importance.csv

  Feature                           GRU     LSTM   Seq-RF  Boosted     Mean
  --------------------------------------------------------------------
  rolling_duration_3             0.0538   0.0167   0.1143   0.0849   0.0616
  sprayed_area                   0.0465   0.0271   0.0434   0.9343   